# Wildfire deep architecture search

Attach **z789456sx/ts-satfire** and **dianatofficial/global-daily-weather-1950-2026-analysis-layer**. Select **GPU T4** and enable Internet, then Save & Run All.

All computation stays on Kaggle. Uses all official training events, screens 17 architecture/fusion recipes plus loss variants, promotes three recipes to full-data multi-seed runs and selects TTA/ensembles on tuning events. Separate whole events are held out for confirmation. Test is never evaluated. Search budget is eight hours plus preprocessing and final inference; one T4 is used. Inspect `/kaggle/working/exploration/report.json` after completion.

The task retains positive-progression-centered 64x64 patches, so scores are not comparable to the official full-frame TS-SatFire benchmark or earlier NDWS scores.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess
root = Path('/kaggle/working/wildfire')
root.mkdir(exist_ok=True)
sources = json.loads("{\"wf/train.py\":\"\\\"\\\"\\\"One training run: config dict in, history + best checkpoint out.\\\"\\\"\\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport contextlib\\r\\nimport json\\r\\nimport math\\r\\nimport time\\r\\nfrom pathlib import Path\\r\\n\\r\\nimport numpy as np\\r\\nimport torch\\r\\nimport torch.nn.functional as F\\r\\nfrom torch.utils.data import DataLoader, Subset\\r\\n\\r\\nfrom .data import CHANNELS, FireDataset, load_ext, split_indices\\r\\nfrom .losses import make_loss, moment\\r\\nfrom .metrics import evaluate, evaluate_area, sweep, sweep_area\\r\\nfrom .models import build_model\\r\\nfrom .prep import norm_for\\r\\n\\r\\n\\r\\ndef build(cfg: dict, root: str | Path = \\\"data/processed\\\"):\\r\\n    \\\"\\\"\\\"Model + the data-preprocessing options it was trained with.\\\"\\\"\\\"\\r\\n    d = cfg[\\\"data\\\"]\\r\\n    prep, drop = d.get(\\\"prep\\\", \\\"base\\\"), d.get(\\\"drop\\\", \\\"\\\")\\r\\n    mean, std = norm_for(root, prep, drop)\\r\\n    model = build_model({**cfg[\\\"model\\\"], \\\"cin\\\": len(mean)}, mean, std)\\r\\n    return model, prep, drop\\r\\n\\r\\n\\r\\ndef norm_stats(root: str | Path, n: int = 20000, seed: int = 0) -> tuple[list[float], list[float]]:\\r\\n    path = Path(root) / \\\"norm.json\\\"\\r\\n    if path.exists():\\r\\n        d = json.loads(path.read_text())\\r\\n        return d[\\\"mean\\\"], d[\\\"std\\\"]\\r\\n    from .data import cache\\r\\n\\r\\n    x = cache(root, \\\"train\\\")[0]\\r\\n    idx = np.random.default_rng(seed).integers(0, len(x), min(n, len(x)))\\r\\n    blocks = np.stack([np.asarray(x[i], dtype=np.float32) for i in idx])\\r\\n    mean = blocks.mean(axis=(0, 2, 3)).tolist()\\r\\n    std = np.maximum(blocks.std(axis=(0, 2, 3)), 1e-3).tolist()\\r\\n    path.write_text(json.dumps({\\\"channels\\\": CHANNELS, \\\"mean\\\": mean, \\\"std\\\": std}, indent=1))\\r\\n    return mean, std\\r\\n\\r\\n\\r\\nclass EMA:\\r\\n    def __init__(self, model: torch.nn.Module, decay: float = 0.999):\\r\\n        self.decay = decay\\r\\n        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items() if v.dtype.is_floating_point}\\r\\n\\r\\n    @torch.no_grad()\\r\\n    def update(self, model: torch.nn.Module, step: int = 10**9) -> None:\\r\\n        # Adaptive warmup: with a fixed 0.999 the shadow is still the init\\r\\n        # weights after a 12-epoch screen, which made short runs look dead.\\r\\n        d = min(self.decay, (1.0 + step) / (10.0 + step))\\r\\n        for k, v in model.state_dict().items():\\r\\n            if k in self.shadow:\\r\\n                self.shadow[k].mul_(d).add_(v.detach(), alpha=1 - d)\\r\\n\\r\\n    @contextlib.contextmanager\\r\\n    def apply(self, model: torch.nn.Module):\\r\\n        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}\\r\\n        model.load_state_dict(self.shadow, strict=False)\\r\\n        try:\\r\\n            yield\\r\\n        finally:\\r\\n            model.load_state_dict(backup)\\r\\n\\r\\n\\r\\ndef _seed_worker(_worker_id):\\r\\n    info = torch.utils.data.get_worker_info()\\r\\n    if hasattr(info.dataset, \\\"rng\\\"):\\r\\n        info.dataset.rng = np.random.default_rng(info.seed % (2 ** 32))\\r\\n\\r\\n\\r\\ndef _loader(ds, bs: int, shuffle: bool, workers: int, seed: int = 0) -> DataLoader:\\r\\n    # Persistent workers: re-spawning 10 forks every epoch cost ~25 s per epoch,\\r\\n    # which dominated short screening runs.\\r\\n    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=workers, pin_memory=False,\\r\\n                      drop_last=shuffle, persistent_workers=workers > 0,\\r\\n                      prefetch_factor=2 if workers else None,\\r\\n                      generator=torch.Generator().manual_seed(seed), worker_init_fn=_seed_worker)\\r\\n\\r\\n\\r\\n@torch.no_grad()\\r\\ndef predict(model, loader, device, dtype):\\r\\n    \\\"\\\"\\\"Probabilities plus, when the model has one, the predicted log1p(area).\\\"\\\"\\\"\\r\\n    model.eval()\\r\\n    probs, tgts, valids, areas = [], [], [], []\\r\\n    for b in loader:\\r\\n        x, y, v = b[0], b[1], b[2]\\r\\n        e = b[4].to(device, non_blocking=True) if len(b) > 4 else None\\r\\n        x = x.to(device, non_blocking=True).float().to(memory_format=torch.channels_last)\\r\\n        with torch.autocast(\\\"cuda\\\", dtype=dtype, enabled=device.type == \\\"cuda\\\"):\\r\\n            o = model(x, e) if e is not None else model(x)\\r\\n        probs.append(torch.sigmoid(o[\\\"main\\\"].float()).cpu().numpy()[:, 0])\\r\\n        if \\\"area\\\" in o:\\r\\n            areas.append(o[\\\"area\\\"].float().cpu().numpy())\\r\\n        tgts.append(y.numpy() > 0.5)\\r\\n        valids.append(v.numpy() > 0.5)\\r\\n    area = np.concatenate(areas) if areas else None\\r\\n    return np.concatenate(probs), np.concatenate(tgts), np.concatenate(valids), area\\r\\n\\r\\n\\r\\ndef run(cfg: dict, run_name: str | None = None, out_dir: str | Path = \\\"results/runs\\\",\\r\\n        resume_from: str | Path | None = None) -> dict:\\r\\n    rng = int(cfg.get(\\\"seed\\\", 0))\\r\\n    torch.manual_seed(rng)\\r\\n    np.random.seed(rng)\\r\\n    torch.backends.cudnn.benchmark = True\\r\\n    torch.set_float32_matmul_precision(\\\"high\\\")\\r\\n\\r\\n    d, m, l, t = cfg[\\\"data\\\"], cfg[\\\"model\\\"], cfg[\\\"loss\\\"], cfg[\\\"train\\\"]\\r\\n    root = d.get(\\\"root\\\", \\\"data/processed\\\")\\r\\n    name = run_name or cfg.get(\\\"name\\\", \\\"run\\\")\\r\\n    dev = torch.device(\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\")\\r\\n    dtype = torch.bfloat16 if t.get(\\\"amp\\\", \\\"bf16\\\") in (True, \\\"bf16\\\") else torch.float16\\r\\n    if dev.type == \\\"cuda\\\" and dtype == torch.bfloat16 and not torch.cuda.is_bf16_supported():\\r\\n        dtype = torch.float16\\r\\n    use_amp = dev.type == \\\"cuda\\\" and bool(t.get(\\\"amp\\\", True))\\r\\n\\r\\n    model, prep, drop = build(cfg, root)\\r\\n    if resume_from:\\r\\n        # Fine-tuning: warm start from a converged checkpoint, keeping whatever\\r\\n        # heads still match (new heads stay at their init).\\r\\n        model.load_state_dict(torch.load(resume_from, map_location=dev)[\\\"model\\\"], strict=False)\\r\\n        print(f\\\"resumed from {resume_from}\\\", flush=True)\\r\\n    model = model.to(dev).to(memory_format=torch.channels_last)\\r\\n    nparam = sum(p.numel() for p in model.parameters())\\r\\n\\r\\n    train_idx = split_indices(root, \\\"train\\\", t.get(\\\"n_train\\\"), seed=rng)\\r\\n    criterion = make_loss(l)\\r\\n    wb, wg = l.get(\\\"boundary_weight\\\", 0.0), l.get(\\\"growth_weight\\\", 0.0)\\r\\n    wm = l.get(\\\"moment_weight\\\", 0.0)\\r\\n    wa = l.get(\\\"area_weight\\\", 0.0)\\r\\n    wds = l.get(\\\"ds_weight\\\", 0.0)\\r\\n    # Fork + copy-on-write makes an in-RAM copy per worker, so the budget is\\r\\n    # per-dataset, not per-row: a 19 GB subset across 10 workers is OOM-killed.\\r\\n    in_mem = d.get(\\\"in_memory\\\")\\r\\n    if in_mem is None:\\r\\n        in_mem = len(train_idx) * len(CHANNELS) * 64 * 64 * 2 <= 4e9\\r\\n    tr_ds = FireDataset(root, \\\"train\\\", train_idx, augment=True, prev_dropout=d.get(\\\"prev_dropout\\\", 0.2),\\r\\n                        seed=rng, aux=bool(wb or wg), in_memory=in_mem, prep=prep, drop=drop,\\r\\n                        pf_aug=d.get(\\\"pf_aug\\\", \\\"\\\"), soft=float(d.get(\\\"soft\\\", 0.0)),\\r\\n                        ext=load_ext(root, \\\"train\\\") if int(m.get(\\\"late\\\", 0)) else None)\\r\\n    ev_idx = split_indices(root, \\\"eval\\\", t.get(\\\"eval_n\\\"), seed=0)\\r\\n    ev_ds = FireDataset(root, \\\"eval\\\", ev_idx, augment=False, aux=False, in_memory=len(ev_idx) <= 30000,\\r\\n                        prep=prep, drop=drop,\\r\\n                        ext=load_ext(root, \\\"eval\\\") if int(m.get(\\\"late\\\", 0)) else None)\\r\\n    if int(m.get(\\\"late\\\", 0)):\\r\\n        for split, ds in ((\\\"train\\\", tr_ds), (\\\"eval\\\", ev_ds)):\\r\\n            if ds.ext is None or ds.ext.shape[1] != int(m[\\\"late\\\"]):\\r\\n                raise ValueError(f\\\"{split}: missing or incompatible ext.npy for late fusion\\\")\\r\\n    bs = min(int(t.get(\\\"batch_size\\\", 256)), len(tr_ds))\\r\\n    tr_ld = _loader(tr_ds, bs, True, int(t.get(\\\"workers\\\", 8)), seed=rng)\\r\\n    ev_ld = _loader(ev_ds, int(t.get(\\\"eval_batch_size\\\", 128)), False, int(t.get(\\\"eval_workers\\\", 4)))\\r\\n    print(f\\\"RUN {name} params={nparam/1e6:.2f}M n_train={len(train_idx)} n_eval={len(ev_idx)} \\\"\\r\\n          f\\\"epochs={int(t.get('epochs', 10))} bs={bs}\\\", flush=True)\\r\\n\\r\\n    opt = torch.optim.AdamW(model.parameters(), lr=float(t.get(\\\"lr\\\", 2e-3)),\\r\\n                            weight_decay=float(t.get(\\\"weight_decay\\\", 1e-4)))\\r\\n    epochs = int(t.get(\\\"epochs\\\", 10))\\r\\n    warm = int(t.get(\\\"warmup_epochs\\\", 1))\\r\\n    steps_per_epoch = max(1, len(tr_ds) // bs)\\r\\n    total_steps = epochs * steps_per_epoch\\r\\n    warm_steps = warm * steps_per_epoch\\r\\n\\r\\n    def lr_at(step: int) -> float:\\r\\n        if step < warm_steps:\\r\\n            return (step + 1) / max(warm_steps, 1)\\r\\n        p = (step - warm_steps) / max(total_steps - warm_steps, 1)\\r\\n        return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * p))\\r\\n\\r\\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)\\r\\n    ema = EMA(model, float(t.get(\\\"ema_decay\\\", 0.999)))\\r\\n    scaler = torch.amp.GradScaler(\\\"cuda\\\", enabled=use_amp and dtype is torch.float16)\\r\\n    clip = float(t.get(\\\"grad_clip\\\", 1.0))\\r\\n\\r\\n    out = Path(out_dir) / name\\r\\n    out.mkdir(parents=True, exist_ok=True)\\r\\n    best = {\\\"macro_dice\\\": -1.0, \\\"epoch\\\": -1}\\r\\n    history = []\\r\\n    t_start = time.time()\\r\\n    step = 0\\r\\n    patience = int(t.get(\\\"patience\\\", epochs))\\r\\n\\r\\n    for epoch in range(epochs):\\r\\n        model.train()\\r\\n        t0, seen, losses, t_data = time.time(), 0, [], 0.0\\r\\n        it = iter(tr_ld)\\r\\n        for _ in range(steps_per_epoch):\\r\\n            t_wait = time.time()\\r\\n            b = next(it)\\r\\n            x, y, v, aux = b[0], b[1], b[2], b[3]\\r\\n            e = b[4].to(dev, non_blocking=True) if len(b) > 4 else None\\r\\n            t_data += time.time() - t_wait\\r\\n            x = x.to(dev, non_blocking=True).float().to(memory_format=torch.channels_last)\\r\\n            y = y.to(dev, non_blocking=True).unsqueeze(1)\\r\\n            v = v.to(dev, non_blocking=True).unsqueeze(1)\\r\\n            aux = aux.to(dev, non_blocking=True)\\r\\n            opt.zero_grad(set_to_none=True)\\r\\n            with torch.autocast(\\\"cuda\\\", dtype=dtype, enabled=use_amp):\\r\\n                o = model(x, e) if e is not None else model(x)\\r\\n                loss = criterion(o[\\\"main\\\"].float(), y, v)\\r\\n                if wm:\\r\\n                    loss = loss + wm * moment(o[\\\"main\\\"].float(), y, v)\\r\\n                if wa and \\\"area\\\" in o:\\r\\n                    # L1 on log area = relative error; a 0.3 loss is ~30% off.\\r\\n                    at = torch.log1p((y * v).sum(dim=(1, 2, 3)))\\r\\n                    loss = loss + wa * F.l1_loss(o[\\\"area\\\"].float(), at)\\r\\n                if wds:\\r\\n                    for k, a in o.items():\\r\\n                        if k.startswith(\\\"aux\\\"):\\r\\n                            s = a.shape[-2:]\\r\\n                            loss = loss + wds * criterion(a.float(),\\r\\n                                                          F.interpolate(y, size=s, mode=\\\"area\\\"),\\r\\n                                                          F.interpolate(v, size=s, mode=\\\"area\\\"))\\r\\n                if wb and \\\"boundary\\\" in o:\\r\\n                    loss = loss + wb * criterion(o[\\\"boundary\\\"].float(), aux[:, 0:1], v)\\r\\n                if wg and \\\"growth\\\" in o:\\r\\n                    loss = loss + wg * criterion(o[\\\"growth\\\"].float(), aux[:, 1:2], v)\\r\\n            scaler.scale(loss).backward()\\r\\n            if clip:\\r\\n                scaler.unscale_(opt)\\r\\n                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)\\r\\n            scaler.step(opt)\\r\\n            scaler.update()\\r\\n            sched.step()\\r\\n            ema.update(model, step)\\r\\n            losses.append(float(loss.detach()))\\r\\n            seen += x.shape[0]\\r\\n            step += 1\\r\\n        dt = time.time() - t0\\r\\n        t_eval = time.time()\\r\\n        with ema.apply(model):\\r\\n            prob, tg, va, ar = predict(model, ev_ld, dev, dtype)\\r\\n        mt = (evaluate_area(prob, np.expm1(ar), tg, va, full=False) if ar is not None\\r\\n              else evaluate(prob, tg, va, 0.5, full=False))\\r\\n        checkpoint_thr = 0.5\\r\\n        if t.get(\\\"checkpoint_sweep\\\") and ar is None:\\r\\n            checkpoint_thr, _ = sweep(prob, tg, va)\\r\\n            mt = evaluate(prob, tg, va, checkpoint_thr, full=False)\\r\\n        t_eval = time.time() - t_eval\\r\\n        row = {\\\"epoch\\\": epoch, \\\"loss\\\": float(np.mean(losses)), \\\"train_sps\\\": int(seen / dt),\\r\\n               \\\"lr\\\": opt.param_groups[0][\\\"lr\\\"], **{f\\\"val_{k}\\\": v for k, v in mt.items()},\\r\\n               \\\"eval_s\\\": round(t_eval, 1), \\\"data_s\\\": round(t_data, 1), \\\"loop_s\\\": round(dt, 1),\\r\\n               \\\"minutes\\\": round((time.time() - t_start) / 60, 1)}\\r\\n        history.append(row)\\r\\n        if mt[\\\"macro_dice\\\"] > best[\\\"macro_dice\\\"]:\\r\\n            best = {**mt, \\\"epoch\\\": epoch, \\\"thr\\\": checkpoint_thr}\\r\\n            t_save = time.time()\\r\\n            torch.save({\\\"model\\\": ema.shadow, \\\"cfg\\\": cfg, \\\"metrics\\\": best, \\\"epoch\\\": epoch,\\r\\n                        \\\"params\\\": nparam}, out / \\\"best.pt\\\")\\r\\n            row[\\\"save_s\\\"] = round(time.time() - t_save, 1)\\r\\n        print(json.dumps(row), flush=True)\\r\\n        if epoch + 1 - best[\\\"epoch\\\"] >= patience and best[\\\"epoch\\\"] >= 0:\\r\\n            print(\\\"early stop\\\", flush=True)\\r\\n            break\\r\\n        if t.get(\\\"abort_below\\\") and epoch + 1 >= int(t.get(\\\"abort_after\\\", 3)) \\\\\\r\\n                and best[\\\"macro_dice\\\"] < float(t[\\\"abort_below\\\"]):\\r\\n            print(\\\"abort: below bar\\\", flush=True)\\r\\n            break\\r\\n        if t.get(\\\"max_seconds\\\") and time.time() - t_start > float(t[\\\"max_seconds\\\"]):\\r\\n            print(\\\"time budget reached\\\", flush=True)\\r\\n            break\\r\\n\\r\\n    del tr_ld, ev_ld\\r\\n    ckpt = out / \\\"best.pt\\\"\\r\\n    if ckpt.exists():\\r\\n        model.load_state_dict(torch.load(ckpt, map_location=dev)[\\\"model\\\"], strict=False)\\r\\n    full_ds = FireDataset(root, \\\"eval\\\", None, aux=False, prep=prep, drop=drop,\\r\\n                          ext=load_ext(root, \\\"eval\\\") if int(m.get(\\\"late\\\", 0)) else None)\\r\\n    prob, tg, va, ar = predict(model, _loader(full_ds, int(t.get(\\\"eval_batch_size\\\", 128)), False, int(t.get(\\\"eval_workers\\\", 4))), dev, dtype)\\r\\n    if ar is not None:\\r\\n        thr, rows = sweep_area(prob, np.expm1(ar), tg, va)\\r\\n        full = evaluate_area(prob, np.expm1(ar) * thr, tg, va, full=True)\\r\\n        decode = \\\"area\\\"\\r\\n    else:\\r\\n        thr, rows = sweep(prob, tg, va)\\r\\n        full = evaluate(prob, tg, va, thr, full=True)\\r\\n        decode = \\\"thr\\\"\\r\\n    summary = {\\\"name\\\": name, \\\"params\\\": nparam, \\\"best_epoch\\\": best[\\\"epoch\\\"], \\\"best_thr\\\": thr,\\r\\n               \\\"decode\\\": decode, \\\"best_macro_dice\\\": best[\\\"macro_dice\\\"], \\\"val_at_best_thr\\\": full,\\r\\n               \\\"minutes\\\": round((time.time() - t_start) / 60, 1), \\\"history\\\": history, \\\"sweep\\\": rows,\\r\\n               \\\"cfg\\\": cfg}\\r\\n    (out / \\\"summary.json\\\").write_text(json.dumps(summary, indent=1))\\r\\n    print(\\\"FINAL\\\", json.dumps({k: summary[k] for k in (\\\"name\\\", \\\"params\\\", \\\"best_epoch\\\", \\\"best_thr\\\", \\\"minutes\\\")}),\\r\\n          json.dumps(full), flush=True)\\r\\n    return summary\\r\\n\",\"wf/data.py\":\"\\\"\\\"\\\"Cached 23-channel patches -> training tensors.\\r\\n\\r\\nThe heavy features (slope, fire_front, fire_dist, ...) are materialised once into\\r\\n`features.npy` by the old pipeline; here we only read rows and augment, which is\\r\\nwhat keeps the L4 busy instead of the CPU.\\r\\n\\\"\\\"\\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nfrom pathlib import Path\\r\\n\\r\\nimport numpy as np\\r\\nimport torch\\r\\nfrom torch.utils.data import Dataset\\r\\n\\r\\nCHANNELS = [\\r\\n    \\\"i1\\\", \\\"i2\\\", \\\"i3\\\", \\\"i4\\\", \\\"i5\\\", \\\"m11\\\", \\\"night_i4\\\", \\\"night_i5\\\", \\\"ndvi\\\", \\\"evi2\\\", \\\"precip\\\",\\r\\n    \\\"wind_speed\\\", \\\"wind_dir_sin\\\", \\\"wind_dir_cos\\\", \\\"tmin\\\", \\\"tmax\\\", \\\"erc\\\", \\\"sph\\\", \\\"slope\\\",\\r\\n    \\\"aspect_sin\\\", \\\"aspect_cos\\\", \\\"elevation\\\", \\\"pdsi\\\", \\\"landcover\\\", \\\"fc_wind_speed\\\",\\r\\n    \\\"fc_dir_sin\\\", \\\"fc_dir_cos\\\", \\\"fc_temp\\\", \\\"prev_fire\\\", \\\"fire_front\\\", \\\"fire_dist\\\",\\r\\n    \\\"coord_x\\\", \\\"coord_y\\\",\\r\\n]\\r\\nNCH = len(CHANNELS)\\r\\n# (u, v) channel pairs that must rotate with the image during dihedral augmentation.\\r\\nPAIRS = ((12, 13), (19, 20), (25, 26))\\r\\nPREV_FIRE = 28\\r\\nDYNAMIC = {\\\"ndvi\\\", \\\"evi2\\\", \\\"precip\\\", \\\"wind_speed\\\", \\\"wind_dir_sin\\\", \\\"wind_dir_cos\\\", \\\"tmin\\\", \\\"tmax\\\",\\r\\n           \\\"erc\\\", \\\"sph\\\", \\\"fc_wind_speed\\\", \\\"fc_dir_sin\\\", \\\"fc_dir_cos\\\", \\\"fc_temp\\\"}\\r\\nDYN_IDX = [CHANNELS.index(c) for c in CHANNELS if c in DYNAMIC]\\r\\nSTA_IDX = [i for i in range(len(CHANNELS)) if i not in DYN_IDX]\\r\\n\\r\\n\\r\\ndef _geo(a: np.ndarray, k: int) -> np.ndarray:\\r\\n    if k & 1:\\r\\n        a = a[..., ::-1]\\r\\n    if k & 2:\\r\\n        a = a[..., ::-1, :]\\r\\n    if k & 4:\\r\\n        a = np.rot90(a, 1, axes=(-2, -1))\\r\\n    return a\\r\\n\\r\\n\\r\\n_JAC: dict[int, np.ndarray] = {}\\r\\n\\r\\n\\r\\ndef jac(k: int) -> np.ndarray:\\r\\n    \\\"\\\"\\\"2x2 rotation part of transform k, for (u=east, v=south) vector channels.\\\"\\\"\\\"\\r\\n    if k not in _JAC:\\r\\n        n = 5\\r\\n        X = np.tile(np.arange(n, dtype=np.float32), (n, 1))\\r\\n        Y = np.tile(np.arange(n, dtype=np.float32)[:, None], (1, n))\\r\\n        Xp, Yp = _geo(X, k), _geo(Y, k)\\r\\n        _JAC[k] = np.array([\\r\\n            [(Xp[2, 3] - Xp[2, 1]) / 2, (Xp[3, 2] - Xp[1, 2]) / 2],\\r\\n            [(Yp[2, 3] - Yp[2, 1]) / 2, (Yp[3, 2] - Yp[1, 2]) / 2],\\r\\n        ], dtype=np.float32)\\r\\n    return _JAC[k]\\r\\n\\r\\n\\r\\ndef dihedral(x: np.ndarray, y: np.ndarray, valid: np.ndarray, prev: np.ndarray, k: int):\\r\\n    x = _geo(x, k).copy()\\r\\n    J = jac(k)\\r\\n    for u, v in PAIRS:\\r\\n        uu, vv = x[u].copy(), x[v].copy()\\r\\n        x[u] = J[0, 0] * uu + J[0, 1] * vv\\r\\n        x[v] = J[1, 0] * uu + J[1, 1] * vv\\r\\n    return x, _geo(y, k).copy(), _geo(valid, k).copy(), _geo(prev, k).copy()\\r\\n\\r\\n\\r\\ndef boundary_of(y: np.ndarray) -> np.ndarray:\\r\\n    p = np.pad(y, 1)\\r\\n    core = p[0:-2, 1:-1] * p[1:-1, 1:-1] * p[2:, 1:-1] * p[1:-1, 0:-2] * p[1:-1, 2:]\\r\\n    return (y - (core > 0.5)).astype(np.float32)\\r\\n\\r\\n\\r\\ndef _pf_aug(x: np.ndarray, mode: str, rng: np.random.Generator) -> None:\\r\\n    \\\"\\\"\\\"TD-FusionUNet's `customized preprocessing` on the input channels, in place.\\r\\n\\r\\n    `margin`: replace the strict 0/1 mask with U(0.01,0.03) / U(0.8,0.99).\\r\\n    `smooth`: Gaussian-mixture softening (sigma 0.4 and 0.8) of the mask and wind speed.\\r\\n    Both were reported to help a lot on sparse masks; cheap enough to test here.\\r\\n    \\\"\\\"\\\"\\r\\n    if mode in (\\\"margin\\\", \\\"both\\\"):\\r\\n        pf = x[PREV_FIRE]\\r\\n        n = rng.random(pf.shape).astype(np.float16)\\r\\n        x[PREV_FIRE] = np.where(pf > 0.5, 0.8 + 0.19 * n, 0.01 + 0.02 * n)\\r\\n    if mode in (\\\"smooth\\\", \\\"both\\\"):\\r\\n        from scipy import ndimage as ndi\\r\\n\\r\\n        for ch in (PREV_FIRE, CHANNELS.index(\\\"wind_speed\\\")):  # pre-fire mask, wind speed\\r\\n            a = x[ch].astype(np.float32)\\r\\n            x[ch] = (0.5 * (ndi.gaussian_filter(a, 0.4) + ndi.gaussian_filter(a, 0.8))).astype(np.float16)\\r\\n\\r\\n\\r\\ndef cache(base: str | Path, split: str = \\\"train\\\"):\\r\\n    \\\"\\\"\\\"Memory-map the raw cache files written by `wf.build`.\\r\\n\\r\\n    Raw flat files instead of .npy: the builder appends shard by shard and does\\r\\n    not know the final row count, so a header it could never rewrite.\\r\\n    \\\"\\\"\\\"\\r\\n    base = Path(base) / split\\r\\n\\r\\n    def open_(name, tail, dtype):\\r\\n        p = base / name\\r\\n        n = p.stat().st_size // (int(np.prod(tail)) * np.dtype(dtype).itemsize)\\r\\n        return np.memmap(p, dtype=dtype, mode=\\\"r\\\", shape=(n, *tail))\\r\\n\\r\\n    return (open_(\\\"features.f16\\\", (NCH, 64, 64), np.float16),\\r\\n            open_(\\\"fire_class.u8\\\", (64, 64), np.uint8),\\r\\n            open_(\\\"prev_class.u8\\\", (64, 64), np.uint8))\\r\\n\\r\\n\\r\\ndef split_indices(root: str | Path, split: str, n: int | None, seed: int = 0) -> np.ndarray:\\r\\n    total = (Path(root) / split / \\\"fire_class.u8\\\").stat().st_size // (64 * 64)\\r\\n    if n is None or n >= total:\\r\\n        return np.arange(total)\\r\\n    rng = np.random.default_rng(seed)\\r\\n    return np.sort(rng.choice(total, size=n, replace=False))\\r\\n\\r\\n\\r\\nclass FireDataset(Dataset):\\r\\n    def __init__(\\r\\n        self,\\r\\n        root: str | Path,\\r\\n        split: str = \\\"train\\\",\\r\\n        indices: np.ndarray | None = None,\\r\\n        augment: bool = False,\\r\\n        prev_dropout: float = 0.0,\\r\\n        seed: int = 0,\\r\\n        aux: bool = True,\\r\\n        in_memory: bool = False,\\r\\n        prep: str = \\\"base\\\",\\r\\n        drop: str = \\\"\\\",\\r\\n        pf_aug: str = \\\"\\\",\\r\\n        soft: float = 0.0,\\r\\n        ext: np.ndarray | None = None,\\r\\n    ) -> None:\\r\\n        base = Path(root) / split\\r\\n        x, fire, prev = cache(root, split)\\r\\n        sel = np.arange(len(fire)) if indices is None else np.asarray(indices)\\r\\n        if in_memory:\\r\\n            # Screening subsets are scattered over a 19 GB memmap on a slow\\r\\n            # overlay FS: re-reading them every epoch cost ~20 s/epoch.\\r\\n            self.x, self.fire, self.prev = x[sel], fire[sel], prev[sel]\\r\\n            self.idx = np.arange(len(sel))\\r\\n        else:\\r\\n            self.x, self.fire, self.prev = x, fire, prev\\r\\n            self.idx = sel\\r\\n        self.augment = augment\\r\\n        self.prev_dropout = prev_dropout\\r\\n        self.aux = aux\\r\\n        self.prep = prep\\r\\n        self.drop = drop\\r\\n        self.pf_aug = pf_aug\\r\\n        self.soft = soft\\r\\n        if ext is not None and (ext.ndim != 2 or len(ext) != len(fire) or not np.isfinite(ext).all()):\\r\\n            raise ValueError(\\\"External features must be finite and aligned with all cache rows\\\")\\r\\n        self.ext = ext[sel] if ext is not None and in_memory else ext\\r\\n        self.rng = np.random.default_rng(seed)\\r\\n\\r\\n    def __len__(self) -> int:\\r\\n        return len(self.idx)\\r\\n\\r\\n    def __getitem__(self, i: int):\\r\\n        j = int(self.idx[i])\\r\\n        # float16 across the worker boundary: half the IPC, cast on GPU instead.\\r\\n        x = np.array(self.x[j], dtype=np.float16)\\r\\n        fire = self.fire[j].astype(np.int16)\\r\\n        y = (fire >= 7).astype(np.float32)\\r\\n        valid = ((fire != 4) & (fire != 6)).astype(np.float32)\\r\\n        prev = (self.prev[j] >= 7).astype(np.float32)\\r\\n        if self.augment:\\r\\n            x, y, valid, prev = dihedral(x, y, valid, prev, int(self.rng.integers(8)))\\r\\n            if self.pf_aug:\\r\\n                _pf_aug(x, self.pf_aug, self.rng)\\r\\n            if self.prev_dropout and self.rng.random() < self.prev_dropout:\\r\\n                x[PREV_FIRE] = 0.0\\r\\n        if self.soft:\\r\\n            # soft target: the ±1 px offsets that dominate small-patch Dice stop\\r\\n            # being treated as full errors (soft-label Jaccard, arXiv 2302.05666)\\r\\n            from scipy import ndimage as ndi\\r\\n\\r\\n            y = ndi.gaussian_filter(y, self.soft).astype(np.float32)\\r\\n        if self.aux:\\r\\n            aux = np.stack([boundary_of(y), y * (1.0 - prev), prev])\\r\\n        else:\\r\\n            aux = np.zeros((1, 1, 1), np.float32)\\r\\n        if self.prep != \\\"base\\\" or self.drop:\\r\\n            from .prep import DROPS, extras\\r\\n\\r\\n            extra = extras(x, self.prep)\\r\\n            keep = [i for i in range(x.shape[0]) if i not in DROPS.get(self.drop, [])]\\r\\n            x = np.concatenate([x[keep], extra], 0)\\r\\n        if self.ext is None:\\r\\n            return (torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(valid),\\r\\n                    torch.from_numpy(aux))\\r\\n        return (torch.from_numpy(x), torch.from_numpy(y), torch.from_numpy(valid),\\r\\n                torch.from_numpy(aux), torch.from_numpy(self.ext[j]))\\r\\n\\r\\n\\r\\ndef load_ext(root: str | Path, split: str) -> np.ndarray | None:\\r\\n    \\\"\\\"\\\"Per-row external features written by `wf.ext`; None when the cache has none.\\\"\\\"\\\"\\r\\n    p = Path(root) / split / \\\"ext.npy\\\"\\r\\n    return np.load(p).astype(np.float32) if p.exists() else None\\r\\n\",\"wf/tsf.py\":\"\\\"\\\"\\\"TS-SatFire (HuggingFace `SamuelWu318/ts-satfire`) -> the float16 patch cache.\\r\\n\\r\\nPer event (ROI) the dataset stores one directory per day:\\r\\n\\r\\n  VIIRS_Day    8 bands   I1 I2 I3 I4 I5 M11, active-fire temperature (NaN off fire),\\r\\n                         burned-area product (>0 where burned)\\r\\n  VIIRS_Night  2 bands   night I4, I5\\r\\n  FirePred    19 bands   NDVI_last, EVI2_last, precip, wind speed/dir, tmin, tmax,\\r\\n                         ERC, sph, slope, aspect, elevation, PDSI, LC_Type1,\\r\\n                         precip sum, forecast wind speed/dir, forecast temp/sph\\r\\n\\r\\nNone of the tifs carry a usable transform (scale 0), only matching bounds, so\\r\\nlayers are resampled onto the day grid by bounds and then cropped.\\r\\n\\r\\nTarget: next-day *progression*, i.e. the pixels that are burned in the\\r\\ncumulative mask of day t+1 but not in that of day t — the label TS-SatFire\\r\\ndefines for this task. Crops are centred on progression pixels, so every row\\r\\nhas fire, exactly like the Kaggle cache this repo was built on.\\r\\n\\r\\nUsage (Colab):\\r\\n    python -m wf.tsf --download                 # ~40 GB, parallel\\r\\n    python -m wf.tsf --out data/tsf --per-day 16\\r\\n\\\"\\\"\\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport argparse\\r\\nimport csv\\r\\nimport shutil\\r\\nfrom multiprocessing import Pool\\r\\nfrom pathlib import Path\\r\\n\\r\\nimport numpy as np\\r\\nfrom scipy import ndimage as ndi\\r\\n\\r\\nREPO = \\\"SamuelWu318/ts-satfire\\\"\\r\\nCACHE = \\\"/content/tsf\\\"\\r\\nPATTERNS = [f\\\"ts-satfire/*/{d}/*.tif\\\" for d in (\\\"VIIRS_Day\\\", \\\"VIIRS_Night\\\", \\\"FirePred\\\")] + [\\\"roi/*.csv\\\"]\\r\\n\\r\\n# official split (dataset_gen_pred.py): 2017-2020 fires minus these ids, 2021 held out\\r\\nVAL = {\\\"20568194\\\", \\\"20701026\\\", \\\"20562846\\\", \\\"20700973\\\", \\\"24462610\\\", \\\"24462788\\\", \\\"24462753\\\",\\r\\n       \\\"24103571\\\", \\\"21998313\\\", \\\"21751303\\\", \\\"22141596\\\", \\\"21999381\\\", \\\"23301962\\\", \\\"22712904\\\",\\r\\n       \\\"22713339\\\"}\\r\\n\\r\\n# aux band index -> our channel (0-based within the 19)\\r\\nAUX = dict(ndvi=0, evi2=1, pr=2, ws=3, wdir=4, tmin=5, tmax=6, erc=7, sph=8, slope=9,\\r\\n           aspect=10, elev=11, pdsi=12, lulc=13, pr_sum=14, fws=15, fdir=16, ftmp=17, fsph=18)\\r\\nCLIP = {\\\"pr\\\": (0, 100), \\\"pr_sum\\\": (0, 300), \\\"ws\\\": (0, 60), \\\"fws\\\": (0, 60), \\\"tmin\\\": (180, 340),\\r\\n        \\\"tmax\\\": (180, 350), \\\"ftmp\\\": (0, 60), \\\"erc\\\": (0, 150), \\\"sph\\\": (0, 0.06),\\r\\n        \\\"fsph\\\": (0, 0.06), \\\"slope\\\": (0, 60), \\\"elev\\\": (0, 5000), \\\"pdsi\\\": (-10, 10),\\r\\n        \\\"lulc\\\": (1, 17)}  # slope is degrees, ftmp is Celsius in this copy\\r\\nSIZE = 64\\r\\nSTRIDE = 16  # centre-bucket size when picking distinct crops, in pixels\\r\\n\\r\\n\\r\\ndef _read(path: Path):\\r\\n    import rasterio as rio  # only the converter needs it; the trainers do not\\r\\n\\r\\n    with rio.open(path) as r:\\r\\n        return r.read().astype(np.float32), r.bounds, r.shape\\r\\n\\r\\n\\r\\ndef _on_grid(a, src_shape, dst_shape):\\r\\n    \\\"\\\"\\\"Resample a layer onto the day grid. The layers share bounds, so a plain\\r\\n    resize is the right transform; the tifs carry no usable affine to warp with.\\\"\\\"\\\"\\r\\n    if src_shape == dst_shape:\\r\\n        return a\\r\\n    zy, zx = dst_shape[0] / src_shape[0], dst_shape[1] / src_shape[1]\\r\\n    out = np.stack([ndi.zoom(b, (zy, zx), order=1) for b in a])\\r\\n    padded = np.zeros((a.shape[0], *dst_shape), np.float32)\\r\\n    h, w = min(out.shape[1], dst_shape[0]), min(out.shape[2], dst_shape[1])\\r\\n    padded[:, :h, :w] = out[:, :h, :w]\\r\\n    return padded\\r\\n\\r\\n\\r\\ndef _fill(a: np.ndarray) -> np.ndarray:\\r\\n    \\\"\\\"\\\"NaN-safe per-band fill: the masked-off pixels are a big fraction of a layer.\\\"\\\"\\\"\\r\\n    out = np.empty_like(a)\\r\\n    for i, b in enumerate(a):\\r\\n        ok = np.isfinite(b)\\r\\n        out[i] = np.where(ok, b, np.median(b[ok]) if ok.any() else 0.0)\\r\\n    return out\\r\\n\\r\\n\\r\\ndef _band(a: np.ndarray, name: str) -> np.ndarray:\\r\\n    b = a[AUX[name]]\\r\\n    lo, hi = CLIP.get(name, (-1e9, 1e9))\\r\\n    return np.clip(b, lo, hi)\\r\\n\\r\\n\\r\\ndef _sincos(deg: np.ndarray):\\r\\n    t = np.deg2rad(deg)\\r\\n    return np.sin(t).astype(np.float32), np.cos(t).astype(np.float32)\\r\\n\\r\\n\\r\\ndef _channels(day, night, aux, prev, box):\\r\\n    \\\"\\\"\\\"One 33 x SIZE x SIZE patch. `prev` is the cumulative fire mask of day t.\\\"\\\"\\\"\\r\\n    r0, c0 = box\\r\\n    sl = (slice(r0, r0 + SIZE), slice(c0, c0 + SIZE))\\r\\n    d = _fill(day)[:, sl[0], sl[1]]\\r\\n    n = _fill(night)[:, sl[0], sl[1]]\\r\\n    a = _fill(aux)[:, sl[0], sl[1]]\\r\\n    p = prev[sl].astype(np.float32)\\r\\n\\r\\n    i1, i2, i3 = d[0] / 100.0, d[1] / 100.0, d[2] / 100.0\\r\\n    ndvi = (i2 - i1) / (i2 + i1 + 1e-3)\\r\\n    evi2 = 2.5 * (i2 - i1) / (i2 + 2.4 * i1 + 1.0)\\r\\n    asp_s, asp_c = _sincos(_band(a, \\\"aspect\\\"))\\r\\n    wd_s, wd_c = _sincos(_band(a, \\\"wdir\\\"))\\r\\n    fd_s, fd_c = _sincos(_band(a, \\\"fdir\\\"))\\r\\n    ws = _band(a, \\\"ws\\\")\\r\\n    yy, xx = np.mgrid[0:SIZE, 0:SIZE].astype(np.float32)\\r\\n    inside = ndi.distance_transform_edt(p > 0.5)\\r\\n    outside = ndi.distance_transform_edt(p <= 0.5)\\r\\n    dist = np.clip(np.where(p > 0.5, -inside, outside) * 0.1, -0.3, 1.0)\\r\\n    return np.stack([\\r\\n        d[0], d[1], d[2], d[3], d[4], d[5], n[0], n[1],\\r\\n        ndvi, evi2, _band(a, \\\"pr_sum\\\"), ws, wd_s, wd_c,\\r\\n        _band(a, \\\"tmin\\\"), _band(a, \\\"tmax\\\"), _band(a, \\\"erc\\\"), _band(a, \\\"sph\\\"),\\r\\n        _band(a, \\\"slope\\\"), asp_s, asp_c, _band(a, \\\"elev\\\"), _band(a, \\\"pdsi\\\"), _band(a, \\\"lulc\\\"),\\r\\n        _band(a, \\\"fws\\\"), fd_s, fd_c, _band(a, \\\"ftmp\\\"),\\r\\n        p, ndi.binary_dilation(p > 0.5).astype(np.float32), dist,\\r\\n        (xx - 31.5) / 31.5, (yy - 31.5) / 31.5,\\r\\n    ]).astype(np.float32)\\r\\n\\r\\n\\r\\ndef _centres(new: np.ndarray, k: int, rng) -> list[tuple[int, int]]:\\r\\n    \\\"\\\"\\\"Up to k crop origins on distinct STRIDExSTRIDE buckets of progression pixels.\\\"\\\"\\\"\\r\\n    ys, xs = np.nonzero(new)\\r\\n    if len(ys) == 0:\\r\\n        return []\\r\\n    order = rng.permutation(len(ys))\\r\\n    seen, out = set(), []\\r\\n    for i in order:\\r\\n        by, bx = ys[i] // STRIDE, xs[i] // STRIDE\\r\\n        if (by, bx) in seen:\\r\\n            continue\\r\\n        seen.add((by, bx))\\r\\n        out.append((int(np.clip(ys[i] - SIZE // 2, 0, new.shape[0] - SIZE)),\\r\\n                    int(np.clip(xs[i] - SIZE // 2, 0, new.shape[1] - SIZE))))\\r\\n        if len(out) >= k:\\r\\n            break\\r\\n    return out\\r\\n\\r\\n\\r\\ndef _by_date(d: Path) -> dict[str, Path]:\\r\\n    return {p.name.split(\\\"_\\\")[0]: p for p in d.glob(\\\"*.tif\\\")}\\r\\n\\r\\n\\r\\ndef convert(roi_dir: Path, per_day: int, rng) -> tuple[list, list, list, list]:\\r\\n    days = sorted((roi_dir / \\\"VIIRS_Day\\\").glob(\\\"*.tif\\\"))\\r\\n    nights = _by_date(roi_dir / \\\"VIIRS_Night\\\")\\r\\n    preds = _by_date(roi_dir / \\\"FirePred\\\")\\r\\n    days = [p for p in days if p.name.split(\\\"_\\\")[0] in preds]\\r\\n    if len(days) < 3:\\r\\n        return [], [], [], []\\r\\n    xs, fires, prevs, meta = [], [], [], []\\r\\n    cum = None\\r\\n    cur = None\\r\\n    for i in range(len(days) - 1):\\r\\n        f0, f1 = days[i], days[i + 1]\\r\\n        day0, b, shape = _read(f0) if cur is None else cur\\r\\n        day1, b1, s1 = _read(f1)\\r\\n        if cum is None:\\r\\n            cum = np.isfinite(day0[6]) | (np.nan_to_num(day0[7]) > 0)\\r\\n        from datetime import datetime\\r\\n        delta = (datetime.strptime(f1.name.split(\\\"_\\\")[0].replace(\\\"-\\\", \\\"\\\"), \\\"%Y%m%d\\\") -\\r\\n                 datetime.strptime(f0.name.split(\\\"_\\\")[0].replace(\\\"-\\\", \\\"\\\"), \\\"%Y%m%d\\\")).days\\r\\n        m1 = (~np.isnan(day1[6])) | (np.nan_to_num(day1[7]) > 0)\\r\\n        nxt = m1 if cum is None else (cum | m1)\\r\\n        new = nxt if cum is None else (nxt & ~cum)\\r\\n        cur = (day1, b1, s1)\\r\\n        if delta != 1 or min(shape) < SIZE or new.sum() < 8:\\r\\n            cum = nxt\\r\\n            continue\\r\\n        key = f0.name.split(\\\"_\\\")[0]\\r\\n        night = nights.get(key)\\r\\n        if night is None:\\r\\n            night = np.zeros((2, *shape), np.float32)\\r\\n        else:\\r\\n            night, nb, ns = _read(night)\\r\\n            night = night[-2:] if night.shape[0] > 2 else night\\r\\n            night = _on_grid(night, ns, shape)\\r\\n        aux = _read(preds[key])\\r\\n        aux = _on_grid(aux[0], aux[2], shape)\\r\\n        prev = np.zeros(shape, bool) if cum is None else cum\\r\\n        for box in _centres(new, per_day, rng):\\r\\n            r0, c0 = box\\r\\n            sl = (slice(r0, r0 + SIZE), slice(c0, c0 + SIZE))\\r\\n            xs.append(_channels(day0, night, aux, prev, box))\\r\\n            fires.append(np.where(new[sl], 7, 5).astype(np.uint8))\\r\\n            prevs.append(np.where(prev[sl], 7, 5).astype(np.uint8))\\r\\n            # crop centre in lon/lat, for the external (late-fusion) join\\r\\n            lat = b.top - (r0 + SIZE / 2) * (b.top - b.bottom) / shape[0]\\r\\n            lon = b.left + (c0 + SIZE / 2) * (b.right - b.left) / shape[1]\\r\\n            meta.append((key, round(lat, 4), round(lon, 4)))\\r\\n        cum = nxt\\r\\n    return xs, fires, prevs, meta\\r\\n\\r\\n\\r\\ndef _split_ids(roi_csv: Path, cache: Path) -> dict[str, list[str]]:\\r\\n    def ids(name):\\r\\n        with (roi_csv / name).open(newline=\\\"\\\") as f:\\r\\n            return [r[\\\"Id\\\"] for r in csv.DictReader(f)]\\r\\n\\r\\n    test = ids(\\\"fire_all_test.csv\\\")\\r\\n    # every ROI on disk that is not held out, so the fires missing from the csv\\r\\n    # lists (the card says 179 events, the lists cover 162) still train\\r\\n    on_disk = sorted(p.name for p in (cache / \\\"ts-satfire\\\").iterdir() if p.is_dir())\\r\\n    train_csvs = [roi_csv / f\\\"us_fire_{year}_out_new.csv\\\" for year in (2017, 2018, 2019, 2020)]\\r\\n    if all(p.exists() for p in train_csvs):\\r\\n        official = {i for p in train_csvs for i in ids(p.name)}\\r\\n        train = [i for i in on_disk if i in official and i not in VAL and i not in set(test)]\\r\\n    else:\\r\\n        train = [i for i in on_disk if i not in VAL and i not in set(test)]\\r\\n    return {\\\"train\\\": train, \\\"eval\\\": sorted(VAL), \\\"test\\\": test}\\r\\n\\r\\n\\r\\ndef _job(spec: tuple[str, int, int]) -> tuple[str, list, list, list, list]:\\r\\n    roi, per_day, seed = spec\\r\\n    try:\\r\\n        xs, fires, prevs, meta = convert(Path(roi), per_day, np.random.default_rng(seed))\\r\\n        return Path(roi).name, xs, fires, prevs, meta\\r\\n    except Exception as exc:  # a truncated tif should cost one ROI, not the run\\r\\n        print(\\\"fail\\\", roi, repr(exc)[:140], flush=True)\\r\\n        return Path(roi).name, [], [], [], []\\r\\n\\r\\n\\r\\ndef main() -> None:\\r\\n    ap = argparse.ArgumentParser()\\r\\n    ap.add_argument(\\\"--download\\\", action=\\\"store_true\\\", help=\\\"fetch every VIIRS/FirePred tif (~40 GB)\\\")\\r\\n    ap.add_argument(\\\"--out\\\", default=\\\"data/tsf\\\")\\r\\n    ap.add_argument(\\\"--cache\\\", default=CACHE)\\r\\n    ap.add_argument(\\\"--roi-csv\\\", default=None, help=\\\"dir with fire_all.csv (default <cache>/roi)\\\")\\r\\n    ap.add_argument(\\\"--per-day\\\", type=int, default=16)\\r\\n    ap.add_argument(\\\"--limit\\\", type=int, default=0, help=\\\"only this many ROIs per split\\\")\\r\\n    ap.add_argument(\\\"--workers\\\", type=int, default=4)\\r\\n    ap.add_argument(\\\"--keep\\\", action=\\\"store_true\\\", help=\\\"do not delete tifs after converting\\\")\\r\\n    ap.add_argument(\\\"--splits\\\", default=\\\"train,eval,test\\\")\\r\\n    args = ap.parse_args()\\r\\n\\r\\n    cache = Path(args.cache)\\r\\n    if not args.download and not (cache / \\\"ts-satfire\\\").is_dir():\\r\\n        raise FileNotFoundError(f\\\"Expected attached dataset at {cache}/ts-satfire; use --cache to select its parent\\\")\\r\\n    if args.download:\\r\\n        from huggingface_hub import snapshot_download\\r\\n\\r\\n        snapshot_download(REPO, repo_type=\\\"dataset\\\", local_dir=str(cache),\\r\\n                          allow_patterns=PATTERNS, max_workers=16)\\r\\n        print(\\\"download complete\\\", flush=True)\\r\\n        if args.download:\\r\\n            return\\r\\n\\r\\n    from .build import _count, _flush, _write_norm\\r\\n\\r\\n    if any(Path(args.out).glob(\\\"*/features.f16\\\")):\\r\\n        raise FileExistsError(\\\"Output cache already exists; select a fresh --out to avoid duplicate rows\\\")\\r\\n    args.keep = args.keep or str(cache.resolve()).startswith(\\\"/kaggle/input\\\")\\r\\n    rng = np.random.default_rng(0)\\r\\n    splits = _split_ids(Path(args.roi_csv or cache / \\\"roi\\\"), cache)\\r\\n    selected_splits = args.splits.split(',')\\r\\n    pairs = [(s, r) for s in selected_splits for r in splits[s]]\\r\\n    if args.limit:\\r\\n        keep = {(s, r) for s in splits for r in splits[s][: args.limit]}\\r\\n        pairs = [p for p in pairs if p in keep]\\r\\n    todo = [p for p in pairs if (cache / \\\"ts-satfire\\\" / p[1] / \\\"VIIRS_Day\\\").exists()]\\r\\n    print(f\\\"{len(pairs)} ROIs, {len(pairs) - len(todo)} missing on disk\\\", flush=True)\\r\\n    jobs = [(str(cache / \\\"ts-satfire\\\" / r), args.per_day, int(rng.integers(1 << 30))) for _, r in todo]\\r\\n    with Pool(args.workers) as pool:\\r\\n        for i, ((split, roi), (_, xs, fires, prevs, meta)) in enumerate(zip(todo, pool.imap(_job, jobs, 1))):\\r\\n            if xs:\\r\\n                _flush(args.out, split, xs, fires, prevs)\\r\\n                d = Path(args.out) / split\\r\\n                new = not (d / \\\"meta.csv\\\").exists()\\r\\n                with (d / \\\"meta.csv\\\").open(\\\"a\\\", newline=\\\"\\\") as f:\\r\\n                    w = csv.writer(f)\\r\\n                    if new:\\r\\n                        w.writerow([\\\"roi\\\", \\\"date\\\", \\\"lat\\\", \\\"lon\\\"])\\r\\n                    w.writerows((roi, *m) for m in meta)\\r\\n            if not args.keep:\\r\\n                shutil.rmtree(cache / \\\"ts-satfire\\\" / roi, ignore_errors=True)\\r\\n            if (i + 1) % 10 == 0 or i + 1 == len(todo):\\r\\n                print(f\\\"[{i + 1}/{len(todo)}] {split} {roi}: {len(xs)} rows\\\", flush=True)\\r\\n    for split in (\\\"train\\\", \\\"eval\\\", \\\"test\\\"):\\r\\n        print(f\\\"{split}: {_count(args.out, split)} rows\\\", flush=True)\\r\\n\\r\\n    _write_norm(args.out)\\r\\n\\r\\n\\r\\nif __name__ == \\\"__main__\\\":\\r\\n    main()\\r\\n\",\"wf/ensemble.py\":\"\\\"\\\"\\\"Blend finished runs with TTA, tune the threshold on eval, test once.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\n\\nfrom . import athr\\nfrom .data import FireDataset\\nfrom .final import _targets, predict_tta\\nfrom .metrics import dice_per_sample, evaluate, sweep\\nfrom .train import _loader, build\\n\\n\\ndef _load_run(name: str, dev, root: str):\\n    ckpt = torch.load(Path(\\\"results/runs\\\") / name / \\\"best.pt\\\", map_location=dev)\\n    model, prep, drop = build(ckpt[\\\"cfg\\\"], root)\\n    model = model.to(dev).to(memory_format=torch.channels_last)\\n    model.load_state_dict(ckpt[\\\"model\\\"], strict=False)\\n    return model, prep, drop\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--runs\\\", required=True, help=\\\"comma separated run dir names under results/runs\\\")\\n    ap.add_argument(\\\"--split\\\", default=\\\"eval\\\")\\n    ap.add_argument(\\\"--tta\\\", type=int, default=8)\\n    ap.add_argument(\\\"--test\\\", action=\\\"store_true\\\")\\n    ap.add_argument(\\\"--weights\\\", default=None, help=\\\"comma separated blend weights\\\")\\n    ap.add_argument(\\\"--athr\\\", action=\\\"store_true\\\", help=\\\"per-sample thresholds from predicted area\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"results/ensemble.json\\\")\\n    args = ap.parse_args()\\n\\n    dev = torch.device(\\\"cuda\\\")\\n    dtype = torch.bfloat16\\n    root = \\\"data/processed\\\"\\n    names = args.runs.split(\\\",\\\")\\n    w = None if args.weights is None else np.array([float(x) for x in args.weights.split(\\\",\\\")])\\n    assert w is None or len(w) == len(names)\\n\\n    head = torch.load(Path(\\\"results/runs\\\") / names[0] / \\\"best.pt\\\", map_location=\\\"cpu\\\")[\\\"cfg\\\"][\\\"data\\\"]\\n    prep, drop = head.get(\\\"prep\\\", \\\"base\\\"), head.get(\\\"drop\\\", \\\"\\\")\\n\\n    ds = FireDataset(root, args.split, None, aux=False, prep=prep, drop=drop)\\n    tg, va = _targets(ds)\\n    ld = _loader(ds, 512, False, 4)\\n\\n    probs = []\\n    for name in names:\\n        cached = Path(\\\"results/runs\\\") / name / f\\\"{args.split}_prob.npy\\\"\\n        if cached.exists():\\n            p = np.load(cached)\\n            print(name, \\\"(cached)\\\", flush=True)\\n        else:\\n            model, _, _ = _load_run(name, dev, root)\\n            p = predict_tta(model, ld, dev, dtype, args.tta)[0]\\n            del model\\n            torch.cuda.empty_cache()\\n        probs.append(p)\\n        print(name, \\\"macro@0.5\\\", round(evaluate(p, tg, va, 0.5, full=False)[\\\"macro_dice\\\"], 4), flush=True)\\n\\n    blend = np.average(probs, axis=0, weights=w)\\n    thr, rows = sweep(blend, tg, va)\\n    rule = None\\n    if args.athr:\\n        rule = athr.fit(blend, tg, va)\\n        honest = athr.cv(blend, tg, va)[\\\"cv\\\"]\\n        global_dice = float(dice_per_sample((blend > thr) & va, tg & va).mean())\\n        print(f\\\"athr: global {global_dice:.4f} fit {athr.score(blend, tg, va, rule):.4f} cv {honest:.4f}\\\",\\n              flush=True)\\n        if honest <= global_dice + 0.002:  # the gain must survive held-out fitting\\n            print(\\\"athr: no honest gain, keeping the global threshold\\\", flush=True)\\n            rule = None\\n    res = {\\\"runs\\\": names, \\\"split\\\": args.split, \\\"tta\\\": args.tta, \\\"thr\\\": thr, \\\"sweep\\\": rows,\\n           \\\"athr\\\": rule, \\\"eval\\\": evaluate(blend, tg, va, athr.apply(blend, rule) if rule else thr, full=True)}\\n\\n    if args.test:\\n        te = FireDataset(root, \\\"test\\\", None, aux=False, prep=prep, drop=drop)\\n        ttg, tva = _targets(te)\\n        tld = _loader(te, 512, False, 4)\\n        tp = []\\n        for name in names:\\n            model, _, _ = _load_run(name, dev, root)\\n            tp.append(predict_tta(model, tld, dev, dtype, args.tta)[0])\\n            del model\\n            torch.cuda.empty_cache()\\n        tblend = np.average(tp, axis=0, weights=w)\\n        res[\\\"test\\\"] = evaluate(tblend, ttg, tva, athr.apply(tblend, rule) if rule else thr, full=True)\\n\\n    Path(args.out).write_text(json.dumps(res, indent=1))\\n    print(json.dumps({k: res[k] for k in (\\\"runs\\\", \\\"thr\\\", \\\"eval\\\")}, indent=1), flush=True)\\n    if args.test:\\n        print(json.dumps({\\\"test\\\": res[\\\"test\\\"]}, indent=1), flush=True)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/prep.py\":\"\\\"\\\"\\\"Derived input channels for the preprocessing sweep.\\n\\nEverything here is computed from a patch that has *already* been dihedral\\naugmented (see `FireDataset.__getitem__`), so no channel needs its own\\ntransform bookkeeping. Every channel is a rotation/flip scalar -- distance,\\nband, projection, dot product, isotropic blur -- which is why that works:\\nrotating the patch rotates the input to the function, and the output is the\\nrotated output.\\n\\nWhy these: `fire_dist` (channel 20) is a *clipped* signed distance,\\n`clip(px * 0.1, -0.3, 1.0)`, so a scar interior deeper than 3 px and a fire\\nfar away are both flat, and `fire_front` is just a 1-px dilation of\\n`prev_fire`. The channels below add the information the clip throws away plus\\nwind anisotropy, which is the physics of spread.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom scipy import ndimage as ndi\\n\\nPREV, FRONT, DIST = 28, 29, 30  # feature indices (see wf.data.CHANNELS)\\nWINDS = (12, 13)  # wind direction as (sin, cos) of the \\\"from\\\" angle\\n\\n\\ndef _pf(x: np.ndarray) -> np.ndarray:\\n    return x[PREV] > 0.5\\n\\n\\ndef _unit(x: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:\\n    \\\"\\\"\\\"Downwind unit vector: the wind channels point where the wind comes from.\\\"\\\"\\\"\\n    u, v = -x[WINDS[0]].astype(np.float32), -x[WINDS[1]].astype(np.float32)\\n    s = np.hypot(u, v) + 1e-6\\n    return u / s, v / s, s\\n\\n\\n_YY, _XX = np.mgrid[0:64, 0:64]\\n_XX = (_XX - 31.5).astype(np.float32)\\n_YY = (_YY - 31.5).astype(np.float32)\\n\\n\\ndef ch_edt_in(x):\\n    return (ndi.distance_transform_edt(_pf(x)) / 8.0).astype(np.float32)\\n\\n\\ndef ch_interior(x):\\n    return (ch_edt_in(x) * 8.0 > 1.5).astype(np.float32)\\n\\n\\ndef ch_edt_out(x):\\n    return (ndi.distance_transform_edt(~_pf(x)) / 32.0).astype(np.float32)\\n\\n\\ndef ch_near(x):\\n    return np.sqrt(np.clip(x[DIST], 0, 1)).astype(np.float32)\\n\\n\\ndef _blur(x, sigma):\\n    return ndi.gaussian_filter(-x[DIST].astype(np.float32), sigma)\\n\\n\\ndef ch_blur2(x):\\n    return _blur(x, 2.0)\\n\\n\\ndef ch_blur5(x):\\n    return _blur(x, 5.0)\\n\\n\\ndef ch_proj(x):\\n    u, v, _ = _unit(x)\\n    return ((_XX * u + _YY * v) / 16.0).astype(np.float32)\\n\\n\\ndef ch_proj_pos(x):\\n    return np.maximum(ch_proj(x), 0).astype(np.float32)\\n\\n\\ndef ch_proj_neg(x):\\n    return np.maximum(-ch_proj(x), 0).astype(np.float32)\\n\\n\\ndef ch_gdot(x):\\n    \\\"\\\"\\\"Approach rate: gradient of distance dotted with the downwind direction.\\\"\\\"\\\"\\n    u, v, _ = _unit(x)\\n    gy, gx = np.gradient(-x[DIST].astype(np.float32))\\n    return (gx * u + gy * v).astype(np.float32)\\n\\n\\ndef ch_aniso(x):\\n    \\\"\\\"\\\"Closeness (1 at the scar, 0 beyond 10 px) blurred 6 px along the wind and 1.5 across.\\\"\\\"\\\"\\n    if not _pf(x).any():  # nothing to spread from; the two rotations are the slowest ops here\\n        return np.zeros((64, 64), np.float32)\\n    u, v, _ = _unit(x)\\n    # rotate takes a scalar angle; wind is near-uniform inside one patch\\n    theta = float(np.degrees(np.arctan2(v.mean(), u.mean())))\\n    c = np.clip(1.0 - x[DIST].astype(np.float32), 0, 1)\\n    a = ndi.gaussian_filter(ndi.rotate(c, -theta, reshape=False, order=1), (1.5, 6.0))\\n    return ndi.rotate(a, theta, reshape=False, order=1).astype(np.float32)\\n\\n\\ndef ch_ring2(x):\\n    pf = _pf(x)\\n    return (ndi.binary_dilation(pf, iterations=2) & ~ndi.binary_dilation(pf, iterations=1)).astype(np.float32)\\n\\n\\ndef ch_pf_ndvi(x):\\n    return (_pf(x) * x[8]).astype(np.float32)\\n\\n\\ndef ch_pf_slope(x):\\n    return (_pf(x) * x[18]).astype(np.float32)\\n\\n\\ndef ch_pf_erc(x):\\n    return (_pf(x) * x[16]).astype(np.float32)\\n\\n\\nCH = {f.__name__[3:]: f for f in [\\n    ch_edt_in, ch_interior, ch_edt_out, ch_near, ch_blur2, ch_blur5,\\n    ch_proj, ch_proj_pos, ch_proj_neg, ch_gdot, ch_aniso, ch_ring2,\\n    ch_pf_ndvi, ch_pf_slope, ch_pf_erc,\\n]}\\n\\nSCAR = [\\\"edt_in\\\", \\\"interior\\\", \\\"ring2\\\", \\\"pf_ndvi\\\", \\\"pf_slope\\\", \\\"pf_erc\\\"]\\nCTX = [\\\"edt_out\\\", \\\"near\\\", \\\"blur2\\\", \\\"blur5\\\"]\\nWIND = [\\\"proj\\\", \\\"proj_pos\\\", \\\"proj_neg\\\", \\\"gdot\\\", \\\"aniso\\\"]\\n\\nPREPS: dict[str, list[str]] = {\\n    \\\"base\\\": [],\\n    \\\"ctx\\\": CTX,\\n    \\\"scar\\\": SCAR,\\n    \\\"wind\\\": WIND,\\n    \\\"scarwind\\\": SCAR + WIND,\\n    \\\"all\\\": CTX + SCAR + WIND,\\n    \\\"dist\\\": [\\\"edt_in\\\", \\\"edt_out\\\"],  # replaces the clipped fire_dist channel\\n}\\n\\n# dropped base channels: coordinates carry no information for a per-pixel decision\\nDROPS: dict[str, list[int]] = {\\\"nocoord\\\": [31, 32], \\\"lean\\\": [31, 32],\\n                               \\\"nodist\\\": [DIST], \\\"noscar\\\": [FRONT, DIST]}\\n\\n\\ndef extras(x: np.ndarray, prep: str) -> np.ndarray:\\n    e = np.stack([CH[c](x) for c in PREPS[prep]]) if PREPS[prep] else np.zeros((0, 64, 64), np.float32)\\n    # float16 across the worker boundary, same as the base channels\\n    return e.astype(np.float16)\\n\\n\\ndef _key(prep: str, drop: str) -> str:\\n    return f\\\"{prep}|{drop}\\\"\\n\\n\\ndef norm_for(root: str | Path, prep: str = \\\"base\\\", drop: str = \\\"\\\") -> tuple[list[float], list[float]]:\\n    \\\"\\\"\\\"Mean/std matching the channel layout the dataset will produce.\\\"\\\"\\\"\\n    d = json.loads((Path(root) / \\\"norm.json\\\").read_text())\\n    keep = [i for i in range(len(d[\\\"mean\\\"])) if i not in DROPS.get(drop, [])]\\n    mean = [d[\\\"mean\\\"][i] for i in keep]\\n    std = [d[\\\"std\\\"][i] for i in keep]\\n    if PREPS[prep]:\\n        path = Path(root) / \\\"norm_extra.json\\\"\\n        cache = json.loads(path.read_text()) if path.exists() else {}\\n        k = _key(prep, drop)\\n        if k not in cache:\\n            cache[k] = _measure(root, prep)\\n            path.write_text(json.dumps(cache, indent=1))\\n        mean += cache[k][\\\"mean\\\"]\\n        std += cache[k][\\\"std\\\"]\\n    return mean, std\\n\\n\\ndef _measure(root: str | Path, prep: str, n: int = 2000, seed: int = 0) -> dict:\\n    from .data import FireDataset\\n\\n    total = (Path(root) / \\\"train\\\" / \\\"fire_class.u8\\\").stat().st_size // (64 * 64)\\n    ds = FireDataset(root, \\\"train\\\", np.random.default_rng(seed).choice(total, min(n, total), replace=False), aux=False)\\n    vals = np.stack([extras(np.asarray(ds.x[i]), prep).astype(np.float32) for i in range(n)])\\n    return {\\\"mean\\\": vals.mean((0, 2, 3)).tolist(),\\n            \\\"std\\\": np.maximum(vals.std((0, 2, 3)), 1e-3).tolist()}\\n\",\"wf/decode.py\":\"\\\"\\\"\\\"Turn eval/test probabilities into a mask: blob-at-centroid shape prior vs threshold.\\n\\nMeasured on td_gate3_s1: the centroid of the thresholded mask is within 1.24 px\\n(median) of the GT centroid, and total probability mass is within 4% of total GT\\narea, but the mask at any threshold is fragmented, so per-patch Dice is 0.459.\\nReplacing the mask by a compact blob at *that centroid* is worth ~+0.04.\\n\\n    python -m wf.decode --run td_gate3_s1\\n\\n`k` is the blob area. `mass` = sum of the probabilities (calibrated, carpet\\nincluded); `px` = number of pixels above the threshold (biased small).\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom .data import FireDataset\\nfrom .final import _targets\\nfrom .metrics import dice_per_sample, sweep\\n\\nYS, XS = np.mgrid[0:64, 0:64]\\nYS = YS.astype(np.float32)\\nXS = XS.astype(np.float32)\\n\\n\\ndef _blob(cy: float, cx: float, k: int) -> np.ndarray:\\n    d = ((YS - cy) ** 2 + (XS - cx) ** 2).ravel()\\n    kk = int(np.clip(k, 1, 4096))\\n    return d <= np.partition(d, kk - 1)[kk - 1]\\n\\n\\ndef _centroid(mask: np.ndarray) -> tuple[float, float]:\\n    ys, xs = np.nonzero(mask)\\n    return float(ys.mean()), float(xs.mean())\\n\\n\\ndef decode(p: np.ndarray, thr: float = 0.65, mode: str = \\\"maskdisk\\\", alpha: float = 1.0,\\n           small: float = 1e9) -> np.ndarray:\\n    \\\"\\\"\\\"`small` only affects `hybrid`: below it the blob, above it the threshold.\\\"\\\"\\\"\\n    out = np.zeros(p.shape, bool)\\n    for i in range(len(p)):\\n        hard = p[i] > thr\\n        if mode == \\\"base\\\":\\n            out[i] = hard\\n            continue\\n        if not hard.any():\\n            continue\\n        mass = float(p[i].sum())\\n        if mode == \\\"hybrid\\\" and mass >= small:\\n            out[i] = hard\\n            continue\\n        cy, cx = _centroid(hard)\\n        k = round(alpha * (mass if mode != \\\"maskdisk_px\\\" else hard.sum()))\\n        out[i] = _blob(cy, cx, k).reshape(64, 64)\\n    return out\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--run\\\", required=True)\\n    ap.add_argument(\\\"--runs\\\", default=\\\"results/runs\\\")\\n    ap.add_argument(\\\"--split\\\", default=\\\"eval\\\")\\n    ap.add_argument(\\\"--root\\\", default=\\\"data/processed\\\")\\n    args = ap.parse_args()\\n\\n    out = Path(args.runs) / args.run\\n    p = np.load(out / f\\\"{args.split}_prob.npy\\\")\\n    ds = FireDataset(args.root, args.split, None, aux=False)\\n    tgt, valid = _targets(ds)\\n    g, v = tgt & valid, valid\\n    area = g.reshape(len(g), -1).sum(1)\\n    thr, _ = sweep(p, tgt, valid)\\n    base = p > thr\\n\\n    def score(masks, sel=None):\\n        if sel is None:\\n            return float(dice_per_sample(masks & v, g).mean())\\n        return float(dice_per_sample(masks & v[sel], g[sel]).mean())\\n\\n    print(f\\\"{args.run} {args.split}  thr {thr:.2f}  base {score(base):.4f}\\\")\\n    rows = {}\\n    for mode, alpha in [(\\\"maskdisk\\\", 0.9), (\\\"maskdisk\\\", 1.0), (\\\"maskdisk\\\", 1.1), (\\\"maskdisk\\\", 1.2),\\n                        (\\\"maskdisk_px\\\", 1.2), (\\\"maskdisk_px\\\", 1.3), (\\\"maskdisk_px\\\", 1.4)]:\\n        rows[f\\\"{mode} a={alpha}\\\"] = round(score(decode(p, float(thr), mode, alpha)), 4)\\n    for small in (10, 20, 40, 80):\\n        rows[f\\\"hybrid mass<{small}\\\"] = round(score(decode(p, float(thr), \\\"hybrid\\\", 1.0, small)), 4)\\n    print(json.dumps(rows, indent=1))\\n\\n    masks = decode(p, float(thr), \\\"maskdisk\\\", 1.0)\\n    for lo, hi in [(0, 9), (10, 29), (30, 99), (100, 10 ** 6)]:\\n        sel = (area >= lo) & (area <= hi)\\n        if not sel.any():\\n            continue\\n        hi_s = \\\"inf\\\" if hi >= 10 ** 6 else str(hi)\\n        print(f\\\"  area {lo:>3}-{hi_s:>3} n={sel.sum():>5}  base {score(base, sel):.4f}  \\\"\\n              f\\\"blob {score(masks, sel):.4f}\\\")\\n\\n    cy = np.array([_centroid(m)[0] if m.any() else 31.5 for m in base])\\n    cx = np.array([_centroid(m)[1] if m.any() else 31.5 for m in base])\\n    orc = np.stack([_blob(cy[i], cx[i], int(area[i])) for i in range(len(p))]).reshape(p.shape)\\n    print(f\\\"  oracle-area blob at predicted centroid: {score(orc):.4f}\\\")\\n\\n    best = decode(p, float(thr), \\\"maskdisk\\\", 1.0)\\n    np.save(out / f\\\"{args.split}_blob.npy\\\", best)\\n    (out / f\\\"{args.split}_decode.json\\\").write_text(json.dumps(rows, indent=1))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/metrics.py\":\"\\\"\\\"\\\"Metrics: macro/micro Dice, IoU, P/R, PR-AUC, boundary F1, hard Dice.\\n\\n`hard_dice` only scores pixels outside the radius-2 disk at the patch centre,\\nwhich removes the trivially centred blob and measures actual spread.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nfrom scipy.ndimage import distance_transform_edt\\n\\n\\ndef _disk(shape=(64, 64), r=2):\\n    yy, xx = np.ogrid[: shape[0], : shape[1]]\\n    return (yy - shape[0] // 2) ** 2 + (xx - shape[1] // 2) ** 2 <= r * r\\n\\n\\nOUTSIDE = ~_disk()\\n\\n\\ndef dice_per_sample(pred: np.ndarray, tgt: np.ndarray, empty_one: bool = True) -> np.ndarray:\\n    inter = np.logical_and(pred, tgt).sum(axis=(1, 2), dtype=np.float64)\\n    den = pred.sum(axis=(1, 2), dtype=np.float64) + tgt.sum(axis=(1, 2), dtype=np.float64)\\n    empty = 1.0 if empty_one else 0.0\\n    return np.where(den > 0, 2 * inter / np.maximum(den, 1), empty)\\n\\n\\ndef _edge(m: np.ndarray) -> np.ndarray:\\n    p = np.pad(m, 1)\\n    core = p[:-2, 1:-1] & p[1:-1, 1:-1] & p[2:, 1:-1] & p[1:-1, :-2] & p[1:-1, 2:]\\n    return m & ~core\\n\\n\\ndef boundary_f1(pred: np.ndarray, tgt: np.ndarray, tol: float = 2.0) -> float:\\n    ps, ts = [], []\\n    for p, t in zip(pred, tgt):\\n        if p.sum() == 0 or t.sum() == 0:\\n            continue\\n        ep, et = _edge(p), _edge(t)\\n        dp, dt = distance_transform_edt(~ep), distance_transform_edt(~et)\\n        ps.append((ep & (dt <= tol)).sum() / max(ep.sum(), 1))\\n        ts.append((et & (dp <= tol)).sum() / max(et.sum(), 1))\\n    if not ps:\\n        return 0.0\\n    ps, ts = np.array(ps), np.array(ts)\\n    f = 2 * ps * ts / np.maximum(ps + ts, 1e-9)\\n    return float(f.mean())\\n\\n\\ndef average_precision(prob: np.ndarray, tgt: np.ndarray, valid: np.ndarray, max_neg: int = 4_000_000,\\n                      seed: int = 0) -> float:\\n    p = prob.ravel()\\n    t = tgt.ravel().astype(bool)\\n    v = valid.ravel().astype(bool)\\n    p, t = p[v], t[v]\\n    neg = np.flatnonzero(~t)\\n    rng = np.random.default_rng(seed)\\n    if len(neg) > max_neg:\\n        neg = rng.choice(neg, max_neg, replace=False)\\n    keep = np.concatenate([neg, np.flatnonzero(t)])\\n    p, t = p[keep], t[keep]\\n    order = np.argsort(-p)\\n    t = t[order]\\n    tp = np.cumsum(t)\\n    prec = tp / np.arange(1, len(t) + 1)\\n    rec = tp / max(t.sum(), 1)\\n    ap = np.sum(np.diff(np.concatenate([[0.0], rec])) * prec)\\n    return float(ap)\\n\\n\\ndef topk_masks(prob: np.ndarray, k) -> np.ndarray:\\n    \\\"\\\"\\\"Per-sample top-`k` pixels of `prob`: the decode an area estimate implies.\\n\\n    Measured on td_gate3_s1: with the GT area this scores 0.554 versus 0.459 for\\n    the best global threshold, so area — not ranking — is the binding constraint.\\n    \\\"\\\"\\\"\\n    n, m = prob.shape[0], prob[0].size\\n    order = np.argsort(-prob.reshape(n, m), axis=1)\\n    k = np.broadcast_to(np.asarray(k, np.float64), (n,))\\n    out = np.zeros((n, m), bool)\\n    for i in range(n):\\n        out[i, order[i, : int(np.clip(k[i], 1, m))]] = True\\n    return out.reshape(prob.shape)\\n\\n\\ndef _score(pred: np.ndarray, tgt: np.ndarray, valid: np.ndarray, prob: np.ndarray,\\n           full: bool) -> dict:\\n    tg = tgt & valid\\n    d = dice_per_sample(pred, tg)\\n    inter = np.logical_and(pred, tg).sum()\\n    union = np.logical_or(pred, tg).sum()\\n    tp = inter\\n    out = {\\n        \\\"macro_dice\\\": float(d.mean()),\\n        \\\"micro_dice\\\": float(2 * inter / max(pred.sum() + tg.sum(), 1)),\\n        \\\"iou\\\": float(inter / max(union, 1)),\\n        \\\"precision\\\": float(tp / max(pred.sum(), 1)),\\n        \\\"recall\\\": float(tp / max(tg.sum(), 1)),\\n    }\\n    if full:\\n        hm = np.logical_and(pred, OUTSIDE)\\n        ht = np.logical_and(tg, OUTSIDE)\\n        # empty-outside counts as 0: this metric is about real spread, not the blob\\n        out[\\\"hard_dice\\\"] = float(dice_per_sample(hm, ht, empty_one=False).mean())\\n        out[\\\"boundary_f1\\\"] = boundary_f1(pred, tg)\\n        out[\\\"ap\\\"] = average_precision(prob, tgt, valid)\\n    return out\\n\\n\\ndef evaluate(prob: np.ndarray, tgt: np.ndarray, valid: np.ndarray, thr: float = 0.5,\\n             full: bool = True) -> dict:\\n    thr = np.asarray(thr, np.float32)\\n    return _score((prob > (thr[:, None, None] if thr.ndim else thr)) & valid, tgt, valid, prob, full)\\n\\n\\ndef evaluate_area(prob: np.ndarray, k, tgt: np.ndarray, valid: np.ndarray,\\n                  full: bool = True) -> dict:\\n    return _score(topk_masks(prob, k) & valid, tgt, valid, prob, full)\\n\\n\\ndef sweep_area(prob: np.ndarray, k, tgt: np.ndarray, valid: np.ndarray,\\n               gammas=None) -> tuple[float, list]:\\n    \\\"\\\"\\\"Global scale on the predicted area, the counterpart of a threshold sweep.\\\"\\\"\\\"\\n    gammas = np.arange(0.60, 1.46, 0.05) if gammas is None else gammas\\n    rows = [(float(g), float(dice_per_sample(topk_masks(prob, np.asarray(k, np.float64) * g) & valid,\\n                                            tgt & valid).mean())) for g in gammas]\\n    best = max(rows, key=lambda r: r[1])\\n    return best[0], rows\\n\\n\\ndef sweep(prob: np.ndarray, tgt: np.ndarray, valid: np.ndarray, thrs=None) -> tuple[float, list]:\\n    thrs = np.arange(0.20, 0.86, 0.05) if thrs is None else thrs\\n    rows = [(float(t), float(dice_per_sample((prob > t) & valid, tgt & valid).mean())) for t in thrs]\\n    best = max(rows, key=lambda r: r[1])\\n    return best[0], rows\\n\",\"wf/explore.py\":\"\\\"\\\"\\\"Kaggle-only architecture/loss/fusion search with grouped confirmation events.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport copy\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\nimport time\\n\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\n\\nfrom .data import FireDataset, cache, load_ext\\nfrom .final import predict_tta\\nfrom .metrics import dice_per_sample, evaluate, sweep\\nfrom .train import _loader, build, run\\n\\n\\ndef write_json(path, value):\\n    Path(path).write_text(json.dumps(value, indent=2))\\n\\n\\ndef grouped_root(source: Path, dest: Path):\\n    \\\"\\\"\\\"Reserve whole eval events before any search; never read test.\\\"\\\"\\\"\\n    if (dest / 'partition.json').exists():\\n        return\\n    dest.mkdir(parents=True, exist_ok=True)\\n    meta = pd.read_csv(source / 'eval/meta.csv', dtype={'roi': str})\\n    events = sorted(meta.roi.unique())\\n    if len(events) < 6:\\n        raise ValueError('Need at least six eval events for grouped confirmation')\\n    order = np.random.default_rng(2026).permutation(events)\\n    confirm = set(order[:max(3, len(events) // 3)])\\n    (dest / 'train').symlink_to((source / 'train').resolve(), target_is_directory=True)\\n    (dest / 'norm.json').write_text((source / 'norm.json').read_text())\\n    arrays = cache(source, 'eval')\\n    ext = load_ext(source, 'eval')\\n    for split, selected in [('eval', ~meta.roi.isin(confirm)), ('confirm', meta.roi.isin(confirm))]:\\n        d = dest / split\\n        d.mkdir(exist_ok=True)\\n        indices = np.flatnonzero(selected)\\n        for arr, name in zip(arrays, ['features.f16', 'fire_class.u8', 'prev_class.u8']):\\n            with (d / name).open('wb') as f:\\n                for start in range(0, len(indices), 16):\\n                    f.write(np.asarray(arr[indices[start:start + 16]]).tobytes())\\n        meta.iloc[indices].to_csv(d / 'meta.csv', index=False)\\n        if ext is not None:\\n            np.save(d / 'ext.npy', ext[indices])\\n    write_json(dest / 'partition.json', {'seed': 2026, 'confirm_events': sorted(confirm),\\n                                        'tuning_events': sorted(set(events) - confirm),\\n                                        'source': str(source), 'test_read': False})\\n\\n\\ndef candidates(weather: bool):\\n    td = {'trunk': 'td', 'branches': ['ht', 'dct', 'conv'], 'width': 16,\\n          'sizes': [64, 32, 16, 8]}\\n    space = {\\n        'td4': {'model': td},\\n        'td4_attention': {'model': {**td, 'attn': True}},\\n        'td4_wide_decoder': {'model': {**td, 'dec': 2}},\\n        'td4_w24': {'model': {**td, 'width': 24}},\\n        'td4_d2': {'model': {**td, 'depth': 2}},\\n        'td4_deep_supervision': {'model': {**td, 'ds': True}, 'loss': {'ds_weight': 0.3}},\\n        'unet': {'model': {'trunk': 'unet', 'widths': [32, 64, 128, 256], 'depths': [2]*4}},\\n        'hrnet': {'model': {'trunk': 'hrnet', 'width': 24, 'stages': 3}},\\n        'dual_fuel_weather': {'model': {'trunk': 'dual', 'widths': [32, 64, 128], 'depths': [2]*3}},\\n    }\\n    for name, arch, encoder in [('unetpp_r18', 'UnetPlusPlus', 'resnet18'),\\n                                 ('unet_r34', 'Unet', 'resnet34'),\\n                                 ('unet_mitb0', 'Unet', 'mit_b0'),\\n                                 ('unet_mitb1', 'Unet', 'mit_b1'),\\n                                 ('unet_effb1', 'Unet', 'efficientnet-b1'),\\n                                 ('unet_convnext', 'Unet', 'tu-convnext_tiny')]:\\n        space[name] = {'model': {'smp': {'arch': arch, 'encoder': encoder, 'weights': 'imagenet'}}}\\n    if weather:\\n        space['td4_weather_scalar'] = {'model': {**td, 'late': 8}}\\n        space['td4_weather_spatial'] = {'model': {**td, 'late': 8, 'late_mode': 'spatial'}}\\n    return space\\n\\n\\ndef configuration(root, patch, epochs, seed, n_train, max_seconds):\\n    cfg = {'seed': seed, 'data': {'root': str(root), 'in_memory': False, 'prev_dropout': 0.2},\\n           'model': {}, 'loss': {'name': 'bce_dice', 'pos_weight': 5., 'dice_weight': 1.},\\n           'train': {'epochs': epochs, 'n_train': n_train, 'batch_size': 32,\\n                     'eval_batch_size': 64, 'workers': 2, 'eval_workers': 2,\\n                     'amp': 'fp16', 'lr': 0.002, 'patience': 12, 'warmup_epochs': 2,\\n                     'checkpoint_sweep': True, 'max_seconds': max_seconds}}\\n    for section, values in patch.items():\\n        cfg[section].update(copy.deepcopy(values))\\n    return cfg\\n\\n\\ndef isolated(cfg, name, out):\\n    \\\"\\\"\\\"Isolate CUDA/OOM failures and resume completed runs without rerunning.\\\"\\\"\\\"\\n    summary = out / name / 'summary.json'\\n    if summary.exists():\\n        return json.loads(summary.read_text())\\n    jobs = out / 'jobs'\\n    jobs.mkdir(exist_ok=True, parents=True)\\n    path = jobs / f'{name}.json'\\n    write_json(path, {'cfg': cfg, 'name': name, 'out': str(out)})\\n    with (jobs / f'{name}.log').open('w') as log:\\n        proc = subprocess.Popen([sys.executable, '-u', '-m', 'wf.explore', '--job', str(path)],\\n                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\\n        for line in proc.stdout:\\n            print(line, end='', flush=True)\\n            log.write(line)\\n            log.flush()\\n        code = proc.wait()\\n    if code or not summary.exists():\\n        with (out / 'failures.jsonl').open('a') as f:\\n            f.write(json.dumps({'name': name, 'exit': code, 'log': str(jobs / f'{name}.log')}) + '\\\\n')\\n        return None\\n    return json.loads(summary.read_text())\\n\\n\\ndef prediction(summary, root, split, tta):\\n    cfg = summary['cfg']\\n    device = torch.device('cuda')\\n    model, prep, drop = build(cfg, root)\\n    model.load_state_dict(torch.load(summary['_checkpoint'], map_location='cpu', weights_only=False)['model'], strict=False)\\n    model = model.to(device).to(memory_format=torch.channels_last)\\n    ds = FireDataset(root, split, aux=False, prep=prep, drop=drop,\\n                     ext=load_ext(root, split) if cfg['model'].get('late') else None)\\n    prob, _ = predict_tta(model, _loader(ds, 64, False, 2), device, torch.float16, tta)\\n    y = np.asarray(ds.fire) >= 7\\n    valid = ~np.isin(np.asarray(ds.fire), (4, 6))\\n    del model\\n    torch.cuda.empty_cache()\\n    return prob, y, valid\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser(description=__doc__)\\n    ap.add_argument('--job')\\n    ap.add_argument('--source', default='/kaggle/working/tsf_full')\\n    ap.add_argument('--out', default='/kaggle/working/exploration')\\n    ap.add_argument('--hours', type=float, default=8.)\\n    args = ap.parse_args()\\n    if args.job:\\n        job = json.loads(Path(args.job).read_text())\\n        run(job['cfg'], job['name'], job['out'])\\n        return\\n    if not Path('/kaggle/working').exists() or not torch.cuda.is_available():\\n        raise RuntimeError('Run this experiment inside Kaggle with GPU enabled')\\n    start = time.monotonic()\\n    deadline = start + args.hours * 3600\\n    out = Path(args.out)\\n    out.mkdir(exist_ok=True, parents=True)\\n    root = out / 'search_data'\\n    grouped_root(Path(args.source), root)\\n    weather = (root / 'train/ext.npy').exists()\\n    # Reject weather when geographic joins produced mostly missing observations.\\n    coverage = json.loads((Path(args.source) / 'weather_audit.json').read_text()) if (Path(args.source) / 'weather_audit.json').exists() else {}\\n    weather = weather and coverage.get('train_complete_fraction', 0) >= 0.1\\n    space = candidates(weather)\\n    successes = []\\n\\n    def attempt(name, patch, epochs, seed=0, n_train=12000, seconds=900):\\n        if time.monotonic() + 180 > deadline:\\n            return None\\n        cfg = configuration(root, patch, epochs, seed, n_train, min(seconds, max(60, deadline-time.monotonic()-120)))\\n        summary = isolated(cfg, name, out / 'runs')\\n        if summary:\\n            summary['_checkpoint'] = str(out / 'runs' / name / 'best.pt')\\n            successes.append(summary)\\n            write_json(out / 'leaderboard.json', sorted(successes, key=lambda s: s['val_at_best_thr']['macro_dice'], reverse=True))\\n        return summary\\n\\n    screened = []\\n    for name, patch in space.items():\\n        result = attempt('screen_' + name, patch, 18)\\n        if result:\\n            screened.append((name, patch, result))\\n    screened.sort(key=lambda r: r[2]['val_at_best_thr']['macro_dice'], reverse=True)\\n    if not screened:\\n        raise RuntimeError('All screening runs failed; see failures.jsonl and job logs')\\n\\n    # Loss/regularization controls on the best architecture, not arbitrary new models.\\n    winner_name, winner_patch, _ = screened[0]\\n    variants = [('default', {}), ('bce_pw1', {'name': 'bce_dice', 'pos_weight': 1.}),\\n                ('tversky_recall', {'name': 'focal_tversky', 'alpha': .4, 'beta': .6, 'gamma': 1.33}),\\n                ('tversky_precision', {'name': 'focal_tversky', 'alpha': .7, 'beta': .3, 'gamma': 1.33}),\\n                ('lovasz', {'name': 'lovasz'})]\\n    promoted = [(n, p, r) for n, p, r in screened[:3]]\\n    for suffix, loss in variants[1:]:\\n        patch = copy.deepcopy(winner_patch)\\n        patch.setdefault('loss', {}).update(loss)\\n        result = attempt(f'loss_{winner_name}_{suffix}', patch, 18)\\n        if result:\\n            promoted.append((f'{winner_name}_{suffix}', patch, result))\\n    promoted.sort(key=lambda r: r[2]['val_at_best_thr']['macro_dice'], reverse=True)\\n\\n    finalists = []\\n    # Fully train the best three recipes, then repeat their seeds if budget permits.\\n    for seed in (0, 1, 2):\\n        for name, patch, _ in promoted[:3]:\\n            result = attempt(f'final_{name}_seed{seed}', patch, 60, seed, None, 2400)\\n            if result:\\n                finalists.append(result)\\n    if not finalists:\\n        finalists = [r[2] for r in promoted[:3]]\\n\\n    # Pick single model vs equal-weight diverse ensemble and TTA on tuning only.\\n    finalists.sort(key=lambda r: r['val_at_best_thr']['macro_dice'], reverse=True)\\n    finalists = finalists[:6]\\n    best = None\\n    for tta in (1, 4):\\n        probs = []\\n        for s in finalists:\\n            p, y, v = prediction(s, root, 'eval', tta)\\n            probs.append(p)\\n        for count in range(1, len(probs) + 1):\\n            blend = np.mean(probs[:count], axis=0)\\n            threshold, _ = sweep(blend, y, v, np.arange(.1, .91, .025))\\n            metrics = evaluate(blend, y, v, threshold, full=False)\\n            recipe = {'tta': tta, 'threshold': threshold, 'members': finalists[:count], 'tuning': metrics}\\n            if best is None or metrics['macro_dice'] > best['tuning']['macro_dice']:\\n                best = recipe\\n        del probs\\n    write_json(out / 'frozen_recipe.json', best)\\n    # Confirm exactly this recipe once on reserved events, with its fixed threshold.\\n    total = None\\n    for summary in best['members']:\\n        p, y, v = prediction(summary, root, 'confirm', best['tta'])\\n        total = p if total is None else total + p\\n    total /= len(best['members'])\\n    np.save(out / 'confirm_probability.npy', total.astype(np.float16))\\n    report = {'tuning': best['tuning'], 'confirmation': evaluate(total, y, v, best['threshold']),\\n              'threshold': best['threshold'], 'tta': best['tta'],\\n              'members': [s['name'] for s in best['members']], 'test_evaluated': False,\\n              'weather_enabled': weather, 'weather_audit': coverage,\\n              'minutes': (time.monotonic()-start)/60}\\n    meta = pd.read_csv(root / 'confirm/meta.csv', dtype={'roi': str})\\n    per_patch = dice_per_sample((total > best['threshold']) & v, y & v)\\n    event_scores = pd.Series(per_patch).groupby(meta.roi).mean()\\n    samples = np.random.default_rng(0).choice(event_scores.to_numpy(),\\n                                             (2000, len(event_scores)), replace=True).mean(1)\\n    report['confirmation_per_event'] = event_scores.to_dict()\\n    report['confirmation_event_macro'] = float(event_scores.mean())\\n    report['confirmation_event_bootstrap_95'] = np.percentile(samples, [2.5, 97.5]).tolist()\\n    write_json(out / 'report.json', report)\\n    print('EXPLORATION_REPORT', json.dumps(report), flush=True)\\n\\n\\nif __name__ == '__main__':\\n    main()\\n\",\"wf/ext.py\":\"\\\"\\\"\\\"External daily weather per cached row, for late fusion.\\r\\n\\r\\nThe cache's weather block already comes from the FirePred (GRIDMET) bands, so the\\r\\nonly honest way to test \\\"does more weather data help\\\" is a different observation\\r\\nnetwork. Source: Kaggle `dianatofficial/global-daily-weather-1950-2026-analysis-layer`\\r\\n(GHCN-Daily core network, one wide parquet per year). That copy ships no station\\r\\ncoordinates, so they come from NOAA's `ghcnd-stations.txt`.\\r\\n\\r\\nPer row we take the K nearest stations that reported on the crop's date and\\r\\ncombine them by inverse distance. Eight features: tmax, tmin, prcp, 3-day prcp,\\r\\ntmax anomaly vs its own trailing week, diurnal range, nearest-station distance,\\r\\nand a missing flag. Scaled with train statistics and written as `ext.npy`\\r\\nalongside each split, aligned row-for-row with `meta.csv` from `wf.tsf`.\\r\\n\\r\\nUsage (Colab):\\r\\n    python -m wf.ext --ghcn x_ghcn --stations ghcnd-stations.txt --out data/tsf\\r\\n\\\"\\\"\\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport argparse\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\n\\r\\nimport numpy as np\\r\\nimport pandas as pd\\r\\nfrom scipy.spatial import cKDTree\\r\\n\\r\\nK = 4              # nearest reporting stations\\r\\nMAX_KM = 150.0     # beyond this a row counts as missing\\r\\nCOLS = [\\\"tmax\\\", \\\"tmin\\\", \\\"prcp\\\", \\\"prcp_3d\\\", \\\"tmax_anom7\\\", \\\"dtr\\\", \\\"nearest_km\\\", \\\"missing\\\"]\\r\\n\\r\\n\\r\\ndef _station_coords(path: Path) -> tuple[list[str], np.ndarray]:\\r\\n    if path.suffix in (\\\".parquet\\\", \\\".csv\\\"):\\r\\n        d = pd.read_parquet(path) if path.suffix == \\\".parquet\\\" else pd.read_csv(path)\\r\\n        names = {c.lower(): c for c in d.columns}\\r\\n        lat = next((names[k] for k in (\\\"latitude\\\", \\\"lat\\\") if k in names), None)\\r\\n        lon = next((names[k] for k in (\\\"longitude\\\", \\\"lon\\\", \\\"lng\\\") if k in names), None)\\r\\n        station = next((names[k] for k in (\\\"station_id\\\", \\\"id\\\", \\\"station\\\") if k in names), None)\\r\\n        if not all((lat, lon, station)):\\r\\n            raise ValueError(f\\\"Station coordinate schema not recognized: {list(d.columns)}\\\")\\r\\n        d = d.dropna(subset=[lat, lon, station])\\r\\n        return d[station].astype(str).tolist(), d[[lat, lon]].to_numpy(np.float64)\\r\\n    ids, ll = [], []\\r\\n    for line in path.read_text().splitlines():\\r\\n        ids.append(line[:11])\\r\\n        ll.append((float(line[12:20]), float(line[21:30])))\\r\\n    return ids, np.asarray(ll, np.float64)\\r\\n\\r\\n\\r\\ndef _grid(ghcn: Path, years: range, ids: list[str], index: dict[str, int],\\r\\n          days: pd.DatetimeIndex) -> dict[str, np.ndarray]:\\r\\n    \\\"\\\"\\\"station x day matrices of the raw fields, NaN where a station did not report.\\\"\\\"\\\"\\r\\n    n = len(ids)\\r\\n    mats = {c: np.full((n, len(days)), np.nan, np.float32) for c in (\\\"tmax\\\", \\\"tmin\\\", \\\"prcp\\\")}\\r\\n    for y in years:\\r\\n        f = ghcn / f\\\"ghcnd_wide_{y}.parquet\\\"\\r\\n        if not f.exists():\\r\\n            continue\\r\\n        d = pd.read_parquet(f, columns=[\\\"station_id\\\", \\\"date\\\", \\\"tmax_c\\\", \\\"tmin_c\\\", \\\"prcp_mm\\\"])\\r\\n        st = d.station_id.map(index)\\r\\n        day = (pd.to_datetime(d.date) - days[0]).dt.days\\r\\n        ok = st.notna() & day.between(0, len(days) - 1)\\r\\n        for name, col in ((\\\"tmax\\\", \\\"tmax_c\\\"), (\\\"tmin\\\", \\\"tmin_c\\\"), (\\\"prcp\\\", \\\"prcp_mm\\\")):\\r\\n            mats[name][st[ok].astype(int), day[ok].astype(int)] = d[col][ok].to_numpy(np.float32)\\r\\n    return mats\\r\\n\\r\\n\\r\\ndef _rolling(a: np.ndarray, w: int, how: str) -> np.ndarray:\\r\\n    df = pd.DataFrame(a.T).rolling(w, min_periods=1)\\r\\n    return (df.sum() if how == \\\"sum\\\" else df.mean()).to_numpy(np.float32).T\\r\\n\\r\\n\\r\\ndef build(meta: pd.DataFrame, mats: dict[str, np.ndarray], ll: np.ndarray,\\r\\n          days: pd.DatetimeIndex) -> np.ndarray:\\r\\n    if len(ll) == 0:\\r\\n        raise ValueError(\\\"No reporting weather stations overlap the crops\\\")\\r\\n    tmax, tmin, prcp = mats[\\\"tmax\\\"], mats[\\\"tmin\\\"], mats[\\\"prcp\\\"]\\r\\n    fields = np.stack([tmax, tmin, prcp, _rolling(prcp, 3, \\\"sum\\\"),\\r\\n                       tmax - _rolling(tmax, 7, \\\"mean\\\"), tmax - tmin])\\r\\n    def xyz(coords):\\r\\n        lat, lon = np.deg2rad(coords).T\\r\\n        return np.c_[np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)]\\r\\n    tree = cKDTree(xyz(ll))\\r\\n    dist, nb = tree.query(xyz(np.c_[meta.lat, meta.lon]), k=min(32, len(ll)))\\r\\n    # query collapses to 1-D when there is a single station\\r\\n    dist, nb = dist.reshape(len(meta), -1), nb.reshape(len(meta), -1)\\r\\n    day = (pd.to_datetime(meta.date.astype(str)) - days[0]).dt.days.to_numpy()\\r\\n    out = np.full((len(meta), len(COLS)), np.nan, np.float32)\\r\\n    for i in range(len(meta)):\\r\\n        t = day[i]\\r\\n        if not (0 <= t < len(days)):\\r\\n            continue\\r\\n        v = fields[:, nb[i], t]                    # (features, k)\\r\\n        d = 2 * 6371.0 * np.arcsin(np.clip(dist[i] / 2, 0, 1))\\r\\n        available = np.isfinite(v) & (d <= MAX_KM)\\r\\n        available &= np.cumsum(available, axis=1) <= K\\r\\n        w = np.where(available, 1.0 / np.maximum(d, 0.5), 0.0)\\r\\n        s = w.sum(1)\\r\\n        good = s > 0\\r\\n        out[i, :6][good] = (np.nan_to_num(v) * w).sum(1)[good] / s[good]\\r\\n        near = np.isfinite(v).any(0)\\r\\n        out[i, 6] = d[near].min() if near.any() else MAX_KM\\r\\n    out[:, 7] = ~np.isfinite(out[:, :6]).all(1)\\r\\n    out[:, 6] = np.minimum(out[:, 6], MAX_KM) / 100.0\\r\\n    return out\\r\\n\\r\\n\\r\\ndef scale_split(raw: dict[str, np.ndarray], out: Path) -> None:\\r\\n    train = np.where(np.isfinite(raw[\\\"train\\\"]), raw[\\\"train\\\"], np.nan)\\r\\n    count = np.isfinite(train).sum(0)\\r\\n    mean = np.nansum(train, axis=0) / np.maximum(count, 1)\\r\\n    var = np.nansum((train - mean) ** 2, axis=0) / np.maximum(count, 1)\\r\\n    std = np.maximum(np.sqrt(var), 1e-6)\\r\\n    for split, a in raw.items():\\r\\n        z = np.nan_to_num((a - mean) / std, nan=0.0)\\r\\n        d = out / split\\r\\n        d.mkdir(parents=True, exist_ok=True)\\r\\n        np.save(d / \\\"ext.npy\\\", z.astype(np.float32))\\r\\n        print(f\\\"{split}: ext {z.shape} missing={float(np.mean(a[:, 7] > 0)):.1%}\\\", flush=True)\\r\\n    (out / \\\"ext_norm.json\\\").write_text(json.dumps({\\\"cols\\\": COLS, \\\"mean\\\": mean.tolist(),\\r\\n                                                   \\\"std\\\": std.tolist()}, indent=1))\\r\\n    audit = {f\\\"{s}_complete_fraction\\\": float(np.mean(a[:, 7] == 0)) for s, a in raw.items()}\\r\\n    audit['feature_observed_fraction'] = {s: np.isfinite(a[:, :6]).mean(0).tolist() for s, a in raw.items()}\\r\\n    (out / 'weather_audit.json').write_text(json.dumps(audit, indent=2))\\r\\n\\r\\n\\r\\ndef main() -> None:\\r\\n    ap = argparse.ArgumentParser()\\r\\n    ap.add_argument(\\\"--ghcn\\\", default=\\\"x_ghcn\\\")\\r\\n    ap.add_argument(\\\"--stations\\\", default=\\\"ghcnd-stations.txt\\\")\\r\\n    ap.add_argument(\\\"--out\\\", default=\\\"data/tsf\\\")\\r\\n    ap.add_argument(\\\"--year0\\\", type=int, default=2017)\\r\\n    ap.add_argument(\\\"--year1\\\", type=int, default=2021)\\r\\n    ap.add_argument(\\\"--splits\\\", default=\\\"train,eval,test\\\")\\r\\n    args = ap.parse_args()\\r\\n\\r\\n    out, ghcn = Path(args.out), Path(args.ghcn)\\r\\n    splits = {s: pd.read_csv(out / s / \\\"meta.csv\\\") for s in args.splits.split(',')}\\r\\n    meta = pd.concat(splits.values(), ignore_index=True)\\r\\n    ids_all, ll_all = _station_coords(Path(args.stations))\\r\\n    # only stations near our crops and present in the core network\\r\\n    near = ((ll_all[:, 0] > meta.lat.min() - 2) & (ll_all[:, 0] < meta.lat.max() + 2)\\r\\n            & (ll_all[:, 1] > meta.lon.min() - 2) & (ll_all[:, 1] < meta.lon.max() + 2))\\r\\n    days = pd.date_range(f\\\"{args.year0}-01-01\\\", f\\\"{args.year1}-12-31\\\", freq=\\\"D\\\")\\r\\n    ids = [i for i, k in zip(ids_all, near) if k]\\r\\n    ll = ll_all[near]\\r\\n    if not any(ghcn.glob(\\\"ghcnd_wide_*.parquet\\\")):\\r\\n        raise FileNotFoundError(f\\\"No ghcnd_wide_*.parquet under {ghcn}\\\")\\r\\n    mats = _grid(ghcn, range(args.year0, args.year1 + 1), ids,\\r\\n                 {s: i for i, s in enumerate(ids)}, days)\\r\\n    keep = np.isfinite(mats[\\\"tmax\\\"]).any(1) | np.isfinite(mats[\\\"prcp\\\"]).any(1)\\r\\n    print(f\\\"{len(ids)} stations in the crop bbox, {int(keep.sum())} with data\\\", flush=True)\\r\\n    mats = {k: v[keep] for k, v in mats.items()}\\r\\n    ll = ll[keep]\\r\\n\\r\\n    raw = {s: build(d, mats, ll, days) for s, d in splits.items()}\\r\\n    scale_split(raw, out)\\r\\n\\r\\n\\r\\nif __name__ == \\\"__main__\\\":\\r\\n    main()\\r\\n\",\"wf/candidates.py\":\"\\\"\\\"\\\"Candidate space for the architecture search.\\n\\nMeasured throughput (L4, bf16, channels_last, bs 256) decides what is worth\\nscreening: SMP encoders run at 2k-6k samples/s while full-resolution dilated\\ntrunks at 250-300, so the search is centred on encoder-decoder models.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport copy\\n\\nPROTOCOLS = {\\n    1: {\\\"n_train\\\": 50000, \\\"epochs\\\": 30, \\\"lr\\\": 3e-3, \\\"batch_size\\\": 384, \\\"warmup_epochs\\\": 2,\\n        \\\"max_seconds\\\": 1500, \\\"patience\\\": 99, \\\"eval_n\\\": 5000},\\n    2: {\\\"n_train\\\": 100000, \\\"epochs\\\": 30, \\\"lr\\\": 2.5e-3, \\\"batch_size\\\": 384, \\\"warmup_epochs\\\": 2,\\n        \\\"patience\\\": 10},\\n    3: {\\\"n_train\\\": None, \\\"epochs\\\": 36, \\\"lr\\\": 2.2e-3, \\\"batch_size\\\": 384, \\\"warmup_epochs\\\": 2, \\\"patience\\\": 8},\\n}\\n\\nBASE = {\\n    \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2},\\n    \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"dice_weight\\\": 1.0},\\n    \\\"train\\\": {\\\"weight_decay\\\": 1e-4, \\\"amp\\\": \\\"bf16\\\", \\\"grad_clip\\\": 1.0, \\\"workers\\\": 10, \\\"eval_workers\\\": 4,\\n              \\\"ema_decay\\\": 0.999},\\n}\\n\\n\\ndef smp(arch: str, encoder: str, **kw) -> dict:\\n    return {\\\"model\\\": {\\\"smp\\\": {\\\"arch\\\": arch, \\\"encoder\\\": encoder, **kw}}}\\n\\n\\ndef smp_up(arch: str, encoder: str, up: int, **kw) -> dict:\\n    return {\\\"model\\\": {**smp(arch, encoder, **kw)[\\\"model\\\"], \\\"up\\\": up}}\\n\\n\\ndef pv(prep: str, drop: str = \\\"\\\") -> dict:\\n    \\\"\\\"\\\"Same trunk as the winner, only the input preprocessing changes.\\\"\\\"\\\"\\n    return {**smp(\\\"UnetPlusPlus\\\", \\\"resnet18\\\"),\\n            \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"prep\\\": prep, \\\"drop\\\": drop}}\\n\\n\\ndef td(**kw) -> dict:\\n    \\\"\\\"\\\"Orthogonal-transform trunk (TD-FusionUNet line).\\\"\\\"\\\"\\n    return {\\\"model\\\": {\\\"trunk\\\": \\\"td\\\", **kw}}\\n\\n\\nCANDIDATES: dict[str, dict] = {\\n    # --- encoder-decoder with pretrained backbones (the fast lane) ---\\n    \\\"smp_unet_r18\\\": smp(\\\"Unet\\\", \\\"resnet18\\\"),\\n    \\\"smp_unet_r34\\\": smp(\\\"Unet\\\", \\\"resnet34\\\"),\\n    \\\"smp_unet_r50\\\": smp(\\\"Unet\\\", \\\"resnet50\\\"),\\n    \\\"smp_unetpp_r18\\\": smp(\\\"UnetPlusPlus\\\", \\\"resnet18\\\"),\\n    \\\"smp_unetpp_r34\\\": smp(\\\"UnetPlusPlus\\\", \\\"resnet34\\\"),\\n    \\\"smp_deeplab_r18\\\": smp(\\\"DeepLabV3Plus\\\", \\\"resnet18\\\"),\\n    \\\"smp_deeplab_r34\\\": smp(\\\"DeepLabV3Plus\\\", \\\"resnet34\\\"),\\n    \\\"smp_fpn_r18\\\": smp(\\\"FPN\\\", \\\"resnet18\\\"),\\n    \\\"smp_fpn_r50\\\": smp(\\\"FPN\\\", \\\"resnet50\\\"),\\n    \\\"smp_psp_r18\\\": smp(\\\"PSPNet\\\", \\\"resnet18\\\"),\\n    \\\"smp_linknet_r34\\\": smp(\\\"LinkNet\\\", \\\"resnet34\\\"),\\n    \\\"smp_manet_b0\\\": smp(\\\"MAnet\\\", \\\"efficientnet-b0\\\"),\\n    \\\"smp_unet_effb1\\\": smp(\\\"Unet\\\", \\\"efficientnet-b1\\\"),\\n    \\\"smp_unet_mitb0\\\": smp(\\\"Unet\\\", \\\"mit_b0\\\"),\\n    \\\"smp_unet_mitb1\\\": smp(\\\"Unet\\\", \\\"mit_b1\\\"),\\n    \\\"smp_unet_convnext_t\\\": smp(\\\"Unet\\\", \\\"tu-convnext_tiny\\\"),\\n    \\\"smp_unet_r18_nopret\\\": smp(\\\"Unet\\\", \\\"resnet18\\\", weights=None),\\n\\n    # --- same encoders on a 2x grid (tiny targets, see notes in SMPNet) ---\\n    \\\"smp_unet_r18_up2\\\": smp_up(\\\"Unet\\\", \\\"resnet18\\\", 2),\\n    \\\"smp_unet_r34_up2\\\": smp_up(\\\"Unet\\\", \\\"resnet34\\\", 2),\\n\\n    # --- custom trunks (heavier per sample, but no ImageNet prior) ---\\n    \\\"unet32_4l\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"unet\\\", \\\"widths\\\": [32, 64, 128, 256], \\\"depths\\\": [2, 2, 2, 2]}},\\n    \\\"unet48\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"unet\\\", \\\"widths\\\": [48, 96, 192], \\\"depths\\\": [2, 2, 2]}},\\n    \\\"unet48_lk\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"unet\\\", \\\"widths\\\": [48, 96, 192], \\\"depths\\\": [2, 2, 2], \\\"block\\\": \\\"lk\\\"}},\\n    \\\"unet64\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"unet\\\", \\\"widths\\\": [64, 128, 256], \\\"depths\\\": [2, 2, 2]}},\\n    \\\"hrnet32\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"hrnet\\\", \\\"width\\\": 32, \\\"stages\\\": 3}},\\n    \\\"hrnet48\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"hrnet\\\", \\\"width\\\": 48, \\\"stages\\\": 3}},\\n    \\\"dil64\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"fullres\\\", \\\"width\\\": 64,\\n                        \\\"blocks\\\": [[1, 2], [2, 2], [4, 2], [8, 2], [16, 2], [8, 2], [4, 2]]}},\\n\\n    # --- dual-branch fuel/weather encoder with a CAFIM gate per scale ---\\n    \\\"dual32\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"dual\\\", \\\"widths\\\": [32, 64, 128], \\\"depths\\\": [2, 2, 2]}},\\n    \\\"dual48\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"dual\\\", \\\"widths\\\": [48, 96, 192], \\\"depths\\\": [2, 2, 2]}},\\n    \\\"dual32_deep\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"dual\\\", \\\"widths\\\": [32, 64, 128], \\\"depths\\\": [3, 3, 3]}},\\n    \\\"dual48_bg\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"dual\\\", \\\"widths\\\": [48, 96, 192], \\\"depths\\\": [2, 2, 2],\\n                            \\\"heads\\\": [\\\"main\\\", \\\"boundary\\\", \\\"growth\\\"]},\\n                  \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"boundary_weight\\\": 0.3, \\\"growth_weight\\\": 0.3}},\\n\\n    # --- multi-head (main + boundary / growth) ---\\n    \\\"unet48_bg\\\": {\\\"model\\\": {\\\"trunk\\\": \\\"unet\\\", \\\"widths\\\": [48, 96, 192], \\\"depths\\\": [2, 2, 2],\\n                            \\\"heads\\\": [\\\"main\\\", \\\"boundary\\\", \\\"growth\\\"]},\\n                  \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"boundary_weight\\\": 0.3, \\\"growth_weight\\\": 0.3}},\\n    \\\"smp_unetpp_r18_bg\\\": {\\\"model\\\": {\\\"smp\\\": {\\\"arch\\\": \\\"UnetPlusPlus\\\", \\\"encoder\\\": \\\"resnet18\\\"},\\n                                    \\\"heads\\\": [\\\"main\\\", \\\"boundary\\\"]},\\n                          \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"boundary_weight\\\": 0.3}},\\n\\n    # --- loss / augmentation ablations on the base encoder ---\\n    \\\"smp_unet_r18_tversky\\\": {**smp(\\\"Unet\\\", \\\"resnet18\\\"), \\\"loss\\\": {\\\"name\\\": \\\"tversky\\\", \\\"alpha\\\": 0.3, \\\"beta\\\": 0.7}},\\n    \\\"smp_unet_r18_ftl\\\": {**smp(\\\"Unet\\\", \\\"resnet18\\\"),\\n                         \\\"loss\\\": {\\\"name\\\": \\\"focal_tversky\\\", \\\"alpha\\\": 0.4, \\\"beta\\\": 0.6, \\\"gamma\\\": 1.33}},\\n    \\\"smp_unet_r18_lovasz\\\": {**smp(\\\"Unet\\\", \\\"resnet18\\\"), \\\"loss\\\": {\\\"name\\\": \\\"lovasz\\\"}},\\n    \\\"smp_unet_r18_drop4\\\": {**smp(\\\"Unet\\\", \\\"resnet18\\\"), \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.4}},\\n    \\\"smp_unet_r18_drop0\\\": {**smp(\\\"Unet\\\", \\\"resnet18\\\"), \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.0}},\\n\\n    # --- preprocessing sweep: identical trunk, different input channels ---\\n    \\\"pp_lean\\\": pv(\\\"base\\\", \\\"lean\\\"),\\n    \\\"pp_ctx\\\": pv(\\\"ctx\\\"),\\n    \\\"pp_scar\\\": pv(\\\"scar\\\"),\\n    \\\"pp_wind\\\": pv(\\\"wind\\\"),\\n    \\\"pp_scarwind\\\": pv(\\\"scarwind\\\"),\\n    \\\"pp_all\\\": pv(\\\"all\\\"),\\n    \\\"pp_all_lean\\\": pv(\\\"all\\\", \\\"lean\\\"),\\n    \\\"pp_dist\\\": pv(\\\"dist\\\", \\\"nodist\\\"),\\n\\n    # --- orthogonal-transform trunks (TD-FusionUNet line) ---\\n    \\\"td_sum\\\": td(branches=[\\\"ht\\\", \\\"dct\\\"], fusion=\\\"sum\\\", width=16),\\n    \\\"td_gate\\\": td(branches=[\\\"ht\\\", \\\"dct\\\"], fusion=\\\"gate\\\", width=16),\\n    \\\"td_gate3\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n    \\\"td_gate_deep\\\": td(branches=[\\\"ht\\\", \\\"dct\\\"], fusion=\\\"gate\\\", width=16, depth=2),\\n    \\\"td_gate_wide\\\": td(branches=[\\\"ht\\\", \\\"dct\\\"], fusion=\\\"gate\\\", width=24),\\n    \\\"td_gate_mh\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\"], fusion=\\\"gate\\\", width=16,\\n                        heads=[\\\"main\\\", \\\"boundary\\\", \\\"growth\\\"]),\\n                   \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"boundary_weight\\\": 0.3, \\\"growth_weight\\\": 0.3}},\\n    \\\"td_gate_pf\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\"], fusion=\\\"gate\\\", width=16),\\n                   \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"pf_aug\\\": \\\"both\\\"}},\\n\\n    # --- round 2: soft labels / scar-channel ablations / physics channels ---\\n    \\\"td_gate3_soft8\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                       \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"soft\\\": 0.8}},\\n    \\\"td_gate3_soft15\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                        \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"soft\\\": 1.5}},\\n    \\\"td_gate3_noscar\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                        \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"drop\\\": \\\"noscar\\\"}},\\n    \\\"td_gate3_scarwind\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                          \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"prep\\\": \\\"scarwind\\\"}},\\n    \\\"td_gate3_ftl\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                     \\\"loss\\\": {\\\"name\\\": \\\"focal_tversky\\\", \\\"alpha\\\": 0.4, \\\"beta\\\": 0.6, \\\"gamma\\\": 1.33}},\\n    \\\"td_gate3_pf\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                    \\\"data\\\": {\\\"root\\\": \\\"data/tsf\\\", \\\"prev_dropout\\\": 0.2, \\\"pf_aug\\\": \\\"both\\\"}},\\n\\n    # --- round 3: precision-weighted objective (FP costs more than FN) ---\\n    \\\"td_ptv\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n               \\\"loss\\\": {\\\"name\\\": \\\"focal_tversky\\\", \\\"alpha\\\": 0.7, \\\"beta\\\": 0.3, \\\"gamma\\\": 1.33}},\\n\\n    # --- round 3: moment-matching term (where the mass sits / how big it is) ---\\n    \\\"td_mom05\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                 \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"moment_weight\\\": 0.5}},\\n    \\\"td_mom10\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                 \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"moment_weight\\\": 1.0}},\\n\\n    # --- round 4: architecture brute force around the same trunk ---\\n    \\\"td_attn\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, attn=True),\\n    \\\"td_dec2\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, dec=2),\\n    \\\"td_4s\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, sizes=(64, 32, 16, 8)),\\n    \\\"td_4s_dec2\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16,\\n                     sizes=(64, 32, 16, 8), dec=2),\\n    \\\"td_w8\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=8),\\n    \\\"td_d3\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, depth=3),\\n\\n    # --- round 8 (TS-SatFire): the 4-scale trunk wins, combine it ---\\n    \\\"td_4s_attn\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16,\\n                     sizes=(64, 32, 16, 8), attn=True),\\n    # late fusion: GHCN station weather (wf.ext, 8 columns) shifts the final logits\\n    \\\"td_4s_late\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16,\\n                     sizes=(64, 32, 16, 8), late=8),\\n\\n    # --- round 5: explicit area head + top-k decode (the measured bottleneck) ---\\n    \\\"td_area\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, area=32),\\n                \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"area_weight\\\": 1.0}},\\n    \\\"td_area_w2\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, area=64),\\n                   \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"area_weight\\\": 2.0}},\\n\\n    # --- round 6: optimise the per-image metric directly ---\\n    \\\"td_lovasz\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16),\\n                  \\\"loss\\\": {\\\"name\\\": \\\"lovasz\\\"}},\\n\\n    # --- round 7: deep supervision and the high-resolution sub-pixel branch ---\\n    \\\"td_ds\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, ds=True),\\n              \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"ds_weight\\\": 0.5}},\\n    \\\"td_hr\\\": td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, hr=8),\\n    \\\"td_hr_ds\\\": {**td(branches=[\\\"ht\\\", \\\"dct\\\", \\\"conv\\\"], fusion=\\\"gate\\\", width=16, hr=8, ds=True),\\n                 \\\"loss\\\": {\\\"name\\\": \\\"bce_dice\\\", \\\"pos_weight\\\": 5.0, \\\"ds_weight\\\": 0.5}},\\n}\\n\\n# Cheapest-first-ish, likely winners on top: the deadline may cut the tail.\\nORDER = [\\n    \\\"td_sum\\\", \\\"td_gate\\\", \\\"td_gate3\\\", \\\"td_gate_deep\\\", \\\"td_gate_wide\\\", \\\"td_gate_mh\\\", \\\"td_gate_pf\\\",\\n    \\\"smp_unet_r18\\\", \\\"smp_unet_r34\\\", \\\"smp_deeplab_r18\\\", \\\"smp_unetpp_r18\\\", \\\"smp_unet_mitb0\\\",\\n    \\\"dual32\\\", \\\"dual48\\\", \\\"smp_fpn_r18\\\", \\\"smp_manet_b0\\\", \\\"unet32_4l\\\", \\\"smp_unet_r50\\\",\\n    \\\"smp_unet_convnext_t\\\", \\\"smp_unetpp_r34\\\", \\\"smp_deeplab_r34\\\", \\\"dual32_deep\\\", \\\"unet48\\\",\\n    \\\"hrnet32\\\", \\\"dual48_bg\\\", \\\"smp_psp_r18\\\", \\\"smp_linknet_r34\\\", \\\"smp_unet_effb1\\\",\\n    \\\"smp_unet_r18_drop4\\\", \\\"smp_unet_r18_tversky\\\", \\\"smp_unet_r18_lovasz\\\", \\\"smp_unet_r18_ftl\\\",\\n    \\\"smp_unetpp_r18_bg\\\", \\\"smp_unet_r18_nopret\\\", \\\"unet48_bg\\\", \\\"unet48_lk\\\", \\\"unet64\\\",\\n    \\\"hrnet48\\\", \\\"dil64\\\", \\\"smp_unet_mitb1\\\", \\\"smp_fpn_r50\\\",\\n]\\n\\n\\ndef make_cfg(name: str, stage: int = 1) -> dict:\\n    cfg = copy.deepcopy(BASE)\\n    cand = copy.deepcopy(CANDIDATES[name])\\n    cfg[\\\"name\\\"] = name\\n    for k, v in cand.items():\\n        cfg.setdefault(k, {}).update(v)\\n    cfg[\\\"train\\\"].update(PROTOCOLS[stage])\\n    return cfg\\n\",\"wf/latefusion.py\":\"\\\"\\\"\\\"Matched baseline / external-weather late-fusion experiment (eval only).\\\"\\\"\\\"\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport numpy as np\\n\\nfrom .data import load_ext\\nfrom .train import run\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser(description=__doc__)\\n    ap.add_argument('--root', default='data/tsf')\\n    ap.add_argument('--out', default='results/latefusion')\\n    ap.add_argument('--epochs', type=int, default=12)\\n    ap.add_argument('--batch-size', type=int, default=32)\\n    ap.add_argument('--n-train', type=int, default=4096)\\n    ap.add_argument('--seed', type=int, default=0)\\n    args = ap.parse_args()\\n    for split in ('train', 'eval'):\\n        ext = load_ext(args.root, split)\\n        if ext is None or ext.shape[1] != 8 or not np.isfinite(ext).all():\\n            raise ValueError(f'{split}: build valid eight-column ext.npy with wf.ext first')\\n    results = {}\\n    for name, late in [('baseline', 0), ('weather_late', 8)]:\\n        cfg = {\\n            'seed': args.seed,\\n            'data': {'root': args.root, 'prev_dropout': 0.2, 'in_memory': False},\\n            'model': {'trunk': 'td', 'branches': ['ht', 'dct', 'conv'],\\n                      'width': 16, 'sizes': [64, 32, 16, 8], 'late': late},\\n            'loss': {'name': 'bce_dice', 'pos_weight': 5.0},\\n            'train': {'epochs': args.epochs, 'batch_size': args.batch_size,\\n                      'eval_batch_size': 64, 'n_train': args.n_train or None,\\n                      'workers': 2, 'eval_workers': 2, 'amp': 'fp16',\\n                      'lr': 0.002, 'patience': args.epochs},\\n        }\\n        results[name] = run(cfg, f'{name}_seed{args.seed}', args.out)\\n    report = {\\n        'seed': args.seed, 'selection_split': 'eval', 'test_evaluated': False,\\n        'eval': {k: v['val_at_best_thr'] for k, v in results.items()},\\n        'thresholds': {k: v['best_thr'] for k, v in results.items()},\\n    }\\n    report['delta_macro_dice'] = (report['eval']['weather_late']['macro_dice'] -\\n                                  report['eval']['baseline']['macro_dice'])\\n    Path(args.out, 'comparison.json').write_text(json.dumps(report, indent=2))\\n    print(json.dumps(report, indent=2))\\n\\n\\nif __name__ == '__main__':\\n    main()\\n\",\"wf/probe.py\":\"\\\"\\\"\\\"Where the Dice is lost: rank vs location vs area vs diffuseness.\\n\\nLoads a finished run, scores its eval probabilities once, and compares the\\nmodel mask against geometric proxies built from the same probabilities:\\n\\n  base       p > thr (the number the search reports)\\n  topk       the |p| highest pixels per patch (same mass, sharpened)\\n  topk_gt    the GT-area highest pixels (perfect area, model ranking)\\n  disk_pre   GT-area disk at the predicted centroid (model location)\\n  disk_both  predicted-area disk at the predicted centroid (no shape at all)\\n  lcc/open   p > thr with specks removed / morphologically opened\\n\\n    python -m wf.probe --run td_gate3_s1\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom pathlib import Path\\n\\nimport numpy as np\\nimport torch\\nfrom scipy import ndimage as ndi\\n\\nfrom .data import FireDataset\\nfrom .final import _targets, predict_tta\\nfrom .metrics import dice_per_sample, sweep\\nfrom .train import build\\n\\nYY, XX = np.mgrid[0:64, 0:64]\\n\\n\\ndef _disk_at(cy: float, cx: float, k: int) -> np.ndarray:\\n    d = (YY - cy) ** 2 + (XX - cx) ** 2\\n    cut = np.sort(d.ravel())[min(max(k, 1), 4096) - 1]\\n    return d <= cut\\n\\n\\ndef _topk(flat: np.ndarray, k: int) -> np.ndarray:\\n    out = np.zeros_like(flat, dtype=bool)\\n    out[np.argsort(-flat)[: max(k, 0)]] = True\\n    return out\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--run\\\", required=True)\\n    ap.add_argument(\\\"--runs\\\", default=\\\"results/runs\\\")\\n    ap.add_argument(\\\"--root\\\", default=\\\"data/processed\\\")\\n    ap.add_argument(\\\"--tta\\\", type=int, default=1)\\n    args = ap.parse_args()\\n\\n    out = Path(args.runs) / args.run\\n    ck = torch.load(out / \\\"best.pt\\\", map_location=\\\"cpu\\\")\\n    cfg = ck[\\\"cfg\\\"]\\n    dev = torch.device(\\\"cuda\\\" if torch.cuda.is_available() else \\\"cpu\\\")\\n    model, prep, drop = build(cfg, args.root)\\n    model = model.to(dev).to(memory_format=torch.channels_last)\\n    model.load_state_dict(ck[\\\"model\\\"], strict=False)\\n\\n    ds = FireDataset(args.root, \\\"eval\\\", None, aux=False, prep=prep, drop=drop)\\n    tgt, valid = _targets(ds)\\n    p = predict_tta(model, torch.utils.data.DataLoader(ds, batch_size=512, num_workers=4),\\n                    dev, torch.bfloat16, args.tta)[0]\\n    np.save(out / \\\"eval_prob.npy\\\", p)\\n    thr, rows = sweep(p, tgt, valid)\\n    g = tgt & valid\\n    area = g.reshape(len(g), -1).sum(1)\\n    mass = p.reshape(len(p), -1).sum(1)\\n\\n    def score(masks: np.ndarray) -> float:\\n        return float(dice_per_sample(masks & valid, g).mean())\\n\\n    res = {\\\"thr\\\": round(float(thr), 3), \\\"base\\\": round(score(p > thr), 4)}\\n    base_masks = p > thr\\n    tk = np.stack([_topk(p[i].ravel(), int(round(mass[i]))) for i in range(len(p))]).reshape(p.shape)\\n    tk_gt = np.stack([_topk(p[i].ravel(), int(area[i])) for i in range(len(p))]).reshape(p.shape)\\n    res[\\\"topk\\\"] = round(score(tk), 4)\\n    res[\\\"topk_gt\\\"] = round(score(tk_gt), 4)\\n\\n    disk_pre = np.zeros_like(base_masks)\\n    disk_both = np.zeros_like(base_masks)\\n    disk_peak = np.zeros_like(base_masks)\\n    cen_err = []\\n    for i in range(len(p)):\\n        m = base_masks[i]\\n        if not m.any():\\n            continue\\n        cy, cx = ndi.center_of_mass(m)\\n        disk_pre[i] = _disk_at(cy, cx, int(area[i]))\\n        disk_both[i] = _disk_at(cy, cx, int(round(m.sum())))\\n        py, px = np.unravel_index(int(np.argmax(ndi.gaussian_filter(p[i], 1.5))), p[i].shape)\\n        disk_peak[i] = _disk_at(py, px, int(round(m.sum())))\\n        gy, gx = ndi.center_of_mass(g[i])\\n        cen_err.append(float(np.hypot(cy - gy, cx - gx)))\\n    res[\\\"disk_pre\\\"] = round(score(disk_pre), 4)\\n    res[\\\"disk_both\\\"] = round(score(disk_both), 4)\\n    res[\\\"disk_peak\\\"] = round(score(disk_peak), 4)\\n    res[\\\"miss\\\"] = round(float(np.mean([not m.any() for m in base_masks])), 4)\\n    res[\\\"cen_err_p50\\\"] = round(float(np.median(cen_err)), 2)\\n    res[\\\"cen_err_p90\\\"] = round(float(np.percentile(cen_err, 90)), 2)\\n    res[\\\"area_ratio\\\"] = round(float(mass.sum() / area.sum()), 3)\\n\\n    lcc = np.zeros_like(base_masks)\\n    opened = np.zeros_like(base_masks)\\n    for i, m in enumerate(base_masks):\\n        lab, n = ndi.label(m)\\n        if n:\\n            sizes = ndi.sum(m, lab, range(1, n + 1))\\n            lcc[i] = lab == int(np.argmax(sizes)) + 1\\n        opened[i] = ndi.binary_opening(m, np.ones((3, 3)))\\n    res[\\\"lcc\\\"] = round(score(lcc), 4)\\n    res[\\\"open3\\\"] = round(score(opened), 4)\\n    res[\\\"best_thr\\\"] = round(float(max(r[1] for r in rows)), 4)\\n    print(args.run, res, flush=True)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"kaggle/run_exploration.py\":\"\\\"\\\"\\\"Run from the repository directory INSIDE Kaggle. No local training.\\\"\\\"\\\"\\nfrom pathlib import Path\\nimport json\\nimport os\\nimport subprocess\\nimport sys\\nimport urllib.request\\n\\nimport numpy as np\\n\\n\\ndef run(module, *args):\\n    subprocess.run([sys.executable, '-u', '-m', module, *map(str, args)], check=True)\\n\\n\\ndef main():\\n    if not Path('/kaggle/working').is_dir():\\n        raise RuntimeError('This runner is for Kaggle only')\\n    os.environ['OMP_NUM_THREADS'] = '2'\\n    run('pip', 'install', '-q', 'segmentation-models-pytorch', 'timm', 'rasterio', 'pyarrow')\\n    import torch\\n    import rasterio\\n    assert torch.cuda.is_available(), 'Enable GPU T4'\\n    print('GPU:', torch.cuda.get_device_name(0), flush=True)\\n    sat = Path('/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire')\\n    if not sat.is_dir():\\n        matches = [p for p in Path('/kaggle/input').rglob('ts-satfire') if any(p.glob('*/VIIRS_Day'))]\\n        if len(matches) != 1:\\n            raise RuntimeError(f'Attach TS-SatFire before Save & Run All: {matches}')\\n        sat = matches[0]\\n    weather = list(Path('/kaggle/input').rglob('ghcnd_wide_2017.parquet'))\\n    if len(weather) != 1:\\n        raise RuntimeError('Attach the GHCN analysis layer before Save & Run All')\\n    weather_root = weather[0].parent\\n    root = Path('/kaggle/working/wildfire')\\n    root.mkdir(exist_ok=True)\\n    roi = root / 'roi'\\n    roi.mkdir(exist_ok=True)\\n    for name in ['fire_all_test.csv'] + [f'us_fire_{y}_out_new.csv' for y in range(2017, 2021)]:\\n        target = roi / name\\n        if not target.exists():\\n            urllib.request.urlretrieve('https://huggingface.co/datasets/SamuelWu318/ts-satfire/resolve/main/roi/' + name, target)\\n\\n    # A small raw-data audit runs on Kaggle, before allocating a large cache.\\n    audit = []\\n    for event in sorted(p for p in sat.iterdir() if p.is_dir())[:6]:\\n        for folder in ('VIIRS_Day', 'FirePred'):\\n            files = sorted((event / folder).glob('*.tif'))\\n            if not files:\\n                continue\\n            with rasterio.open(files[0]) as src:\\n                a = src.read().astype(np.float32)\\n                bands = []\\n                for band in a:\\n                    finite = band[np.isfinite(band)]\\n                    bands.append({'finite': float(np.isfinite(band).mean()),\\n                                  'quantiles': np.percentile(finite, [1, 50, 99]).tolist() if finite.size else []})\\n                audit.append({'path': str(files[0]), 'shape': list(a.shape), 'crs': str(src.crs),\\n                              'bounds': list(src.bounds), 'bands': bands})\\n    (root / 'raw_audit.json').write_text(json.dumps(audit, indent=2))\\n\\n    # Check new spatial fusion in the requested GPU environment, never locally.\\n    from wf.models import Net\\n    m = Net({'trunk': 'unet', 'widths': [8, 16], 'depths': [1, 1], 'cin': 33,\\n             'late': 8, 'late_mode': 'spatial'}, [0.]*33, [1.]*33).cuda()\\n    result = m(torch.randn(2, 33, 64, 64, device='cuda'), torch.randn(2, 8, device='cuda'))['main']\\n    assert result.shape == (2, 1, 64, 64)\\n    result.mean().backward()\\n    assert torch.isfinite(m.late[-1].weight.grad).all()\\n    del m, result\\n    torch.cuda.empty_cache()\\n\\n    cache = Path('/kaggle/working/tsf_full')\\n    if not (cache / 'norm.json').exists():\\n        run('wf.tsf', '--cache', sat.parent, '--roi-csv', roi, '--out', cache,\\n            '--keep', '--per-day', 16, '--workers', 2, '--splits', 'train,eval')\\n    try:\\n        station = weather_root / 'ghcnd_wide_stations.parquet'\\n        if not station.exists():\\n            station = root / 'ghcnd-stations.txt'\\n            urllib.request.urlretrieve('https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt', station)\\n        run('wf.ext', '--ghcn', weather_root, '--stations', station, '--out', cache,\\n            '--splits', 'train,eval')\\n    except Exception as exc:\\n        # Preserve usable vision experiments when an external schema/join fails.\\n        (cache / 'weather_audit.json').write_text(json.dumps({'train_complete_fraction': 0,\\n                                                            'error': str(exc)}, indent=2))\\n        print('WEATHER_DISABLED:', repr(exc), flush=True)\\n    run('wf.explore', '--source', cache, '--out', '/kaggle/working/exploration', '--hours', 8)\\n\\n\\nif __name__ == '__main__':\\n    main()\\n\",\"wf/athr.py\":\"\\\"\\\"\\\"Per-sample thresholds fitted on eval.\\n\\nA single global threshold is wrong for this data: a 3-px fire and a 300-px fire\\nwant very different cut-offs (the eval oracle for a per-patch threshold scores\\n0.54 vs 0.46 for the best global one). Predicted area at 0.5 is known without\\nany extra inference, so bin on it and pick the best cut-off per bin.\\n\\nBin edges are fixed rather than fitted so they carry over to `test` unchanged;\\nonly the five thresholds are estimated. Fitted on eval only, with a 2-fold check\\nso the reported gain is not the fit itself.\\n`python -m wf.athr --prob results/runs/<run>/eval_prob.npy`\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\n\\nimport numpy as np\\n\\nfrom .metrics import dice_per_sample\\n\\nGRID = np.arange(0.30, 0.86, 0.025)\\n# predicted-fire-size bins: 0-3, 4-9, 10-24, 25-59, 60+ pixels\\nEDGES = np.array([0.0, 4.0, 10.0, 25.0, 60.0, 1e9])\\nMIN_BIN = 100\\n\\n\\ndef _area(prob: np.ndarray) -> np.ndarray:\\n    return (prob > 0.5).sum(axis=(1, 2))\\n\\n\\ndef _bin(prob: np.ndarray) -> np.ndarray:\\n    return np.clip(np.searchsorted(EDGES, _area(prob), side=\\\"right\\\") - 1, 0, len(EDGES) - 2)\\n\\n\\ndef fit(prob: np.ndarray, tgt: np.ndarray, valid: np.ndarray) -> dict:\\n    d = np.stack([dice_per_sample((prob > g) & valid, tgt & valid) for g in GRID], 1)  # (n, |GRID|)\\n    b = _bin(prob)\\n    g = float(GRID[int(np.argmax(d.mean(0)))])\\n    thr = []\\n    for i in range(len(EDGES) - 1):\\n        m = b == i\\n        thr.append(float(GRID[int(np.argmax(d[m].mean(0)))]) if m.sum() >= MIN_BIN else g)\\n    return {\\\"edges\\\": EDGES.tolist(), \\\"thr\\\": thr, \\\"global\\\": g, \\\"bin_n\\\": np.bincount(b, minlength=5).tolist()}\\n\\n\\ndef apply(prob: np.ndarray, rule: dict) -> np.ndarray:\\n    return np.asarray(rule[\\\"thr\\\"], np.float32)[_bin(prob)]\\n\\n\\ndef score(prob, tgt, valid, rule) -> float:\\n    return float(dice_per_sample((prob > apply(prob, rule)[:, None, None]) & valid, tgt & valid).mean())\\n\\n\\ndef cv(prob, tgt, valid, folds: int = 2, seed: int = 0) -> dict:\\n    \\\"\\\"\\\"Fit on one part, score the other; the honest number for the gain.\\\"\\\"\\\"\\n    idx = np.random.default_rng(seed).permutation(len(prob))\\n    parts = np.array_split(idx, folds)\\n    scores = []\\n    for i in range(folds):\\n        held = parts[i]\\n        rest = np.concatenate([parts[j] for j in range(folds) if j != i])\\n        rule = fit(prob[rest], tgt[rest], valid[rest])\\n        thr = apply(prob[held], rule)\\n        scores.append(float(dice_per_sample((prob[held] > thr[:, None, None]) & valid[held],\\n                                            tgt[held] & valid[held]).mean()))\\n    return {\\\"cv\\\": float(np.mean(scores)), \\\"folds\\\": scores}\\n\\n\\ndef main() -> None:\\n    from .data import FireDataset\\n    from .metrics import evaluate\\n\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--prob\\\", required=True, help=\\\"eval_prob.npy written by wf.final\\\")\\n    ap.add_argument(\\\"--root\\\", default=\\\"data/processed\\\")\\n    ap.add_argument(\\\"--out\\\", default=None)\\n    args = ap.parse_args()\\n\\n    ds = FireDataset(args.root, \\\"eval\\\", None, aux=False)\\n    tgt = np.stack([np.asarray(ds.fire[i]) >= 7 for i in range(len(ds))])\\n    valid = np.stack([~np.isin(np.asarray(ds.fire[i]), (4, 6)) for i in range(len(ds))])\\n    prob = np.load(args.prob)\\n\\n    rule = fit(prob, tgt, valid)\\n    res = {\\\"global_thr\\\": rule[\\\"global\\\"],\\n           \\\"global_dice\\\": float(dice_per_sample((prob > rule[\\\"global\\\"]) & valid, tgt & valid).mean()),\\n           \\\"rule\\\": rule, \\\"fit_dice\\\": score(prob, tgt, valid, rule), **cv(prob, tgt, valid),\\n           \\\"full\\\": evaluate(prob, tgt, valid, apply(prob, rule))}\\n    print(json.dumps(res, indent=1), flush=True)\\n    if args.out:\\n        open(args.out, \\\"w\\\").write(json.dumps(res, indent=1))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/post.py\":\"\\\"\\\"\\\"Post-processing sweep: coherent blob growth vs threshold.\\n\\nThe model under-predicts area (recall well below precision), and Dice punishes a\\nmissing pixel exactly as hard as a false one only when the blob is big. Growing\\nthe predicted mask by a small disk grows the blob without the scattered false\\npixels that a lower threshold adds, so it can beat the threshold sweep at the\\nsame recall.\\n\\n`python -m wf.post --prob results/runs/<run>/eval_prob.npy` prints the grid.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\n\\nimport numpy as np\\nfrom scipy import ndimage as ndi\\n\\nfrom .metrics import dice_per_sample\\n\\n\\ndef _disk(r: int) -> np.ndarray:\\n    y, x = np.ogrid[-r:r + 1, -r:r + 1]\\n    return x * x + y * y <= r * r\\n\\n\\ndef grow(mask: np.ndarray, r: int) -> np.ndarray:\\n    if not r:\\n        return mask\\n    d = _disk(r)\\n    return ndi.binary_dilation(mask, structure=d.reshape((1,) * (mask.ndim - 2) + d.shape))\\n\\n\\ndef score(prob: np.ndarray, tgt: np.ndarray, valid: np.ndarray, thr: float, r: int = 0) -> float:\\n    pred = grow((prob > thr) & valid, r)\\n    return float(dice_per_sample(pred, tgt & valid).mean())\\n\\n\\ndef sweep(prob, tgt, valid, thrs=None, rs=(0, 1, 2)) -> list[list[float]]:\\n    thrs = np.arange(0.30, 0.86, 0.05) if thrs is None else thrs\\n    return [[score(prob, tgt, valid, float(t), r) for r in rs] for t in thrs]\\n\\n\\ndef main() -> None:\\n    from .data import FireDataset\\n    from .final import _targets\\n\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--prob\\\", required=True)\\n    ap.add_argument(\\\"--split\\\", default=\\\"eval\\\")\\n    ap.add_argument(\\\"--root\\\", default=\\\"data/processed\\\")\\n    args = ap.parse_args()\\n\\n    ds = FireDataset(args.root, args.split, None, aux=False)\\n    tgt, valid = _targets(ds)\\n    prob = np.load(args.prob)\\n    rs = (0, 1, 2)\\n    rows = sweep(prob, tgt, valid, rs=rs)\\n    best = max(((t, r, v) for t, row in zip(np.arange(0.30, 0.86, 0.05), rows) for r, v in zip(rs, row)),\\n               key=lambda z: z[2])\\n    print(\\\"thr  \\\" + \\\"  \\\".join(f\\\"r={r}\\\" for r in rs))\\n    for t, row in zip(np.arange(0.30, 0.86, 0.05), rows):\\n        print(f\\\"{t:.2f} \\\" + \\\"  \\\".join(f\\\"{v:.4f}\\\" for v in row))\\n    print(json.dumps({\\\"best_thr\\\": round(best[0], 3), \\\"best_r\\\": best[1], \\\"best\\\": round(best[2], 4)}))\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/losses.py\":\"\\\"\\\"\\\"Segmentation losses that play well with ~0.2% positives per patch.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport torch\\nimport torch.nn.functional as F\\n\\nEPS = 1e-6\\n\\n\\ndef _dice(p, t, dims=(1, 2, 3)) -> torch.Tensor:\\n    inter = (p * t).sum(dims)\\n    den = p.sum(dims) + t.sum(dims)\\n    return (2 * inter + EPS) / (den + EPS)\\n\\n\\ndef bce_dice(logits, y, valid, pos_weight=5.0, dice_w=1.0):\\n    bce = F.binary_cross_entropy_with_logits(\\n        logits, y, reduction=\\\"none\\\", pos_weight=torch.tensor(pos_weight, device=logits.device))\\n    bce = (bce * valid).sum() / valid.sum().clamp_min(1)\\n    d = _dice(torch.sigmoid(logits) * valid, y * valid)\\n    return bce + dice_w * (1 - d.mean())\\n\\n\\ndef tversky(logits, y, valid, alpha=0.3, beta=0.7, gamma=1.0):\\n    p = torch.sigmoid(logits) * valid\\n    t = y * valid\\n    dims = (1, 2, 3)\\n    tp = (p * t).sum(dims)\\n    fp = (p * (1 - t)).sum(dims)\\n    fn = ((1 - p) * t).sum(dims)\\n    ti = (tp + EPS) / (tp + alpha * fp + beta * fn + EPS)\\n    return (1 - ti).pow(gamma).mean()\\n\\n\\ndef _lovasz_grad(gt_sorted):\\n    p = len(gt_sorted)\\n    gts = gt_sorted.sum()\\n    inter = gts - gt_sorted.float().cumsum(0)\\n    union = gts + (1 - gt_sorted).float().cumsum(0)\\n    jac = 1.0 - inter / union\\n    if p > 1:\\n        jac[1:p] = jac[1:p] - jac[0:p - 1]\\n    return jac\\n\\n\\ndef lovasz_hinge(logits, y, valid):\\n    losses = []\\n    for lg, lb, vb in zip(logits, y, valid):\\n        m = vb.reshape(-1) > 0.5\\n        if m.sum() == 0:\\n            continue\\n        lg, lb = lg.reshape(-1)[m], lb.reshape(-1)[m]\\n        signs = 2.0 * lb - 1.0\\n        errors = 1.0 - lg * signs\\n        errors_sorted, perm = torch.sort(errors, descending=True)\\n        gt_sorted = lb[perm]\\n        losses.append(torch.dot(F.relu(errors_sorted), _lovasz_grad(gt_sorted)))\\n    return torch.stack(losses).mean() if losses else logits.sum() * 0\\n\\n\\ndef moment(logits, y, valid, scale: float = 64.0, floor: float = 0.05, cap: float = 4.0):\\n    \\\"\\\"\\\"Match the first two moments of the predicted fire mass to the target.\\n\\n    The metric is per-patch Dice on a ~6 px blob, so what decides it is where the\\n    mass sits and how far it spreads. Segmentation losses are blind to both: a\\n    diffuse-but-centred blob has the same Dice as a compact one of the same\\n    overlap, and nothing pulls the mass into the blob. This term does.\\n\\n    `floor` removes the background probability carpet (BCE with pos_weight on\\n    0.2% positives leaves p~0.01 everywhere, whose total mass rivals the fire);\\n    `cap` keeps the early \\\"all pixels at 0.5\\\" phase from dominating the loss.\\n    Units: pixels squared, divided by `scale` (8 px mean+spread error ~1.0).\\n    \\\"\\\"\\\"\\n    p = torch.relu(torch.sigmoid(logits) - floor) * valid\\n    t = y * valid\\n    h, w = logits.shape[-2:]\\n    ys = torch.arange(h, device=logits.device, dtype=torch.float32) - (h - 1) / 2\\n    xs = torch.arange(w, device=logits.device, dtype=torch.float32) - (w - 1) / 2\\n\\n    def stats(a):\\n        s = a.sum(dim=(1, 2, 3), keepdim=True).clamp_min(EPS)\\n        my = (a * ys.view(1, 1, h, 1)).sum(dim=(1, 2, 3), keepdim=True) / s\\n        mx = (a * xs.view(1, 1, 1, w)).sum(dim=(1, 2, 3), keepdim=True) / s\\n        vy = (a * (ys.view(1, 1, h, 1) - my) ** 2).sum(dim=(1, 2, 3), keepdim=True) / s\\n        vx = (a * (xs.view(1, 1, 1, w) - mx) ** 2).sum(dim=(1, 2, 3), keepdim=True) / s\\n        return my, mx, torch.sqrt(vy + 0.25), torch.sqrt(vx + 0.25)\\n\\n    d = sum(((a - b) ** 2).mean() for a, b in zip(stats(p), stats(t))) / scale\\n    return d.clamp(max=cap).mean()\\n\\n\\ndef make_loss(lcfg: dict):\\n    name = lcfg.get(\\\"name\\\", \\\"bce_dice\\\")\\n    if name == \\\"bce_dice\\\":\\n        pw, dw = lcfg.get(\\\"pos_weight\\\", 5.0), lcfg.get(\\\"dice_weight\\\", 1.0)\\n        return lambda lg, y, v: bce_dice(lg, y, v, pw, dw)\\n    if name == \\\"tversky\\\" or name == \\\"focal_tversky\\\":\\n        a, b, g = lcfg.get(\\\"alpha\\\", 0.3), lcfg.get(\\\"beta\\\", 0.7), lcfg.get(\\\"gamma\\\", 1.0)\\n        g = g if name == \\\"focal_tversky\\\" else 1.0\\n        return lambda lg, y, v: tversky(lg, y, v, a, b, g)\\n    if name == \\\"lovasz\\\":\\n        return lovasz_hinge\\n    if name == \\\"dice\\\":\\n        return lambda lg, y, v: 1 - _dice(torch.sigmoid(lg) * v, y * v).mean()\\n    raise ValueError(name)\\n\",\"tests/test_exploration.py\":\"\\\"\\\"\\\"Run these checks on Kaggle, together with the experiment preflight.\\\"\\\"\\\"\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\n\\nfrom wf.data import CHANNELS, FireDataset\\nfrom wf.ext import build, scale_split\\nfrom wf.models import Net\\nfrom wf.train import _loader\\n\\n\\ndef test_spatial_fusion_has_spatial_effect_and_gradients():\\n    torch.manual_seed(0)\\n    model = Net({'trunk': 'unet', 'widths': [8, 16], 'depths': [1, 1],\\n                 'cin': 33, 'late': 8, 'late_mode': 'spatial'}, [0.]*33, [1.]*33)\\n    x, ext = torch.randn(2, 33, 64, 64), torch.randn(2, 8)\\n    zero = model(x, ext)['main']\\n    zero.mean().backward()\\n    assert model.late[-1].weight.grad.abs().sum() > 0\\n    with torch.no_grad():\\n        model.late[-1].weight.fill_(0.1)\\n    a, b = model(x, ext)['main'], model(x, torch.zeros_like(ext))['main']\\n    assert a.shape == (2, 1, 64, 64)\\n    assert (a - b).flatten(1).std(1).max() > 0\\n\\n\\ndef test_missing_weather_does_not_erase_observed_column(tmp_path):\\n    raw = np.array([[1., 2., 3., 4., 5., 6., 0., 0.],\\n                    [3., 4., 5., 6., 7., 8., 0., 0.],\\n                    [np.nan]*7 + [1.]], np.float32)\\n    scale_split({'train': raw, 'eval': raw.copy()}, tmp_path)\\n    z = np.load(tmp_path / 'train/ext.npy')\\n    assert np.isfinite(z).all()\\n    assert z[0, 0] != z[1, 0]\\n    assert z[2, 0] == 0\\n\\n\\ndef test_spherical_weather_join_and_integer_date():\\n    days = pd.date_range('2020-01-01', periods=2)\\n    mats = {key: np.array([[10., 12.], [20., 22.]], np.float32)\\n            for key in ('tmax', 'tmin', 'prcp')}\\n    meta = pd.DataFrame({'date': [20200102], 'lat': [40.], 'lon': [-120.]})\\n    out = build(meta, mats, np.array([[40., -120.], [0., 0.]]), days)\\n    assert out[0, 0] == 12.\\n    assert out[0, 6] == 0\\n    assert out[0, 7] == 0\\n\\n\\ndef test_subset_and_loader_independent_of_model_rng(tmp_path):\\n    d = tmp_path / 'train'\\n    d.mkdir()\\n    np.zeros((4, len(CHANNELS), 64, 64), np.float16).tofile(d / 'features.f16')\\n    labels = np.full((4, 64, 64), 5, np.uint8)\\n    labels.tofile(d / 'fire_class.u8')\\n    labels.tofile(d / 'prev_class.u8')\\n    ext = np.arange(32, dtype=np.float32).reshape(4, 8)\\n    ds = FireDataset(tmp_path, indices=np.array([3, 1, 2]), in_memory=True, ext=ext)\\n    assert np.array_equal(ds[0][4].numpy(), ext[3])\\n    a = next(iter(_loader(ds, 2, True, 0, seed=42)))[4]\\n    torch.randn(300)\\n    b = next(iter(_loader(ds, 2, True, 0, seed=42)))[4]\\n    assert torch.equal(a, b)\\n\",\"wf/inspect_test.py\":\"\\\"\\\"\\\"Final ensemble vs ground truth on the test split: per-sample stats + visual grid.\\n\\nReads the same 5 checkpoints and settings as `ensemble.py`, so `metrics` here\\nshould reproduce the single reported test pass exactly.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nfrom pathlib import Path\\n\\nimport matplotlib\\n\\nmatplotlib.use(\\\"Agg\\\")\\nimport matplotlib.pyplot as plt\\nimport numpy as np\\nimport torch\\n\\nfrom .data import FireDataset\\nfrom .ensemble import _load_run\\nfrom .final import _targets, predict_tta\\nfrom .metrics import dice_per_sample, evaluate\\nfrom .train import _loader\\n\\nRUNS = \\\"smp_unetpp_r18_s3,td_gate3_s3\\\"\\nWEIGHTS = \\\"0.6,0.4\\\"\\n\\n\\ndef _rgb(x: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Per-patch stretched elevation+NDVI relief with prev_fire in blue.\\\"\\\"\\\"\\n\\n    def n(i):\\n        a = x[i].astype(np.float32)\\n        lo, hi = np.percentile(a, [2, 98])\\n        return np.clip((a - lo) / (hi - lo + 1e-6), 0, 1)\\n\\n    base = 0.6 * n(1) + 0.4 * n(0)\\n    img = np.repeat(base[..., None], 3, axis=2) * 0.85\\n    img[..., 2] = np.clip(img[..., 2] + 0.9 * n(18), 0, 1)\\n    return img\\n\\n\\ndef _overlay(img: np.ndarray, gt: np.ndarray, pr: np.ndarray) -> np.ndarray:\\n    out = img * 0.45\\n    out[gt] = [0.15, 0.85, 0.35]\\n    out[pr] = [0.95, 0.25, 0.20]\\n    out[gt & pr] = [1.0, 0.95, 0.20]\\n    return out\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--runs\\\", default=RUNS)\\n    ap.add_argument(\\\"--weights\\\", default=WEIGHTS)\\n    ap.add_argument(\\\"--thr\\\", type=float, default=0.60)\\n    ap.add_argument(\\\"--tta\\\", type=int, default=4)\\n    ap.add_argument(\\\"--split\\\", default=\\\"test\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"results/test_vs_gt.png\\\")\\n    args = ap.parse_args()\\n\\n    dev = torch.device(\\\"cuda\\\")\\n    dtype = torch.bfloat16\\n    root = \\\"data/processed\\\"\\n    names = args.runs.split(\\\",\\\")\\n    w = np.array([float(v) for v in args.weights.split(\\\",\\\")])\\n    assert len(w) == len(names)\\n\\n    head = torch.load(Path(\\\"results/runs\\\") / names[0] / \\\"best.pt\\\", map_location=\\\"cpu\\\")[\\\"cfg\\\"][\\\"data\\\"]\\n    prep, drop = head.get(\\\"prep\\\", \\\"base\\\"), head.get(\\\"drop\\\", \\\"\\\")\\n\\n    ds = FireDataset(root, args.split, None, aux=False, prep=prep, drop=drop)\\n    tg, va = _targets(ds)\\n    ld = _loader(ds, 512, False, 4)\\n\\n    cache = Path(f\\\"results/{args.split}_blend.npy\\\")\\n    if cache.exists():\\n        blend = np.load(cache)\\n        print(\\\"blend (cached)\\\", flush=True)\\n    else:\\n        probs = []\\n        for name in names:\\n            model, _, _ = _load_run(name, dev, root)\\n            probs.append(predict_tta(model, ld, dev, dtype, args.tta)[0])\\n            del model\\n            torch.cuda.empty_cache()\\n        blend = np.average(probs, axis=0, weights=w)\\n        del probs\\n        np.save(cache, blend.astype(np.float32))\\n\\n    metrics = evaluate(blend, tg, va, args.thr, full=True)\\n    print(json.dumps(metrics, indent=1), flush=True)\\n\\n    pred = (blend > args.thr) & va\\n    gt = tg & va\\n    d = dice_per_sample(pred, gt)\\n    fp = np.logical_and(pred, ~gt).sum((1, 2))\\n    tp = np.logical_and(pred, gt).sum((1, 2))\\n    fn = np.logical_and(~pred, gt).sum((1, 2))\\n\\n    order = np.argsort(d)\\n    picks, seen = [], set()\\n    for tag, i in [\\n        (\\\"worst\\\", order[0]),\\n        (\\\"2nd worst\\\", order[1]),\\n        (\\\"median\\\", order[len(d) // 2]),\\n        (\\\"best\\\", order[-1]),\\n        (\\\"largest GT\\\", int(np.argmax(gt.sum((1, 2))))),\\n        (\\\"most false fire\\\", int(np.argmax(fp - tp))),\\n    ]:\\n        if i not in seen:\\n            seen.add(int(i))\\n            picks.append((tag, int(i), float(d[i])))\\n\\n    stats = {\\n        \\\"split\\\": args.split, \\\"thr\\\": args.thr, \\\"tta\\\": args.tta, \\\"runs\\\": names, \\\"weights\\\": w.tolist(),\\n        \\\"metrics\\\": metrics,\\n        \\\"per_sample\\\": {\\n            \\\"mean\\\": float(d.mean()), \\\"median\\\": float(np.median(d)),\\n            \\\"p10\\\": float(np.percentile(d, 10)), \\\"p90\\\": float(np.percentile(d, 90)),\\n            \\\"share_ge_0.5\\\": float((d >= 0.5).mean()), \\\"share_zero\\\": float((d == 0).mean()),\\n            \\\"share_pred_empty_gt_not\\\": float(((pred.sum((1, 2)) == 0) & (gt.sum((1, 2)) > 0)).mean()),\\n        },\\n        \\\"pixels\\\": {\\\"tp\\\": int(tp.sum()), \\\"fp\\\": int(fp.sum()), \\\"fn\\\": int(fn.sum()),\\n                   \\\"recall\\\": float(tp.sum() / max(tp.sum() + fn.sum(), 1)),\\n                   \\\"precision\\\": float(tp.sum() / max(tp.sum() + fp.sum(), 1))},\\n        \\\"cases\\\": [{\\\"tag\\\": t, \\\"index\\\": i, \\\"dice\\\": dd,\\n                   \\\"gt_px\\\": int(gt[i].sum()), \\\"pred_px\\\": int(pred[i].sum()),\\n                   \\\"tp\\\": int(tp[i]), \\\"fp\\\": int(fp[i]), \\\"fn\\\": int(fn[i])} for t, i, dd in picks],\\n    }\\n    Path(\\\"results/test_vs_gt.json\\\").write_text(json.dumps(stats, indent=1))\\n\\n    fig, axes = plt.subplots(len(picks), 5, figsize=(11.5, 2.3 * len(picks)), facecolor=\\\"white\\\")\\n    for r, (tag, i, dd) in enumerate(picks):\\n        img = _rgb(np.asarray(ds.x[i]))\\n        g, p = gt[i], pred[i]\\n        grey = np.repeat(img.mean(2, keepdims=True), 3, 2) * 0.35\\n        panels = [\\n            (img, \\\"input (elevation · ndvi · prev_fire)\\\"),\\n            (blend[i], f\\\"P(fire) — max {blend[i].max():.2f}\\\"),\\n            (grey + g[..., None] * np.array([0.15, 0.85, 0.35]), \\\"ground truth\\\"),\\n            (grey + p[..., None] * np.array([0.95, 0.25, 0.20]), f\\\"prediction (thr {args.thr})\\\"),\\n            (_overlay(img, g, p), \\\"overlay: hit / miss / false\\\"),\\n        ]\\n        for c, (im, title) in enumerate(panels):\\n            ax = axes[r, c]\\n            if c == 1:\\n                ax.imshow(im, cmap=\\\"magma\\\", vmin=0, vmax=1, interpolation=\\\"nearest\\\")\\n            else:\\n                ax.imshow(np.clip(im, 0, 1), interpolation=\\\"nearest\\\")\\n            ax.set_title(title, fontsize=8)\\n            ax.set_xticks([]); ax.set_yticks([])\\n        axes[r, 0].set_ylabel(\\n            f\\\"{tag}\\\\ndice {dd:.3f}\\\\nGT {int(gt[i].sum())}px · pred {int(pred[i].sum())}px\\\", fontsize=8\\n        )\\n    fig.suptitle(\\n        f\\\"test split: {len(names)}-model blend, TTA-{args.tta}, thr {args.thr} — macro Dice {metrics['macro_dice']:.4f} \\\"\\n        f\\\"(median {np.median(d):.3f}, {100 * (d >= 0.5).mean():.0f}% of patches >= 0.50)\\\",\\n        fontsize=9,\\n    )\\n    fig.tight_layout(rect=(0, 0, 1, 0.97))\\n    fig.savefig(args.out, dpi=140)\\n    print(json.dumps(stats[\\\"per_sample\\\"], indent=1), flush=True)\\n    print(\\\"wrote\\\", args.out, flush=True)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/final.py\":\"\\\"\\\"\\\"Finalist: full-data training, 8-way dihedral TTA, threshold on eval, one test pass.\\\"\\\"\\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport argparse\\r\\nimport json\\r\\nfrom pathlib import Path\\r\\n\\r\\nimport numpy as np\\r\\nimport torch\\r\\n\\r\\nfrom .candidates import make_cfg\\r\\nfrom .data import load_ext, PAIRS, FireDataset, jac, split_indices\\r\\nfrom .metrics import evaluate, evaluate_area, sweep, sweep_area\\r\\nfrom .train import _loader, build, norm_stats, run\\r\\n\\r\\n\\r\\ndef _geo(x: torch.Tensor, k: int) -> torch.Tensor:\\r\\n    if k & 1:\\r\\n        x = torch.flip(x, [-1])\\r\\n    if k & 2:\\r\\n        x = torch.flip(x, [-2])\\r\\n    if k & 4:\\r\\n        x = torch.rot90(x, 1, dims=(-2, -1))\\r\\n    return x\\r\\n\\r\\n\\r\\ndef _vec(x: torch.Tensor, k: int) -> torch.Tensor:\\r\\n    J = torch.tensor(jac(k), device=x.device, dtype=x.dtype)\\r\\n    x = x.clone()\\r\\n    for u, v in PAIRS:\\r\\n        uu, vv = x[:, u].clone(), x[:, v].clone()\\r\\n        x[:, u] = J[0, 0] * uu + J[0, 1] * vv\\r\\n        x[:, v] = J[1, 0] * uu + J[1, 1] * vv\\r\\n    return x\\r\\n\\r\\n\\r\\ndef _inv(p: torch.Tensor, k: int) -> torch.Tensor:\\r\\n    if k & 4:\\r\\n        p = torch.rot90(p, -1, dims=(-2, -1))\\r\\n    if k & 2:\\r\\n        p = torch.flip(p, [-2])\\r\\n    if k & 1:\\r\\n        p = torch.flip(p, [-1])\\r\\n    return p\\r\\n\\r\\n\\r\\n@torch.no_grad()\\r\\ndef predict_tta(model, loader, device, dtype, tta: int = 8):\\r\\n    \\\"\\\"\\\"Mean probability over the dihedral group, plus the area prediction if the\\r\\n    model has an area head (invariant under the group, so the plain mean).\\\"\\\"\\\"\\r\\n    model.eval()\\r\\n    ks = list(range(tta)) if tta > 1 else [0]\\r\\n    probs = [None] * len(ks)\\r\\n    areas = []\\r\\n    for i, k in enumerate(ks):\\r\\n        acc = []\\r\\n        for batch in loader:\\r\\n            x = batch[0]\\r\\n            ext = batch[4].to(device) if len(batch) > 4 else None\\r\\n            x = x.to(device, non_blocking=True).float().to(memory_format=torch.channels_last)\\r\\n            xk = _geo(x, k)\\r\\n            if k:\\r\\n                xk = _vec(xk, k)\\r\\n            xk = xk.contiguous(memory_format=torch.channels_last)\\r\\n            with torch.autocast(\\\"cuda\\\", dtype=dtype, enabled=device.type == \\\"cuda\\\"):\\r\\n                o = model(xk, ext) if ext is not None else model(xk)\\r\\n            acc.append(_inv(torch.sigmoid(o[\\\"main\\\"].float()), k).cpu().numpy()[:, 0])\\r\\n            if i == 0 and \\\"area\\\" in o:\\r\\n                areas.append(o[\\\"area\\\"].float().cpu().numpy())\\r\\n        probs[i] = np.concatenate(acc)\\r\\n    area = np.concatenate(areas) if areas else None\\r\\n    return np.mean(probs, axis=0), area\\r\\n\\r\\n\\r\\ndef _targets(ds):\\r\\n    tgt = np.stack([np.asarray(ds.fire[i]) >= 7 for i in range(len(ds))])\\r\\n    val = np.stack([~np.isin(np.asarray(ds.fire[i]), (4, 6)) for i in range(len(ds))])\\r\\n    return tgt, val\\r\\n\\r\\n\\r\\ndef main() -> None:\\r\\n    ap = argparse.ArgumentParser()\\r\\n    ap.add_argument(\\\"--name\\\", required=True)\\r\\n    ap.add_argument(\\\"--stage\\\", type=int, default=3)\\r\\n    ap.add_argument(\\\"--tta\\\", type=int, default=8)\\r\\n    ap.add_argument(\\\"--test\\\", action=\\\"store_true\\\")\\r\\n    ap.add_argument(\\\"--runs\\\", default=\\\"results/runs\\\")\\r\\n    ap.add_argument(\\\"--skip-train\\\", action=\\\"store_true\\\")\\r\\n    ap.add_argument(\\\"--seed\\\", type=int, default=0)\\r\\n    ap.add_argument(\\\"--resume\\\", default=None, help=\\\"checkpoint to fine-tune from, e.g. a stage-1 best.pt\\\")\\r\\n    ap.add_argument(\\\"--tag\\\", default=\\\"\\\", help=\\\"suffix for the run dir, e.g. _seed1\\\")\\r\\n    ap.add_argument(\\\"--lr\\\", type=float, default=0.0, help=\\\"override the protocol lr (fine-tuning)\\\")\\r\\n    ap.add_argument(\\\"--epochs\\\", type=int, default=0, help=\\\"override the protocol epochs (fine-tuning)\\\")\\r\\n    args = ap.parse_args()\\r\\n\\r\\n    dev = torch.device(\\\"cuda\\\")\\r\\n    dtype = torch.bfloat16\\r\\n    root = \\\"data/tsf\\\"\\r\\n    cfg = make_cfg(args.name, args.stage)\\r\\n    cfg[\\\"seed\\\"] = args.seed\\r\\n    if args.lr:\\r\\n        cfg[\\\"train\\\"][\\\"lr\\\"] = args.lr\\r\\n    if args.epochs:\\r\\n        cfg[\\\"train\\\"].update(epochs=args.epochs, patience=args.epochs)\\r\\n    run_name = f\\\"{args.name}_s{args.stage}{args.tag}\\\"\\r\\n    out = Path(args.runs) / run_name\\r\\n\\r\\n    if not args.skip_train:\\r\\n        run(cfg, run_name, args.runs, resume_from=args.resume)\\r\\n\\r\\n    ckpt = torch.load(out / \\\"best.pt\\\", map_location=dev)\\r\\n    model, prep, drop = build(cfg, root)\\r\\n    model = model.to(dev).to(memory_format=torch.channels_last)\\r\\n    model.load_state_dict(ckpt[\\\"model\\\"], strict=False)\\r\\n\\r\\n    ev_ds = FireDataset(root, \\\"eval\\\", None, aux=False, prep=prep, drop=drop,\\r\\n                        ext=load_ext(root, \\\"eval\\\") if cfg[\\\"model\\\"].get(\\\"late\\\") else None)\\r\\n    tg, va = _targets(ev_ds)\\r\\n    ld = _loader(ev_ds, 512, False, 4)\\r\\n    prob, ar = predict_tta(model, ld, dev, dtype, args.tta)\\r\\n    if ar is not None:\\r\\n        thr, rows = sweep_area(prob, np.expm1(ar), tg, va)\\r\\n        ev = evaluate_area(prob, np.expm1(ar) * thr, tg, va, full=True)\\r\\n        res_decode = \\\"area\\\"\\r\\n    else:\\r\\n        thr, rows = sweep(prob, tg, va)\\r\\n        ev = evaluate(prob, tg, va, thr, full=True)\\r\\n        res_decode = \\\"thr\\\"\\r\\n    res = {\\\"name\\\": args.name, \\\"run\\\": run_name, \\\"seed\\\": args.seed, \\\"stage\\\": args.stage, \\\"tta\\\": args.tta,\\r\\n           \\\"decode\\\": res_decode, \\\"thr\\\": thr, \\\"sweep\\\": rows, \\\"eval\\\": ev, \\\"prep\\\": prep, \\\"drop\\\": drop}\\r\\n    np.save(out / \\\"eval_prob.npy\\\", prob)\\r\\n    if ar is not None:\\r\\n        np.save(out / \\\"eval_area.npy\\\", ar)\\r\\n\\r\\n    if args.test:\\r\\n        te_ds = FireDataset(root, \\\"test\\\", None, aux=False, prep=prep, drop=drop,\\r\\n                            ext=load_ext(root, \\\"test\\\") if cfg[\\\"model\\\"].get(\\\"late\\\") else None)\\r\\n        ttg, tva = _targets(te_ds)\\r\\n        tprob, tar = predict_tta(model, _loader(te_ds, 512, False, 4), dev, dtype, args.tta)\\r\\n        res[\\\"test\\\"] = (evaluate_area(tprob, np.expm1(tar) * thr, ttg, tva, full=True) if tar is not None\\r\\n                       else evaluate(tprob, ttg, tva, thr, full=True))\\r\\n\\r\\n    (out / \\\"final.json\\\").write_text(json.dumps(res, indent=1))\\r\\n    print(json.dumps({k: res[k] for k in (\\\"name\\\", \\\"thr\\\", \\\"eval\\\")}, indent=1), flush=True)\\r\\n    if args.test:\\r\\n        print(json.dumps({\\\"test\\\": res[\\\"test\\\"]}, indent=1), flush=True)\\r\\n\\r\\n\\r\\nif __name__ == \\\"__main__\\\":\\r\\n    main()\\r\\n\",\"wf/models.py\":\"\\\"\\\"\\\"Trunks + heads, small enough to try a lot of variants on one L4.\\\"\\\"\\\"\\r\\n\\r\\nfrom __future__ import annotations\\r\\n\\r\\nimport numpy as np\\r\\nimport torch\\r\\nimport torch.nn as nn\\r\\nimport torch.nn.functional as F\\r\\n\\r\\nfrom .data import DYN_IDX, STA_IDX\\r\\n\\r\\n\\r\\ndef cna(cin: int, cout: int, k: int = 3, stride: int = 1, dilation: int = 1) -> nn.Sequential:\\r\\n    return nn.Sequential(\\r\\n        nn.Conv2d(cin, cout, k, stride, padding=dilation * (k // 2), dilation=dilation, bias=False),\\r\\n        nn.BatchNorm2d(cout),\\r\\n        nn.GELU(),\\r\\n    )\\r\\n\\r\\n\\r\\nclass SE(nn.Module):\\r\\n    def __init__(self, c: int, r: int = 8):\\r\\n        super().__init__()\\r\\n        self.fc1 = nn.Conv2d(c, max(8, c // r), 1)\\r\\n        self.fc2 = nn.Conv2d(max(8, c // r), c, 1)\\r\\n\\r\\n    def forward(self, x):\\r\\n        w = F.adaptive_avg_pool2d(x, 1)\\r\\n        return x * torch.sigmoid(self.fc2(F.gelu(self.fc1(w))))\\r\\n\\r\\n\\r\\nclass Res(nn.Module):\\r\\n    def __init__(self, c: int, dilation: int = 1, drop_path: float = 0.0):\\r\\n        super().__init__()\\r\\n        self.c1 = nn.Conv2d(c, c, 3, 1, dilation, dilation, bias=False)\\r\\n        self.n1 = nn.BatchNorm2d(c)\\r\\n        self.c2 = nn.Conv2d(c, c, 3, 1, dilation, dilation, bias=False)\\r\\n        self.n2 = nn.BatchNorm2d(c)\\r\\n        self.se = SE(c)\\r\\n        self.dp = drop_path\\r\\n\\r\\n    def forward(self, x):\\r\\n        h = F.gelu(self.n1(self.c1(x)))\\r\\n        h = self.se(self.n2(self.c2(h)))\\r\\n        if self.training and self.dp > 0:\\r\\n            keep = 1.0 - self.dp\\r\\n            mask = torch.rand(h.shape[0], 1, 1, 1, device=h.device) < keep\\r\\n            h = h * mask / keep\\r\\n        return x + h\\r\\n\\r\\n\\r\\nclass LK(nn.Module):\\r\\n    \\\"\\\"\\\"ConvNeXt-lite: 7x7 depthwise + inverted bottleneck, gives wide context cheaply.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, c: int, exp: int = 2, drop_path: float = 0.0):\\r\\n        super().__init__()\\r\\n        self.dw = nn.Conv2d(c, c, 7, 1, 3, groups=c, bias=False)\\r\\n        self.n = nn.BatchNorm2d(c)\\r\\n        self.pw1 = nn.Conv2d(c, exp * c, 1)\\r\\n        self.pw2 = nn.Conv2d(exp * c, c, 1)\\r\\n        self.gamma = nn.Parameter(torch.ones(c, 1, 1) * 1e-2)\\r\\n        self.dp = drop_path\\r\\n\\r\\n    def forward(self, x):\\r\\n        h = self.n(self.dw(x))\\r\\n        h = self.pw2(F.gelu(self.pw1(h)))\\r\\n        if self.training and self.dp > 0:\\r\\n            keep = 1.0 - self.dp\\r\\n            h = h * (torch.rand(h.shape[0], 1, 1, 1, device=h.device) < keep) / keep\\r\\n        return x + self.gamma * h\\r\\n\\r\\n\\r\\nclass FullRes(nn.Module):\\r\\n    def __init__(self, cin: int, width: int, blocks: list[list[int]], block: str = \\\"res\\\",\\r\\n                 fusion: str = \\\"concat\\\", drop_path: float = 0.0):\\r\\n        super().__init__()\\r\\n        self.fusion = fusion\\r\\n        if fusion == \\\"gated\\\":\\r\\n            half = max(16, width // 2)\\r\\n            self.stem_s = cna(len(STA_IDX), half)\\r\\n            self.stem_d = cna(len(DYN_IDX), half)\\r\\n            self.gate = nn.Sequential(nn.Conv2d(width, width, 1), nn.Sigmoid())\\r\\n        else:\\r\\n            self.stem = cna(cin, width)\\r\\n        mk = Res if block == \\\"res\\\" else LK\\r\\n        self.blocks = nn.ModuleList([mk(width, *([d] if block == \\\"res\\\" else []), drop_path=drop_path)\\r\\n                                     for d, n in blocks for _ in range(n)])\\r\\n        self.out_c = width\\r\\n\\r\\n    def forward(self, x):\\r\\n        if self.fusion == \\\"gated\\\":\\r\\n            s, d = x[:, STA_IDX], x[:, DYN_IDX]\\r\\n            h = torch.cat([self.stem_s(s), self.stem_d(d)], 1)\\r\\n            h = h * self.gate(h)\\r\\n        else:\\r\\n            h = self.stem(x)\\r\\n        for b in self.blocks:\\r\\n            h = b(h)\\r\\n        return h\\r\\n\\r\\n\\r\\nclass ASPP(nn.Module):\\r\\n    def __init__(self, c: int, dilations=(1, 2, 4, 8)):\\r\\n        super().__init__()\\r\\n        self.branches = nn.ModuleList([cna(c, c // 2, 3, dilation=d) for d in dilations])\\r\\n        self.pool = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(c, c // 2, 1), nn.GELU())\\r\\n        self.out = cna(c // 2 * (len(dilations) + 1), c, 1)\\r\\n\\r\\n    def forward(self, x):\\r\\n        feats = [b(x) for b in self.branches]\\r\\n        p = F.interpolate(self.pool(x), size=x.shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n        return self.out(torch.cat(feats + [p], 1))\\r\\n\\r\\n\\r\\nclass AG(nn.Module):\\r\\n    \\\"\\\"\\\"Attention gate on the skip connection.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, f_g: int, f_l: int, f_int: int | None = None):\\r\\n        super().__init__()\\r\\n        f_int = f_int or max(8, f_l // 2)\\r\\n        self.wg = nn.Conv2d(f_g, f_int, 1)\\r\\n        self.wx = nn.Conv2d(f_l, f_int, 1)\\r\\n        self.psi = nn.Conv2d(f_int, 1, 1)\\r\\n\\r\\n    def forward(self, g, x):\\r\\n        a = torch.sigmoid(self.psi(F.gelu(self.wg(g) + self.wx(x))))\\r\\n        return x * a\\r\\n\\r\\n\\r\\nclass UNet(nn.Module):\\r\\n    def __init__(self, cin: int, widths: list[int], depths: list[int], block: str = \\\"res\\\",\\r\\n                 drop_path: float = 0.0, gated_skip: bool = True):\\r\\n        super().__init__()\\r\\n        mk = Res if block == \\\"res\\\" else LK\\r\\n        self.stem = cna(cin, widths[0])\\r\\n        self.enc = nn.ModuleList()\\r\\n        self.down = nn.ModuleList()\\r\\n        for i, w in enumerate(widths):\\r\\n            self.enc.append(nn.Sequential(*[mk(w, 1, drop_path) if block == \\\"res\\\" else mk(w, drop_path=drop_path)\\r\\n                                            for _ in range(depths[i])]))\\r\\n            if i + 1 < len(widths):\\r\\n                self.down.append(cna(w, widths[i + 1], 3, stride=2))\\r\\n        self.bott = ASPP(widths[-1])\\r\\n        self.up = nn.ModuleList()\\r\\n        self.dec = nn.ModuleList()\\r\\n        self.ag = nn.ModuleList()\\r\\n        for i in range(len(widths) - 1, 0, -1):\\r\\n            self.up.append(cna(widths[i], widths[i - 1], 3))\\r\\n            self.dec.append(nn.Sequential(*[mk(widths[i - 1], 1, drop_path) if block == \\\"res\\\"\\r\\n                                            else mk(widths[i - 1], drop_path=drop_path) for _ in range(depths[i - 1])]))\\r\\n            self.ag.append(AG(widths[i - 1], widths[i - 1]) if gated_skip else None)\\r\\n        self.out_c = widths[0]\\r\\n\\r\\n    def forward(self, x):\\r\\n        h = self.stem(x)\\r\\n        skips = []\\r\\n        for i, e in enumerate(self.enc):\\r\\n            h = e(h)\\r\\n            if i + 1 < len(self.enc):\\r\\n                skips.append(h)\\r\\n                h = self.down[i](h)\\r\\n        h = self.bott(h)\\r\\n        for i, (u, d) in enumerate(zip(self.up, self.dec)):\\r\\n            s = skips[-1 - i]\\r\\n            h = F.interpolate(u(h), size=s.shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n            if self.ag[i] is not None:\\r\\n                s = self.ag[i](h, s)\\r\\n            h = d(h + s)\\r\\n        return h\\r\\n\\r\\n\\r\\nclass HRNet(nn.Module):\\r\\n    \\\"\\\"\\\"3 parallel resolutions with repeated fusion; keeps full-res detail for fire fronts.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, cin: int, width: int = 32, stages: int = 3, block: str = \\\"res\\\", drop_path: float = 0.0):\\r\\n        super().__init__()\\r\\n        mk = Res if block == \\\"res\\\" else LK\\r\\n        ws = [width, width * 2, width * 4]\\r\\n        self.stem = nn.Sequential(cna(cin, ws[0]), cna(ws[0], ws[0]))\\r\\n        self.branch = nn.ModuleList()\\r\\n        self.fuse = nn.ModuleList()\\r\\n        self.trans = nn.ModuleList()\\r\\n        for i in range(1, 3):\\r\\n            self.trans.append(cna(ws[i - 1], ws[i], 3, stride=2))\\r\\n        for s in range(stages):\\r\\n            b = nn.ModuleList([nn.Sequential(*[mk(w, 1, drop_path) if block == \\\"res\\\" else mk(w, drop_path=drop_path)\\r\\n                                               for _ in range(2)]) for w in ws])\\r\\n            self.branch.append(b)\\r\\n            f = nn.ModuleList([nn.ModuleList([nn.Conv2d(o, w, 1, bias=False) for o in ws]) for w in ws])\\r\\n            self.fuse.append(f)\\r\\n        self.out_c = ws[0] + ws[1] + ws[2]\\r\\n\\r\\n    def forward(self, x):\\r\\n        h = self.stem(x)\\r\\n        feats = [h]\\r\\n        for i, t in enumerate(self.trans):\\r\\n            feats.append(t(feats[-1]))\\r\\n        for b, f in zip(self.branch, self.fuse):\\r\\n            hs = [b[i](feats[i]) for i in range(3)]\\r\\n            new = []\\r\\n            for i in range(3):\\r\\n                acc = hs[i]\\r\\n                for j in range(3):\\r\\n                    if j == i:\\r\\n                        continue\\r\\n                    t = f[i][j](hs[j])\\r\\n                    if j > i:\\r\\n                        t = F.interpolate(t, size=hs[i].shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n                    elif j < i:\\r\\n                        t = F.avg_pool2d(t, 1 << (i - j))\\r\\n                    acc = acc + t\\r\\n                new.append(acc)\\r\\n            feats = new\\r\\n        up = [feats[0]] + [F.interpolate(f, size=feats[0].shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n                           for f in feats[1:]]\\r\\n        return torch.cat(up, 1)\\r\\n\\r\\n\\r\\nclass CAFIM(nn.Module):\\r\\n    \\\"\\\"\\\"Cross-attentive fuel/weather gate: alpha for fuel, 1-alpha for weather.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, c: int):\\r\\n        super().__init__()\\r\\n        self.wf = nn.Conv2d(c, c, 1)\\r\\n        self.ww = nn.Conv2d(c, c, 1)\\r\\n        self.g = nn.Sequential(nn.Conv2d(2 * c, c, 3, padding=1), nn.GELU(), nn.Conv2d(c, 1, 3, padding=1))\\r\\n\\r\\n    def forward(self, fs, fd):\\r\\n        a = torch.sigmoid(self.g(torch.cat([self.wf(fs), self.ww(fd)], 1)))\\r\\n        return torch.cat([a * fs, (1 - a) * fd], 1)\\r\\n\\r\\n\\r\\nclass DualUNet(nn.Module):\\r\\n    \\\"\\\"\\\"Two encoders (fuel 3x3, weather 5x5) fused per scale, U-Net decoder.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, cin: int, widths=(32, 64, 128), depths=(2, 2, 2), drop_path: float = 0.0):\\r\\n        super().__init__()\\r\\n        w = widths\\r\\n        self.stem_s = cna(len(STA_IDX), w[0], 3)\\r\\n        self.stem_d = cna(len(DYN_IDX), w[0], 5, dilation=2)\\r\\n\\r\\n        def block(c, k):\\r\\n            return nn.Sequential(cna(c, c, k), cna(c, c, k))\\r\\n\\r\\n        self.es = nn.ModuleList([nn.Sequential(*[block(w[i], 3) for _ in range(depths[i])]) for i in range(3)])\\r\\n        self.ed = nn.ModuleList([nn.Sequential(*[block(w[i], 5) for _ in range(depths[i])]) for i in range(3)])\\r\\n        self.fuse = nn.ModuleList([CAFIM(w[i]) for i in range(3)])\\r\\n        self.ds = nn.ModuleList([cna(w[i], w[i + 1], 3, stride=2) for i in range(2)])\\r\\n        self.dd = nn.ModuleList([cna(w[i], w[i + 1], 3, stride=2) for i in range(2)])\\r\\n        self.up = nn.ModuleList([cna(2 * w[2], 2 * w[1]), cna(w[1], 2 * w[0])])\\r\\n        self.mix = nn.ModuleList([cna(2 * w[1] + 2 * w[1], w[1]), cna(2 * w[0] + 2 * w[0], w[0])])\\r\\n        self.out_c = w[0]\\r\\n\\r\\n    def forward(self, x):\\r\\n        fs, fd = self.stem_s(x[:, STA_IDX]), self.stem_d(x[:, DYN_IDX])\\r\\n        skips = []\\r\\n        for i in range(3):\\r\\n            fs, fd = self.es[i](fs), self.ed[i](fd)\\r\\n            f = self.fuse[i](fs, fd)\\r\\n            if i < 2:\\r\\n                skips.append(f)\\r\\n                fs, fd = self.ds[i](fs), self.dd[i](fd)\\r\\n        h = F.interpolate(self.up[0](f), size=skips[1].shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n        h = self.mix[0](torch.cat([h, skips[1]], 1))\\r\\n        h = F.interpolate(self.up[1](h), size=skips[0].shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n        return self.mix[1](torch.cat([h, skips[0]], 1))\\r\\n\\r\\n\\r\\n\\\"\\\"\\\"Orthogonal-transform trunks (TD-FusionUNet, ICIP 2026).\\r\\n\\r\\nIdea borrowed from Luo et al.: a U-Net whose blocks mix globally through a fixed\\r\\northogonal transform (Walsh-Hadamard / DCT) with a learned per-coefficient gain\\r\\nand soft-threshold. That gives a 64x64 patch long-range mixing for the price of\\r\\ntwo small matmuls per channel, instead of attention or a deep dilated stack.\\r\\n\\\"\\\"\\\"\\r\\n\\r\\n_BASIS: dict[tuple[str, int], torch.Tensor] = {}\\r\\n\\r\\n\\r\\ndef basis(kind: str, n: int) -> torch.Tensor:\\r\\n    \\\"\\\"\\\"Orthonormal n x n transform matrix (Walsh-Hadamard or DCT-II).\\\"\\\"\\\"\\r\\n    key = (kind, n)\\r\\n    if key not in _BASIS:\\r\\n        if kind == \\\"ht\\\":\\r\\n            h = np.ones((1, 1), np.float32)\\r\\n            while h.shape[0] < n:\\r\\n                h = np.block([[h, h], [h, -h]])\\r\\n            h /= np.sqrt(n)\\r\\n        else:\\r\\n            k = np.arange(n)[:, None]\\r\\n            i = np.arange(n)[None, :]\\r\\n            h = np.cos(np.pi * (i + 0.5) * k / n) * np.sqrt(2.0 / n)\\r\\n            h[0] = np.sqrt(1.0 / n)\\r\\n        _BASIS[key] = torch.tensor(h, dtype=torch.float32)\\r\\n    return _BASIS[key]\\r\\n\\r\\n\\r\\nclass Spectral(nn.Module):\\r\\n    \\\"\\\"\\\"Transform, per-coefficient gain, soft-threshold, inverse transform.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, c: int, size: int, kind: str = \\\"ht\\\"):\\r\\n        super().__init__()\\r\\n        self.register_buffer(\\\"B\\\", basis(kind, size))\\r\\n        self.gain = nn.Parameter(torch.ones(1, c, size, size))\\r\\n        self.thr = nn.Parameter(torch.full((1, c, size, size), -5.0))  # softplus(-5) ~ 0.007\\r\\n\\r\\n    def forward(self, x):\\r\\n        z = self.B @ x @ self.B.t()\\r\\n        z = z * self.gain\\r\\n        z = torch.sign(z) * F.relu(z.abs() - F.softplus(self.thr))\\r\\n        return self.B.t() @ z @ self.B\\r\\n\\r\\n\\r\\nclass SpecBlock(nn.Module):\\r\\n    \\\"\\\"\\\"Conv block + spectral perceptron, residual. `conv` kind = plain control.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, c: int, size: int, kind: str):\\r\\n        super().__init__()\\r\\n        self.body = nn.Sequential(cna(c, c), cna(c, c))\\r\\n        self.spec = None if kind == \\\"conv\\\" else Spectral(c, size, kind)\\r\\n\\r\\n    def forward(self, x):\\r\\n        y = self.body(x)\\r\\n        return x + (y if self.spec is None else self.spec(y))\\r\\n\\r\\n\\r\\nclass BranchEnc(nn.Module):\\r\\n    def __init__(self, cin: int, width: int, depth: int, kind: str, sizes):\\r\\n        super().__init__()\\r\\n        self.stem = cna(cin, width)\\r\\n        ws = [width * 2 ** i for i in range(len(sizes))]\\r\\n        self.stages = nn.ModuleList([\\r\\n            nn.Sequential(*[SpecBlock(ws[i], sizes[i], kind) for _ in range(depth)]) for i in range(len(sizes))])\\r\\n        self.down = nn.ModuleList([cna(ws[i], ws[i + 1], 3, stride=2) for i in range(len(sizes) - 1)])\\r\\n\\r\\n    def forward(self, x):\\r\\n        h, out = self.stem(x), []\\r\\n        for i, st in enumerate(self.stages):\\r\\n            h = st(h)\\r\\n            out.append(h)\\r\\n            if i < len(self.stages) - 1:\\r\\n                h = self.down[i](h)\\r\\n        return out\\r\\n\\r\\n\\r\\nclass Fuse(nn.Module):\\r\\n    \\\"\\\"\\\"Merge k branch feature maps: per-branch scalar sum, or a learned pixel gate.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, c: int, k: int, mode: str = \\\"gate\\\"):\\r\\n        super().__init__()\\r\\n        self.mode = mode\\r\\n        if mode == \\\"sum\\\":\\r\\n            self.w = nn.Parameter(torch.ones(k))\\r\\n        else:\\r\\n            self.proj = nn.ModuleList([nn.Conv2d(c, c, 1) for _ in range(k)])\\r\\n            self.g = nn.Sequential(nn.Conv2d(c * k, c, 3, padding=1), nn.GELU(), nn.Conv2d(c, k, 1))\\r\\n\\r\\n    def forward(self, fs):\\r\\n        if self.mode == \\\"sum\\\":\\r\\n            return sum(w * f for w, f in zip(self.w, fs))\\r\\n        z = torch.stack([p(f) for p, f in zip(self.proj, fs)], 1)\\r\\n        a = torch.softmax(self.g(torch.cat(fs, 1)), 1).unsqueeze(2)\\r\\n        return (z * a).sum(1)\\r\\n\\r\\n\\r\\nclass TDUNet(nn.Module):\\r\\n    \\\"\\\"\\\"k parallel transform encoders, shared decoder, fused at every scale.\\r\\n\\r\\n    `dec` widens only the decoder (the trunk stays 1M), `attn` adds SE channel\\r\\n    attention after each fusion. Both default to the screened winner's shape.\\r\\n    `ds` exposes the half-resolution decoder feature for deep supervision.\\r\\n    `hr` adds a 2x sub-pixel branch: a stem on the upsampled input, concatenated\\r\\n    with the upsampled decoder feature, so the head decides on a 128x128 lattice\\r\\n    and its logits are pooled back to 64 (`out_up`).\\r\\n    \\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, cin: int, width: int = 16, branches=(\\\"ht\\\", \\\"dct\\\"), depth: int = 1,\\r\\n                 fusion: str = \\\"gate\\\", sizes=(64, 32, 16), attn: bool = False, dec: int = 1,\\r\\n                 ds: bool = False, hr: int = 0):\\r\\n        super().__init__()\\r\\n        w, k, n = width, len(branches), len(sizes)\\r\\n        ch = [w * 2 ** i for i in range(n)]\\r\\n        self.enc = nn.ModuleList([BranchEnc(cin, w, depth, b, sizes) for b in branches])\\r\\n        fuses = []\\r\\n        for c in ch:\\r\\n            f = Fuse(c, k, fusion)\\r\\n            fuses.append(nn.Sequential(f, SE(c)) if attn else f)\\r\\n        self.fuse = nn.ModuleList(fuses)\\r\\n        self.up = nn.ModuleList([cna(ch[i + 1] * (dec if i < n - 2 else 1), ch[i] * dec)\\r\\n                                 for i in range(n - 1)])\\r\\n        self.mix = nn.ModuleList([cna(ch[i] * dec + ch[i], ch[i] * dec) for i in range(n - 1)])\\r\\n        self.out_c = w * dec\\r\\n        self.ds, self.hr, self.aux_channels = ds, hr, []\\r\\n        self.out_up = 2 if hr else 1\\r\\n        if ds and n >= 3:\\r\\n            self.aux_channels = [ch[n - 2] * dec]\\r\\n        if hr:\\r\\n            self.hr_stem = nn.Sequential(cna(cin, hr), cna(hr, hr))\\r\\n            self.hr_mix = cna(self.out_c + hr, self.out_c)\\r\\n\\r\\n    def forward(self, x):\\r\\n        fs = [e(x) for e in self.enc]\\r\\n        h = self.fuse[-1]([f[-1] for f in fs])\\r\\n        aux, first = [], len(fs[0]) - 2\\r\\n        for i in range(first, -1, -1):\\r\\n            h = self.up[i](h)\\r\\n            h = F.interpolate(h, size=fs[0][i].shape[-2:], mode=\\\"bilinear\\\", align_corners=False)\\r\\n            if self.ds and i == first and self.aux_channels:\\r\\n                aux.append(h)  # coarsest decoder stage, 32x32 for the 3-scale trunk\\r\\n            h = self.mix[i](torch.cat([h, self.fuse[i]([f[i] for f in fs])], 1))\\r\\n        if self.hr:\\r\\n            up = dict(scale_factor=2, mode=\\\"bilinear\\\", align_corners=False)\\r\\n            h = self.hr_mix(torch.cat([F.interpolate(h, **up), self.hr_stem(F.interpolate(x, **up))], 1))\\r\\n        return (h, aux) if self.ds else h\\r\\n\\r\\n\\r\\nclass Head(nn.Module):\\r\\n    def __init__(self, c: int, out: int = 1):\\r\\n        super().__init__()\\r\\n        self.f = cna(c, max(16, c // 2))\\r\\n        self.o = nn.Conv2d(max(16, c // 2), out, 1)\\r\\n\\r\\n    def forward(self, x):\\r\\n        return self.o(self.f(x))\\r\\n\\r\\n\\r\\nclass Net(nn.Module):\\r\\n    def __init__(self, cfg: dict, mean: list[float], std: list[float]):\\r\\n        super().__init__()\\r\\n        self.register_buffer(\\\"mean\\\", torch.tensor(mean).view(1, -1, 1, 1))\\r\\n        self.register_buffer(\\\"std\\\", torch.tensor(std).view(1, -1, 1, 1))\\r\\n        cfg = dict(cfg)\\r\\n        trunk = cfg.pop(\\\"trunk\\\", \\\"fullres\\\")\\r\\n        heads = cfg.pop(\\\"heads\\\", [\\\"main\\\"])\\r\\n        cin = cfg.pop(\\\"cin\\\", 23)\\r\\n        late = int(cfg.pop(\\\"late\\\", 0))\\r\\n        self.late_mode = cfg.pop(\\\"late_mode\\\", \\\"scalar\\\")\\r\\n        if self.late_mode not in (\\\"scalar\\\", \\\"spatial\\\"):\\r\\n            raise ValueError(\\\"late_mode must be scalar or spatial\\\")\\r\\n        self.trunk_name = trunk\\r\\n        if trunk == \\\"fullres\\\":\\r\\n            self.trunk = FullRes(cin, cfg.pop(\\\"width\\\", 96), cfg.pop(\\\"blocks\\\", [[1, 2], [2, 2], [4, 2], [8, 2], [16, 2]]),\\r\\n                                 cfg.pop(\\\"block\\\", \\\"res\\\"), cfg.pop(\\\"fusion\\\", \\\"concat\\\"), cfg.pop(\\\"drop_path\\\", 0.0))\\r\\n        elif trunk == \\\"unet\\\":\\r\\n            self.trunk = UNet(cin, cfg.pop(\\\"widths\\\", [48, 96, 192]), cfg.pop(\\\"depths\\\", [2, 2, 2]),\\r\\n                              cfg.pop(\\\"block\\\", \\\"res\\\"), cfg.pop(\\\"drop_path\\\", 0.0), cfg.pop(\\\"gated_skip\\\", True))\\r\\n        elif trunk == \\\"hrnet\\\":\\r\\n            self.trunk = HRNet(cin, cfg.pop(\\\"width\\\", 32), cfg.pop(\\\"stages\\\", 3), cfg.pop(\\\"block\\\", \\\"res\\\"),\\r\\n                               cfg.pop(\\\"drop_path\\\", 0.0))\\r\\n        elif trunk == \\\"dual\\\":\\r\\n            self.trunk = DualUNet(cin, cfg.pop(\\\"widths\\\", (32, 64, 128)), cfg.pop(\\\"depths\\\", (2, 2, 2)),\\r\\n                                  cfg.pop(\\\"drop_path\\\", 0.0))\\r\\n        elif trunk == \\\"td\\\":\\r\\n            self.trunk = TDUNet(cin, cfg.pop(\\\"width\\\", 16), tuple(cfg.pop(\\\"branches\\\", (\\\"ht\\\", \\\"dct\\\"))),\\r\\n                                cfg.pop(\\\"depth\\\", 1), cfg.pop(\\\"fusion\\\", \\\"gate\\\"),\\r\\n                                tuple(cfg.pop(\\\"sizes\\\", (64, 32, 16))), cfg.pop(\\\"attn\\\", False),\\r\\n                                cfg.pop(\\\"dec\\\", 1), cfg.pop(\\\"ds\\\", False), cfg.pop(\\\"hr\\\", 0))\\r\\n        else:\\r\\n            raise ValueError(f\\\"unknown trunk {trunk}\\\")\\r\\n        wide = cfg.pop(\\\"area\\\", 0)\\r\\n        if cfg:\\r\\n            raise ValueError(f\\\"unused model keys: {sorted(cfg)}\\\")\\r\\n        c = self.trunk.out_c\\r\\n        self.heads = nn.ModuleDict({h: Head(c, 4 if h == \\\"dist\\\" else 1) for h in heads})\\r\\n        self.aux = nn.ModuleDict({f\\\"aux{i}\\\": Head(ac, 1)\\r\\n                                  for i, ac in enumerate(getattr(self.trunk, \\\"aux_channels\\\", []))})\\r\\n        # Per-patch fire size, regressed from the decoder's global statistics. The\\r\\n        # mask's own probability mass is a poor size estimate (corr 0.56 with the\\r\\n        # GT area), while the area decides ~0.09 Dice at fixed ranking.\\r\\n        self.area_head = None\\r\\n        if wide:\\r\\n            self.area_head = nn.Sequential(nn.Linear(2 * c, wide), nn.GELU(), nn.Linear(wide, 1))\\r\\n            # Start exactly at the mean GT area: a random last layer can land far\\r\\n            # from it, which puts the joint training in a useless basin.\\r\\n            nn.init.zeros_(self.area_head[-1].weight)\\r\\n            nn.init.constant_(self.area_head[-1].bias, 2.9)  # ~ln 17 px\\r\\n        # Late fusion: an external stream (GHCN station weather) shifts the final\\r\\n        # logit map per image. Zero-init, so the run starts as the plain trunk and\\r\\n        # the external data has to earn its weight.\\r\\n        self.late = None\\r\\n        if late:\\r\\n            self.late = nn.Sequential(nn.Linear(late, 32), nn.SiLU(),\\r\\n                                      nn.Linear(32, c + 1 if self.late_mode == \\\"spatial\\\" else 1))\\r\\n            nn.init.zeros_(self.late[-1].weight)\\r\\n            nn.init.zeros_(self.late[-1].bias)\\r\\n\\r\\n    def forward(self, x, ext=None):\\r\\n        x = (x - self.mean) / self.std\\r\\n        f = self.trunk(x)\\r\\n        aux = []\\r\\n        if isinstance(f, tuple):\\r\\n            f, aux = f\\r\\n        out = {h: m(f) for h, m in self.heads.items()}\\r\\n        up = getattr(self.trunk, \\\"out_up\\\", 1)\\r\\n        if up > 1:\\r\\n            # The head decides on the finer lattice; average-pool its logits so\\r\\n            # the loss and the metrics stay on the 64x64 label grid.\\r\\n            out = {k: (F.avg_pool2d(v, up) if k == \\\"main\\\" else v) for k, v in out.items()}\\r\\n        if self.late is not None:\\r\\n            if ext is None:\\r\\n                raise ValueError(\\\"Late fusion requires external features\\\")\\r\\n            z = self.late(ext.float())\\r\\n            if self.late_mode == \\\"spatial\\\":\\r\\n                correction = (f * z[:, :-1, None, None]).sum(1, keepdim=True) / f.shape[1] ** 0.5\\r\\n                correction = correction + z[:, -1, None, None, None]\\r\\n                if up > 1:\\r\\n                    correction = F.avg_pool2d(correction, up)\\r\\n            else:\\r\\n                correction = z.view(-1, 1, 1, 1)\\r\\n            out[\\\"main\\\"] = out[\\\"main\\\"] + correction\\r\\n        for i, a in enumerate(aux):\\r\\n            out[f\\\"aux{i}\\\"] = self.aux[f\\\"aux{i}\\\"](a)\\r\\n        if self.area_head is not None:\\r\\n            out[\\\"area\\\"] = self.area_head(torch.cat([f.mean((2, 3)), f.amax((2, 3))], 1)).squeeze(1)\\r\\n        return out\\r\\n\\r\\n\\r\\nclass SMPNet(nn.Module):\\r\\n    \\\"\\\"\\\"segmentation_models_pytorch wrapper with the same normalisation + dict output.\\\"\\\"\\\"\\r\\n\\r\\n    def __init__(self, arch: str, encoder: str, weights: str | None, mean: list[float], std: list[float],\\r\\n                 cin: int = 23, heads=(\\\"main\\\",), up: int = 1, area: int = 0):\\r\\n        super().__init__()\\r\\n        import segmentation_models_pytorch as smp\\r\\n\\r\\n        self.register_buffer(\\\"mean\\\", torch.tensor(mean).view(1, -1, 1, 1))\\r\\n        self.register_buffer(\\\"std\\\", torch.tensor(std).view(1, -1, 1, 1))\\r\\n        cls = getattr(smp, arch)\\r\\n        self.model = cls(encoder_name=encoder, encoder_weights=weights, in_channels=cin, classes=len(heads))\\r\\n        self.head_names = list(heads)\\r\\n        # Tiny targets in a 64x64 patch: run the encoder on a finer grid, then\\r\\n        # bring the logits back so the loss and metrics stay at 64x64.\\r\\n        self.up = int(up)\\r\\n        self.area_head = self._feat = None\\r\\n        if area:\\r\\n            # The decoder's full-res features are the segmentation head's input;\\r\\n            # grab them there so this works for every smp architecture.\\r\\n            conv = next(m for m in self.model.segmentation_head.modules() if isinstance(m, nn.Conv2d))\\r\\n            self.model.segmentation_head.register_forward_pre_hook(\\r\\n                lambda _m, a: setattr(self, \\\"_feat\\\", a[0]))\\r\\n            self.area_head = nn.Sequential(nn.Linear(2 * conv.in_channels, area), nn.GELU(),\\r\\n                                           nn.Linear(area, 1))\\r\\n            nn.init.zeros_(self.area_head[-1].weight)\\r\\n            nn.init.constant_(self.area_head[-1].bias, 2.9)\\r\\n\\r\\n    def forward(self, x):\\r\\n        x = (x - self.mean) / self.std\\r\\n        size = x.shape[-2:]\\r\\n        if self.up > 1:\\r\\n            x = F.interpolate(x, size=(size[0] * self.up, size[1] * self.up),\\r\\n                              mode=\\\"bilinear\\\", align_corners=False)\\r\\n        o = self.model(x)\\r\\n        if self.up > 1:\\r\\n            o = F.interpolate(o, size=size, mode=\\\"bilinear\\\", align_corners=False)\\r\\n        out = {h: o[:, i:i + 1] for i, h in enumerate(self.head_names)}\\r\\n        if self.area_head is not None:\\r\\n            f = self._feat\\r\\n            out[\\\"area\\\"] = self.area_head(torch.cat([f.mean((2, 3)), f.amax((2, 3))], 1)).squeeze(1)\\r\\n        return out\\r\\n\\r\\n\\r\\ndef build_model(mcfg: dict, mean, std) -> nn.Module:\\r\\n    mcfg = dict(mcfg)\\r\\n    if \\\"smp\\\" in mcfg:\\r\\n        smp = mcfg.pop(\\\"smp\\\")\\r\\n        heads = mcfg.pop(\\\"heads\\\", [\\\"main\\\"])\\r\\n        up = mcfg.pop(\\\"up\\\", 1)\\r\\n        return SMPNet(smp[\\\"arch\\\"], smp[\\\"encoder\\\"], smp.get(\\\"weights\\\", \\\"imagenet\\\"), mean, std,\\r\\n                      cin=mcfg.pop(\\\"cin\\\", 23), heads=heads, up=up, area=mcfg.pop(\\\"area\\\", 0))\\r\\n    return Net(mcfg, mean, std)\\r\\n\",\"wf/__init__.py\":\"\\\"\\\"\\\"Wildfire spread experiment harness.\\\"\\\"\\\"\\r\\n\",\"wf/build.py\":\"\\\"\\\"\\\"TFRecord -> the 23-channel float16 cache the trainers read.\\n\\nThe shards store 14 raw float fields; the old `data/processed` cache was built\\nfrom them by a pipeline that is not in this repo, so this rebuild re-derives the\\n23 channels and calibrates the per-channel scales against the statistics that\\nwere recorded from the old cache (`OLD_STATS` below). Channels 0-10 must land on\\nthose numbers; the rest use the derivations measured earlier (fire_dist =\\nclip(signed EDT * 0.1, -0.3, 1), fire_front = 1-px dilation of prev_fire).\\n\\nUsage (Colab):\\n    python -m wf.build --shards 10 --out data/processed\\n    python -m wf.build --calibrate --shards 2      # stats only, no files written\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport json\\nimport struct\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom scipy import ndimage as ndi\\n\\nRAW = [\\\"elevation\\\", \\\"population\\\", \\\"pdsi\\\", \\\"NDVI\\\", \\\"pr\\\", \\\"sph\\\", \\\"th\\\", \\\"tmmn\\\", \\\"tmmx\\\",\\n       \\\"vs\\\", \\\"erc\\\", \\\"PrevFireMask\\\", \\\"waterOccurrence\\\", \\\"FireMask\\\"]\\n\\n# mean/sd/min/max/nonzero of channels 0-10, measured on the old cache (3000 samples).\\nOLD_STATS = {\\n    \\\"ndvi\\\": (0.601, 0.219, -0.200, 1.000, 0.99),\\n    \\\"elevation\\\": (0.404, 0.639, 0.000, 4.121, 0.98),\\n    \\\"slope\\\": (0.025, 0.053, 0.000, 0.780, 0.94),\\n    \\\"aspect_sin\\\": (0.032, 0.670, None, None, 0.88),\\n    \\\"aspect_cos\\\": (-0.060, 0.698, None, None, 0.88),\\n    \\\"population\\\": (0.177, 0.193, 0.000, 1.082, 0.78),\\n    \\\"water\\\": (0.152, 0.276, 0.000, 1.000, 0.41),\\n    \\\"th_sin\\\": (-0.101, 0.714, None, None, 0.99),\\n    \\\"th_cos\\\": (-0.368, 0.587, None, None, 1.00),\\n    \\\"wind_speed\\\": (0.344, 0.126, 0.000, 3.000, 0.99),\\n    \\\"wind_u\\\": (-0.037, 0.261, -2.992, 1.647, 0.99),\\n}\\n\\n# Scales fitted by matching the old cache's channel statistics on 3k raw shards:\\n#   water        clip(w,0,100)/100      -> mean 0.152 sd 0.276 max 1.0 (old 0.152/0.276/1.0)\\n#   population   log10(1+pop)/4.46      -> mean 0.173 sd 0.188 max 1.082 (old 0.177/0.193/1.082)\\n#   wind_speed   vs*0.1 (clip 0..30)    -> mean 0.344 sd 0.126 max 3.0  (old 0.344/0.126/3.0)\\n#   elevation    m/1000                 -> mean 0.404 sd 0.638            (old 0.404/0.639)\\nNDVI_SCALE = 1e4\\nELEV_SCALE = 1000.0\\nPOP_LOG = 4.46\\nWATER_SCALE = 100.0\\nWIND_SCALE = 0.1\\nSPACING = 1000.0  # metres between patch pixels\\nSLOPE_MAX = 0.78  # the old cache capped here\\n\\n\\ndef _clean(ex: dict) -> dict:\\n    \\\"\\\"\\\"Kill the int16/projection sentinels the raw export contains.\\\"\\\"\\\"\\n    c = dict(ex)\\n    c[\\\"NDVI\\\"] = np.clip(ex[\\\"NDVI\\\"], -2000, NDVI_SCALE)\\n    c[\\\"elevation\\\"] = np.clip(ex[\\\"elevation\\\"], 0, 5000)\\n    c[\\\"th\\\"] = np.where((ex[\\\"th\\\"] >= 0) & (ex[\\\"th\\\"] <= 360), ex[\\\"th\\\"], 0.0)\\n    c[\\\"vs\\\"] = np.clip(ex[\\\"vs\\\"], 0, 30)\\n    c[\\\"waterOccurrence\\\"] = np.clip(ex[\\\"waterOccurrence\\\"], 0, WATER_SCALE)\\n    c[\\\"tmmx\\\"] = np.clip(ex[\\\"tmmx\\\"], 200, 350)\\n    c[\\\"tmmn\\\"] = np.clip(ex[\\\"tmmn\\\"], 190, 340)\\n    c[\\\"sph\\\"] = np.clip(ex[\\\"sph\\\"], 0, 0.06)\\n    c[\\\"pr\\\"] = np.clip(ex[\\\"pr\\\"], 0, 100)\\n    c[\\\"pdsi\\\"] = np.clip(ex[\\\"pdsi\\\"], -10, 10)\\n    c[\\\"erc\\\"] = np.clip(ex[\\\"erc\\\"], 0, 150)\\n    return c\\n\\n\\ndef _varint(b: bytes, i: int):\\n    shift = val = 0\\n    while True:\\n        x = b[i]\\n        i += 1\\n        val |= (x & 0x7F) << shift\\n        if not x & 0x80:\\n            return val, i\\n        shift += 7\\n\\n\\ndef _fields(b: bytes):\\n    i, n = 0, len(b)\\n    while i < n:\\n        tag, i = _varint(b, i)\\n        fno, wt = tag >> 3, tag & 7\\n        if wt == 0:\\n            v, i = _varint(b, i)\\n        elif wt == 2:\\n            ln, i = _varint(b, i)\\n            v, i = b[i:i + ln], i + ln\\n        elif wt == 5:\\n            v, i = b[i:i + 4], i + 4\\n        elif wt == 1:\\n            v, i = b[i:i + 8], i + 8\\n        else:\\n            raise ValueError(f\\\"wire type {wt}\\\")\\n        yield fno, wt, v\\n\\n\\ndef read_shard(path: Path):\\n    \\\"\\\"\\\"Yield {field: 64x64 float32} per example. 1 GFLOP-free, no tensorflow.\\\"\\\"\\\"\\n    with path.open(\\\"rb\\\", buffering=1 << 20) as f:\\n        while True:\\n            head = f.read(8)\\n            if len(head) < 8:\\n                return\\n            (ln,) = struct.unpack(\\\"<Q\\\", head)\\n            f.read(4)\\n            payload = f.read(ln)\\n            f.read(4)\\n            if len(payload) < ln:\\n                return\\n            for fno, _, feats in _fields(payload):\\n                if fno != 1:\\n                    continue\\n                ex = {}\\n                for _, _, entry in _fields(feats):\\n                    key, raw = None, None\\n                    for kfno, _, kf in _fields(entry):\\n                        if kfno == 1:\\n                            key = kf.decode()\\n                        elif kfno == 2:\\n                            for g, _, gv in _fields(kf):\\n                                if g == 2:\\n                                    raw = gv\\n                    if key in RAW and raw is not None:\\n                        ex[key] = np.frombuffer(raw[:4096 * 4], \\\"<f4\\\").reshape(64, 64)\\n                if len(ex) == len(RAW):\\n                    yield ex\\n\\n\\ndef derive(ex: dict) -> np.ndarray:\\n    \\\"\\\"\\\"14 raw fields -> 23 model channels (float32).\\\"\\\"\\\"\\n    ex = _clean(ex)\\n    elev_m = ex[\\\"elevation\\\"]\\n    gy, gx = np.gradient(elev_m / SPACING)\\n    th = np.deg2rad(ex[\\\"th\\\"])\\n    vs, r = ex[\\\"vs\\\"], np.hypot(gx, gy)\\n    prev = (ex[\\\"PrevFireMask\\\"] >= 7).astype(np.float32)\\n    inside = ndi.distance_transform_edt(prev > 0.5)\\n    outside = ndi.distance_transform_edt(prev <= 0.5)\\n    dist = np.clip(np.where(prev > 0.5, -inside, outside) * 0.1, -0.3, 1.0)\\n    yy, xx = np.mgrid[0:64, 0:64].astype(np.float32)\\n    ang = np.arctan2(gy, gx)\\n    x = np.stack([\\n        ex[\\\"NDVI\\\"] / NDVI_SCALE,\\n        elev_m / ELEV_SCALE,\\n        np.minimum(r, SLOPE_MAX),\\n        np.sin(ang),\\n        np.cos(ang),\\n        np.log10(1.0 + ex[\\\"population\\\"]) / POP_LOG,\\n        ex[\\\"waterOccurrence\\\"] / WATER_SCALE,\\n        np.sin(th),\\n        np.cos(th),\\n        vs * WIND_SCALE,\\n        vs * WIND_SCALE * np.sin(th),\\n        vs * WIND_SCALE * np.cos(th),\\n        ex[\\\"tmmx\\\"], ex[\\\"tmmn\\\"], ex[\\\"sph\\\"], ex[\\\"pr\\\"], ex[\\\"pdsi\\\"], ex[\\\"erc\\\"],\\n        prev,\\n        ndi.binary_dilation(prev > 0.5).astype(np.float32),\\n        dist,\\n        (xx - 31.5) / 31.5,\\n        (yy - 31.5) / 31.5,\\n    ]).astype(np.float32)\\n    return x\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--data\\\", default=\\\"datasets/next-day-wildfire-spread\\\")\\n    ap.add_argument(\\\"--out\\\", default=\\\"data/processed\\\")\\n    ap.add_argument(\\\"--shards\\\", type=int, default=0, help=\\\"train shards to use (0 = all)\\\")\\n    ap.add_argument(\\\"--calibrate\\\", action=\\\"store_true\\\")\\n    args = ap.parse_args()\\n\\n    root = Path(args.data)\\n    train = sorted((root / \\\"train\\\").rglob(\\\"*.tfrecord\\\"))\\n    ev = sorted((root / \\\"eval\\\").rglob(\\\"*.tfrecord\\\"))\\n    splits = {\\\"train\\\": train[: args.shards or None],\\n              # a small --shards build must still fit its split ratios\\n              \\\"eval\\\": ev[: max(2, args.shards // 4)] if args.shards else ev,\\n              \\\"test\\\": sorted((root / \\\"test\\\").rglob(\\\"*.tfrecord\\\"))}\\n    if args.calibrate:\\n        splits.pop(\\\"test\\\")  # 3k examples are enough to fit the scales\\n\\n    acc = {k: [] for k in OLD_STATS}\\n    for split, files in splits.items():\\n        xs, fires, prevs = [], [], []\\n        for f in files:\\n            for ex in read_shard(f):\\n                if args.calibrate:\\n                    if len(acc[\\\"ndvi\\\"]) < 3000:\\n                        x = derive(ex)\\n                        for i, name in enumerate([\\\"ndvi\\\", \\\"elevation\\\", \\\"slope\\\", \\\"aspect_sin\\\", \\\"aspect_cos\\\",\\n                                                  \\\"population\\\", \\\"water\\\", \\\"th_sin\\\", \\\"th_cos\\\", \\\"wind_speed\\\",\\n                                                  \\\"wind_u\\\"]):\\n                            acc[name].append(x[i])\\n                    continue\\n                xs.append(derive(ex).astype(np.float16))\\n                fires.append(ex[\\\"FireMask\\\"].astype(np.uint8))\\n                prevs.append(ex[\\\"PrevFireMask\\\"].astype(np.uint8))\\n                if len(xs) >= 20000:\\n                    _flush(args.out, split, xs, fires, prevs)\\n                    xs, fires, prevs = [], [], []\\n            print(f\\\"{split} {f.name}: total {_count(args.out, split) + len(xs)}\\\", flush=True)\\n        if xs:\\n            _flush(args.out, split, xs, fires, prevs)\\n\\n    if args.calibrate:\\n        for name, old in OLD_STATS.items():\\n            a = np.stack(acc[name])\\n            print(f\\\"{name:12s} mean {a.mean():7.3f} (old {old[0]:7.3f})  sd {a.std():6.3f} (old {old[1]:6.3f})  \\\"\\n                  f\\\"min {a.min():7.3f}  max {a.max():7.3f} (old {old[3] if old[3] is not None else '?'})\\\")\\n    else:\\n        _write_norm(args.out)\\n\\n\\ndef _flush(out, split: str, xs, fires, prevs) -> None:\\n    d = Path(out) / split\\n    d.mkdir(parents=True, exist_ok=True)\\n    n0 = _count(out, split)\\n    with (d / \\\"features.f16\\\").open(\\\"ab\\\") as f:\\n        f.write(np.stack(xs).astype(np.float16).tobytes())\\n    with (d / \\\"fire_class.u8\\\").open(\\\"ab\\\") as f:\\n        f.write(np.stack(fires).astype(np.uint8).tobytes())\\n    with (d / \\\"prev_class.u8\\\").open(\\\"ab\\\") as f:\\n        f.write(np.stack(prevs).astype(np.uint8).tobytes())\\n    print(f\\\"  wrote {len(xs)} rows -> {n0 + len(xs)}\\\", flush=True)\\n\\n\\ndef _count(out, split: str) -> int:\\n    p = Path(out) / split / \\\"fire_class.u8\\\"\\n    return p.stat().st_size // (64 * 64) if p.exists() else 0\\n\\n\\ndef _write_norm(out) -> None:\\n    from .data import cache\\n\\n    x = cache(out, \\\"train\\\")[0]\\n    idx = np.random.default_rng(0).choice(len(x), min(20000, len(x)), replace=False)\\n    sums, squares, count = np.zeros(x.shape[1]), np.zeros(x.shape[1]), 0\\n    for start in range(0, len(idx), 32):\\n        block = np.asarray(x[idx[start:start + 32]], dtype=np.float64)\\n        if not np.isfinite(block).all():\\n            raise ValueError(\\\"Nonfinite cached features; inspect source bands before training\\\")\\n        sums += block.sum((0, 2, 3))\\n        squares += (block ** 2).sum((0, 2, 3))\\n        count += block.shape[0] * block.shape[2] * block.shape[3]\\n    mean_array = sums / count\\n    mean = mean_array.tolist()\\n    std = np.maximum(np.sqrt(np.maximum(squares / count - mean_array ** 2, 0)), 1e-3).tolist()\\n    (Path(out) / \\\"norm.json\\\").write_text(json.dumps({\\\"mean\\\": mean, \\\"std\\\": std}, indent=1))\\n    print(\\\"norm.json written\\\", flush=True)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/search.py\":\"\\\"\\\"\\\"Search driver: runs candidates in ORDER, skips finished ones, writes a CSV row each.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport csv\\nimport json\\nimport time\\nimport traceback\\nfrom pathlib import Path\\n\\nfrom .candidates import ORDER, make_cfg\\nfrom .train import run\\n\\nCOLS = [\\\"name\\\", \\\"stage\\\", \\\"params\\\", \\\"best_epoch\\\", \\\"best_thr\\\", \\\"macro_dice\\\", \\\"micro_dice\\\", \\\"iou\\\",\\n        \\\"precision\\\", \\\"recall\\\", \\\"ap\\\", \\\"boundary_f1\\\", \\\"hard_dice\\\", \\\"minutes\\\", \\\"error\\\"]\\n\\n\\ndef _done(path: Path) -> set[str]:\\n    if not path.exists():\\n        return set()\\n    with path.open() as f:\\n        return {r[\\\"name\\\"] for r in csv.DictReader(f)}\\n\\n\\ndef _append(path: Path, row: dict) -> None:\\n    new = not path.exists()\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    with path.open(\\\"a\\\", newline=\\\"\\\") as f:\\n        w = csv.DictWriter(f, fieldnames=COLS, extrasaction=\\\"ignore\\\")\\n        if new:\\n            w.writeheader()\\n        w.writerow(row)\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--stage\\\", type=int, default=1)\\n    ap.add_argument(\\\"--only\\\", default=None, help=\\\"comma separated candidate names\\\")\\n    ap.add_argument(\\\"--deadline\\\", type=float, default=0.0, help=\\\"minutes for this whole invocation\\\")\\n    ap.add_argument(\\\"--csv\\\", default=None)\\n    ap.add_argument(\\\"--runs\\\", default=\\\"results/runs\\\")\\n    args = ap.parse_args()\\n\\n    names = args.only.split(\\\",\\\") if args.only else ORDER\\n    csv_path = Path(args.csv or f\\\"results/search_s{args.stage}.csv\\\")\\n    done = _done(csv_path)\\n    t0 = time.time()\\n    for name in names:\\n        if name in done:\\n            print(\\\"skip\\\", name, flush=True)\\n            continue\\n        if args.deadline and (time.time() - t0) / 60 > args.deadline:\\n            print(\\\"deadline reached, stopping\\\", flush=True)\\n            break\\n        cfg = make_cfg(name, args.stage)\\n        print(\\\"START\\\", name, json.dumps(cfg[\\\"model\\\"]), flush=True)\\n        row = {\\\"name\\\": name, \\\"stage\\\": args.stage}\\n        try:\\n            s = run(cfg, f\\\"{name}_s{args.stage}\\\", args.runs)\\n            v = s[\\\"val_at_best_thr\\\"]\\n            row.update(params=s[\\\"params\\\"], best_epoch=s[\\\"best_epoch\\\"], best_thr=s[\\\"best_thr\\\"],\\n                       macro_dice=round(v[\\\"macro_dice\\\"], 4), micro_dice=round(v[\\\"micro_dice\\\"], 4),\\n                       iou=round(v[\\\"iou\\\"], 4), precision=round(v[\\\"precision\\\"], 4), recall=round(v[\\\"recall\\\"], 4),\\n                       ap=round(v[\\\"ap\\\"], 4), boundary_f1=round(v[\\\"boundary_f1\\\"], 4),\\n                       hard_dice=round(v[\\\"hard_dice\\\"], 4), minutes=s[\\\"minutes\\\"])\\n        except Exception as exc:  # keep the campaign going\\n            traceback.print_exc()\\n            row[\\\"error\\\"] = repr(exc)[:200]\\n        _append(csv_path, row)\\n        print(\\\"ROW\\\", json.dumps(row), flush=True)\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/audit.py\":\"\\\"\\\"\\\"Dataset audit: label classes, target size, scar overlap, trivial baselines.\\n\\nPrints the numbers the cleaning/experiment decisions rest on. Cheap: pure\\nTFRecord reads, no cache needed.\\n\\n    python -m wf.audit --split eval --limit 4000\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nfrom pathlib import Path\\n\\nimport numpy as np\\nfrom scipy import ndimage as ndi\\n\\nfrom .build import read_shard\\n\\nROOT = Path(\\\"datasets/next-day-wildfire-spread\\\")\\n\\n\\ndef _dice(pred: np.ndarray, tgt: np.ndarray) -> float:\\n    den = pred.sum() + tgt.sum()\\n    return 1.0 if den == 0 else float(2 * np.logical_and(pred, tgt).sum() / den)\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--split\\\", default=\\\"eval\\\")\\n    ap.add_argument(\\\"--limit\\\", type=int, default=4000)\\n    ap.add_argument(\\\"--root\\\", default=str(ROOT))\\n    args = ap.parse_args()\\n\\n    files = sorted((Path(args.root) / args.split).rglob(\\\"*.tfrecord\\\"))\\n    fires, prevs, classes = [], [], []\\n    for f in files:\\n        for ex in read_shard(f):\\n            fires.append(ex[\\\"FireMask\\\"].astype(np.uint8))\\n            prevs.append((ex[\\\"PrevFireMask\\\"] >= 7).astype(bool))\\n            classes.append(ex[\\\"FireMask\\\"].astype(np.uint8).ravel())\\n            if len(fires) >= args.limit:\\n                break\\n        if len(fires) >= args.limit:\\n            break\\n\\n    fire = np.stack(fires)\\n    prev = np.stack(prevs)\\n    tgt = fire >= 7\\n    cls, cnt = np.unique(np.concatenate(classes), return_counts=True)\\n    area = tgt.sum((1, 2))\\n\\n    print(f\\\"{args.split}: {len(fire)} patches\\\")\\n    print(\\\"class counts (px, %):\\\",\\n          {int(c): f\\\"{n} ({100 * n / fire.size:.2f}%)\\\" for c, n in zip(cls, cnt)})\\n    print(\\\"valid-mask classes in use: target >=7, ignore {4,6}\\\")\\n    print(f\\\"empty GT: {(area == 0).mean():.1%}   area pct 10/25/50/75/90/99: \\\"\\n          + \\\"/\\\".join(str(int(np.percentile(area, q))) for q in (10, 25, 50, 75, 90, 99)))\\n    print(f\\\"patches with <10 GT px: {(area < 10).mean():.1%}   >=100: {(area >= 100).mean():.1%}\\\")\\n\\n    # how much of the target is copyable from the previous scar\\n    inter = np.logical_and(tgt, prev).sum((1, 2))\\n    print(f\\\"GT px inside prev scar: {inter.sum() / max(tgt.sum(), 1):.1%}   \\\"\\n          f\\\"patches where GT == prev: {np.mean([np.array_equal(a, b) for a, b in zip(tgt, prev)]):.1%}\\\")\\n\\n    # 5 is background here; 4 and 6 are the ignore labels that leak into the loss\\n    soft = np.isin(fire, (4, 6))\\n    edge = tgt & ~ndi.binary_erosion(tgt)\\n    near = ndi.binary_dilation(soft, iterations=1)\\n    print(f\\\"target edge px adjacent to class 4/6: {(edge & near).sum() / max(edge.sum(), 1):.1%}   \\\"\\n          f\\\"patches containing 4/6: {np.mean([s.any() for s in soft]):.1%}   all 4/6 px: {soft.sum()}\\\")\\n\\n    # how much of the metric is just \\\"put a correctly sized blob at the right place\\\"\\n    yy, xx = np.mgrid[0:64, 0:64]\\n    disk_d, disk_c, cent = [], [], []\\n    for t in tgt:\\n        k = int(t.sum())\\n        cy, cx = ndi.center_of_mass(t)\\n        d = (yy - cy) ** 2 + (xx - cx) ** 2\\n        near_px = d <= np.sort(d.ravel())[min(k, 4096) - 1]\\n        disk_d.append(_dice(near_px, t))\\n        c = (yy - 31.5) ** 2 + (xx - 31.5) ** 2 <= 2 * 2\\n        disk_c.append(_dice(c, t))\\n        cent.append(float(np.hypot(cy - 31.5, cx - 31.5)))\\n    print(f\\\"dice of best-area disk at GT centroid: {np.mean(disk_d):.4f}   \\\"\\n          f\\\"GT centroid dist to patch centre: p50 {np.median(cent):.1f} p90 {np.percentile(cent, 90):.1f}\\\")\\n    print(f\\\"dice of fixed radius-2 centre disk: {np.mean(disk_c):.4f}\\\")\\n\\n    # trivial baselines, same metric as the model (macro Dice, empty = 1.0)\\n    out = {\\\"copy_prev_scar\\\": [], \\\"prev_dilate1\\\": [], \\\"prev_both_wind\\\": [], \\\"empty\\\": []}\\n    for t, p in zip(tgt, prev):\\n        out[\\\"copy_prev_scar\\\"].append(_dice(p, t))\\n        out[\\\"prev_dilate1\\\"].append(_dice(ndi.binary_dilation(p), t))\\n        out[\\\"prev_both_wind\\\"].append(_dice(p | ndi.binary_dilation(p), t))\\n        out[\\\"empty\\\"].append(_dice(np.zeros_like(t), t))\\n    print(\\\"trivial macro Dice:\\\", {k: round(float(np.mean(v)), 4) for k, v in out.items()})\\n    # upper bound if only the area were right: top-k of the GT itself\\n    print(f\\\"mean GT area on non-empty: {area[area > 0].mean():.1f} px\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\",\"wf/area.py\":\"\\\"\\\"\\\"Can the per-patch fire size be predicted at all? Ridge feasibility check.\\n\\nThe area head on the TD decoder learns almost nothing beyond the mask's own\\nprobability mass (corr 0.557 vs 0.541), which says the decoder representation\\nhas thrown the size information away. This asks the prior question: is the size\\nin the *inputs*? Per-patch statistics of the 23 cached channels (weather,\\nterrain, vegetation, previous scar) plus the mask features are regressed on\\nlog1p(area) with a closed-form ridge, fitted on train and scored on eval.\\n\\nIf this cannot beat the mask's own mass, no head on this data will, and the\\narea route is closed.\\n\\n    python -m wf.area --n 40000\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\n\\nimport numpy as np\\n\\nfrom .data import FireDataset, cache, split_indices\\n\\n\\ndef _features(x: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Patch-level summaries of the input channels.\\\"\\\"\\\"\\n    x = np.asarray(x, np.float32)\\n    return np.concatenate([x.mean((1, 2)), x.std((1, 2)), x.max((1, 2))])\\n\\n\\ndef _fit(A: np.ndarray, y: np.ndarray, lam: float = 1.0) -> np.ndarray:\\n    A = np.hstack([A, np.ones((len(A), 1), np.float32)])\\n    reg = lam * np.eye(A.shape[1], dtype=np.float32)\\n    reg[-1, -1] = 0.0\\n    return np.linalg.solve(A.T @ A + reg, A.T @ y)\\n\\n\\ndef main() -> None:\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\\"--root\\\", default=\\\"data/processed\\\")\\n    ap.add_argument(\\\"--n\\\", type=int, default=40000)\\n    ap.add_argument(\\\"--lam\\\", type=float, default=10.0)\\n    args = ap.parse_args()\\n\\n    xin = cache(args.root, \\\"train\\\")[0]\\n    fire = cache(args.root, \\\"train\\\")[1]\\n    idx = split_indices(args.root, \\\"train\\\", args.n, seed=0)\\n    X, A = [], []\\n    for i in idx:\\n        X.append(_features(np.asarray(xin[i])[None])[0])\\n        f = np.asarray(fire[i])\\n        A.append(np.log1p((f >= 7).sum()))\\n    X, A = np.stack(X), np.stack(A)\\n    print(f\\\"train {len(X)} features {X.shape[1]}  log-area mean {A.mean():.2f} sd {A.std():.2f}\\\")\\n\\n    w = _fit(X, A, args.lam)\\n    pred_tr = np.hstack([X, np.ones((len(X), 1), np.float32)]) @ w\\n    print(f\\\"train med relerr {np.median(np.abs(np.expm1(pred_tr) / np.expm1(A) - 1)):.2f} \\\"\\n          f\\\"corr {np.corrcoef(pred_tr, A)[0, 1]:.3f}\\\")\\n\\n    ds = FireDataset(args.root, \\\"eval\\\", None, aux=False)\\n    fe, ae = [], []\\n    for i in range(len(ds)):\\n        fe.append(_features(np.asarray(ds.x[i])[None])[0])\\n        ae.append(np.log1p(int((np.asarray(ds.fire[i]) >= 7).sum())))\\n    fe, ae = np.stack(fe), np.stack(ae)\\n    pe = np.hstack([fe, np.ones((len(fe), 1), np.float32)]) @ w\\n    ke, ka = np.expm1(pe), np.expm1(ae)\\n    print(f\\\"eval  med relerr {np.median(np.abs(ke / np.maximum(ka, 1) - 1)):.2f} \\\"\\n          f\\\"corr {np.corrcoef(pe, ae)[0, 1]:.3f}  mean k {ke.mean():.1f} vs area {ka.mean():.1f}\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    main()\\n\"}")
for name, source in sources.items():
    path = root / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ['PYTHONPATH'] = str(root)
os.environ['OMP_NUM_THREADS'] = '2'


In [ ]:
import runpy
runpy.run_path('kaggle/run_exploration.py', run_name='__main__')

In [ ]:
print(Path('/kaggle/working/exploration/report.json').read_text())